# Qwen3-ASR 会議文字起こし v3

音声/動画 → **句読点つき SRT・読みやすいテキスト・話者つき議事録用テキスト** を作るノートブックです。

**使い方**: ⓪ → ① を実行 → ②〜④ のフォームを埋める → ⑤ 実行（メニューの「ランタイム → すべてのセルを実行」でもOK）

| | v2 からの主な改善 |
|---|---|
| モデル | Qwen3-ASR 1.7B/0.6B・vLLM 版・Whisper large-v3/turbo・kotoba-whisper・Parakeet 日本語 などをプルダウンで切り替え。**⑥で並べて比較**も |
| 環境 | モデルごとに別の venv で動かすので **依存の衝突なし・再起動いらず** |
| 字幕 | アライナーの単語に **句読点・記号を付け直す**ので SRT も句読点つき。句読点の所で自然に改行 |
| 区切り | v2 で消えていた「長すぎる区間のハード切り」を復活（静かな所を探して切る＋重なり除去） |
| 失敗対策 | context 復唱・ループ・空振りを検出 → context なし → 2分割 の順で自動リトライ |
| 話者 | 文字起こしと**並行して**話者分離。文単位の多数決でチラつき補正・話者名の置換・RTTM |
| 出力 | txt / srt / vtt / json / csv(Excel) / md(議事録用) / プレーン |
| 長時間 | 結果を少しずつキャッシュ → 切断されても**続きから再開**。フォルダ一括処理・アップロード対応 |
| 速度 | GPU に合わせてバッチ自動調整・長さ順バッチ・OOM で自動縮小・A100/H100 は vLLM |

<!-- 詳細（この行と末尾のコメント記号を外すと表示される）

## しくみ
- 音声は ffmpeg で 16kHz モノラルに変換（動画もOK）→ VAD で発話区間 → 最大 30 秒のクリップ（モデルによっては短め）
- ASR は各モデル専用の venv の中の「ワーカー」プロセスで動く（カーネルは汚さない）
- タイムスタンプは Qwen3-ForcedAligner（11言語）で全モデル共通に付ける
- 話者分離は pyannote community-1 の exclusive 出力（重なりなし）を単語に割り当て
- 本体コードは GitHub の `v3/asrkit/` と同じもの（⓪のセルに埋め込み）

-->

In [ ]:
#@title ⓪ ライブラリの展開（さわらなくてOK・そのまま実行）
#@markdown v3 の本体コード（asrkit）を書き出して読み込みます。コードは GitHub の `v3/asrkit/` と同じです。
import os, sys, importlib
ASR_HOME = os.environ.setdefault("ASR_V3_HOME", "/content/asr_v3")
_LIB = os.path.join(ASR_HOME, "lib")
_FILES = {}
_FILES['__init__.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit — Qwen3-ASR v3 ノートブックの本体

core     : 音声・区間分割・テキスト処理・書き出し(numpy だけで動く純粋ロジック)
runtime  : venv づくり・ワーカープロセス・vLLM サーバー・GPU/Colab まわり
engines  : ワーカーの中で動く各モデルのアダプタ(Qwen3-ASR / アライナー / Whisper / NeMo / pyannote …)
worker   : ワーカーのメインループ(JSON を1行ずつやりとり)
presets  : モデルのプリセットと venv の中身
pipeline : ノートブックから呼ぶ高レベル処理(文字起こし・モデル比較)
"""
__version__ = "3.0.0"
'''
_FILES['core.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.core — 音声の読み込み・区間分割・テキスト処理・出力の「純粋ロジック」

ノートブック本体(カーネル)とエンジン用ワーカー(別venv)の両方から import される。
import 時に必要なのは標準ライブラリと numpy だけ(torch などは使わない)。
"""
from __future__ import annotations

import bisect
import csv
import io
import json
import math
import os
import re
import struct
import subprocess
import unicodedata
import wave
import zlib
from collections import Counter
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

SR = 16000

# =====================================================================
# 時刻の表記
# =====================================================================


def _split_ms(t: float) -> Tuple[int, int, int, int]:
    ms = int(round(max(0.0, float(t)) * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return h, m, s, ms


def fmt_hms(t: float) -> str:
    """[HH:MM:SS] 用。ミリ秒は切り捨て"""
    t = max(0, int(t))
    return f"{t // 3600:02d}:{t % 3600 // 60:02d}:{t % 60:02d}"


def fmt_srt_time(t: float) -> str:
    h, m, s, ms = _split_ms(t)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def fmt_vtt_time(t: float) -> str:
    h, m, s, ms = _split_ms(t)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def fmt_dur(sec: float) -> str:
    """ログ用のざっくり表記 (例: 1:02:03 / 4:05 / 12.3秒)"""
    sec = float(sec)
    if sec < 60:
        return f"{sec:.1f}秒"
    t = int(round(sec))
    h, r = divmod(t, 3600)
    m, s = divmod(r, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


# =====================================================================
# 音声 I/O (ffmpeg でデコード → 16kHz モノラル int16 WAV → memmap)
# =====================================================================

AUDIO_EXTS = (
    ".wav", ".mp3", ".m4a", ".aac", ".flac", ".ogg", ".oga", ".opus", ".wma", ".aiff", ".aif",
    ".amr", ".3gp", ".webm", ".mp4", ".m4v", ".mov", ".mkv", ".avi", ".wmv", ".mts", ".ts",
)


def _ffmpeg_info(path: str) -> Dict[str, Any]:
    """ffprobe が無い環境用: `ffmpeg -i` の表示から読み取る"""
    try:
        p = subprocess.run(["ffmpeg", "-hide_banner", "-i", path], capture_output=True, text=True, timeout=120)
    except Exception:
        return {}
    err = p.stderr or ""
    out: Dict[str, Any] = {"channels": 0, "sample_rate": 0, "duration": None, "has_audio": False}
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", err)
    if m:
        out["duration"] = int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))
    m = re.search(r"Stream #\S+.*?Audio:.*?(\d+) Hz,\s*([^,]+)", err)
    if m:
        out["has_audio"] = True
        out["sample_rate"] = int(m.group(1))
        lay = m.group(2).strip()
        mm = re.match(r"(\d+) channels", lay)
        out["channels"] = int(mm.group(1)) if mm else {"mono": 1, "stereo": 2}.get(lay, 2 if "." in lay else 1)
    return out


def ffprobe(path: str) -> Dict[str, Any]:
    """先頭の音声ストリームの情報(channels, sample_rate, duration)を返す。取れなければ空dict"""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=channels,sample_rate,duration:format=duration",
        "-of", "json", path,
    ]
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        d = json.loads(p.stdout or "{}")
    except FileNotFoundError:
        return _ffmpeg_info(path)
    except Exception:
        return {}
    st = (d.get("streams") or [{}])[0]
    fmt = d.get("format") or {}
    dur = st.get("duration") or fmt.get("duration")
    out = {
        "channels": int(st.get("channels") or 0),
        "sample_rate": int(st.get("sample_rate") or 0),
        "duration": float(dur) if dur not in (None, "N/A") else None,
        "has_audio": bool(st),
    }
    return out


def ffmpeg_to_wav16(
    src: str,
    dst: str,
    *,
    sr: int = SR,
    channel: str = "mix",
    af: str = "",
    start: Optional[float] = None,
    duration: Optional[float] = None,
) -> None:
    """どんな音声/動画でも 16kHz モノラル int16 の WAV に変換する。

    channel: "mix"(全チャンネル平均) / "left" / "right"
    af     : ffmpeg の追加フィルタ(例: "loudnorm=I=-20:TP=-2:LRA=11")
    """
    info = ffprobe(src)
    if info and not info.get("has_audio"):
        raise RuntimeError(f"音声ストリームが見つかりません: {src}")
    cmd = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y"]
    if start:
        cmd += ["-ss", f"{float(start):.3f}"]
    if duration:
        cmd += ["-t", f"{float(duration):.3f}"]
    cmd += ["-i", src, "-map", "0:a:0", "-vn", "-sn", "-dn"]
    filters = []
    nch = info.get("channels") or 0
    if channel in ("left", "right") and nch >= 2:
        filters.append("pan=mono|c0=c0" if channel == "left" else "pan=mono|c0=c1")
    if af and af.strip():
        filters.append(af.strip())
    if filters:
        cmd += ["-af", ",".join(filters)]
    cmd += ["-ac", "1", "-ar", str(sr), "-c:a", "pcm_s16le", "-f", "wav", dst]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0 or not os.path.exists(dst):
        raise RuntimeError(f"ffmpeg での変換に失敗しました: {src}\n{p.stderr[-3000:]}")


def _parse_wav_header(path: str) -> Tuple[int, int, int, int, int]:
    """(data_offset, data_bytes, channels, sample_rate, bits) を返す。RIFF の LIST 等も読み飛ばす"""
    size_total = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(12)
        if len(head) < 12 or head[:4] != b"RIFF" or head[8:12] != b"WAVE":
            raise ValueError(f"WAV ではありません: {path}")
        ch = sr = bits = None
        while True:
            hdr = f.read(8)
            if len(hdr) < 8:
                raise ValueError(f"WAV の data チャンクが見つかりません: {path}")
            cid = hdr[:4]
            size = struct.unpack("<I", hdr[4:])[0]
            if cid == b"fmt ":
                fmt = f.read(size)
                _fmt_tag, ch, sr, _br, _ba, bits = struct.unpack("<HHIIHH", fmt[:16])
                if size % 2:
                    f.read(1)
            elif cid == b"data":
                off = f.tell()
                if size in (0, 0xFFFFFFFF) or off + size > size_total:
                    size = size_total - off
                if ch is None:
                    raise ValueError("fmt チャンクがありません")
                return off, size, ch, sr, bits
            else:
                f.seek(size + (size % 2), 1)


class Wav16:
    """16bit モノラル WAV を memmap で持つ。get(start, end) で float32 の切り出しを返す"""

    def __init__(self, path: str):
        off, nbytes, ch, sr, bits = _parse_wav_header(path)
        if ch != 1 or bits != 16:
            raise ValueError(f"16bit モノラルのみ対応: ch={ch} bits={bits}")
        self.path = path
        self.sr = int(sr)
        n = nbytes // 2
        self.data = np.memmap(path, dtype="<i2", mode="r", offset=off, shape=(n,)) if n else np.zeros(0, "<i2")

    def __len__(self) -> int:
        return int(self.data.shape[0])

    @property
    def duration(self) -> float:
        return len(self) / float(self.sr)

    def get(self, start: float, end: float) -> np.ndarray:
        a = max(0, int(round(start * self.sr)))
        b = min(len(self), int(round(end * self.sr)))
        if b <= a:
            return np.zeros(0, np.float32)
        return np.asarray(self.data[a:b], dtype=np.float32) / 32768.0

    def int16(self) -> np.ndarray:
        return np.asarray(self.data)


def wav_bytes(audio: np.ndarray, sr: int = SR) -> bytes:
    """float32 [-1,1] → WAV(int16) のバイト列 (HTTP 送信用)"""
    x = np.clip(np.asarray(audio, np.float32), -1.0, 1.0)
    pcm = (x * 32767.0).astype("<i2").tobytes()
    bio = io.BytesIO()
    with wave.open(bio, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes(pcm)
    return bio.getvalue()


def write_wav16(path: str, audio: np.ndarray, sr: int = SR) -> None:
    with open(path, "wb") as f:
        f.write(wav_bytes(audio, sr))


# =====================================================================
# エネルギー(dB) と エネルギーVAD
# =====================================================================

HOP = 0.02  # 20ms


def frame_db(wav: Wav16, hop: float = HOP, chunk_frames: int = 200_000) -> np.ndarray:
    """20ms ごとの RMS(dB, int16 スケール)。3時間でも数秒で終わるようにチャンク処理"""
    hop_n = max(1, int(round(hop * wav.sr)))
    n_frames = len(wav) // hop_n
    out = np.empty(n_frames, np.float32)
    for i in range(0, n_frames, chunk_frames):
        j = min(n_frames, i + chunk_frames)
        x = np.asarray(wav.data[i * hop_n: j * hop_n], dtype=np.float32).reshape(j - i, hop_n)
        out[i:j] = 10.0 * np.log10(np.mean(x * x, axis=1) + 1e-3)
    return out


def _runs(mask: np.ndarray) -> List[Tuple[int, int]]:
    """True が続く区間 [i0, i1) の一覧"""
    if mask.size == 0:
        return []
    m = np.concatenate([[False], mask.astype(bool), [False]])
    d = np.diff(m.astype(np.int8))
    starts = np.flatnonzero(d == 1)
    ends = np.flatnonzero(d == -1)
    return list(zip(starts.tolist(), ends.tolist()))


def energy_vad(
    db: np.ndarray,
    hop: float = HOP,
    top_db: float = 45.0,
    min_speech: float = 0.25,
    min_silence: float = 0.4,
) -> List[Tuple[float, float]]:
    """依存なしの簡易VAD(v1 の librosa.effects.split 相当)。top_db は最大音量からの相対しきい値"""
    if db.size == 0:
        return []
    ref = float(np.percentile(db, 99.5))
    speech = db > (ref - top_db)
    segs = _runs(speech)
    merged: List[List[int]] = []
    gap_n = int(round(min_silence / hop))
    for a, b in segs:
        if merged and a - merged[-1][1] < gap_n:
            merged[-1][1] = b
        else:
            merged.append([a, b])
    min_n = int(round(min_speech / hop))
    return [(a * hop, b * hop) for a, b in merged if b - a >= min_n]


def normalize_segments(
    segs: Iterable[Sequence[float]], total: float, min_len: float = 0.05
) -> List[Tuple[float, float]]:
    """並べ替え・範囲外カット・重なり結合"""
    out: List[List[float]] = []
    for s, e in sorted((float(a), float(b)) for a, b in segs):
        s, e = max(0.0, s), min(total, e)
        if e - s < min_len:
            continue
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(a, b) for a, b in out]


# =====================================================================
# クリップ(ASRに渡す単位)の組み立て
# =====================================================================


@dataclass
class Clip:
    id: int
    start: float  # 音声の切り出し開始(秒, パディング/オーバーラップ込み)
    end: float
    own_start: float = -1e18  # 重複除去用の担当区間(ハード切りの境界だけ有限)
    own_end: float = 1e18
    speech: float = 0.0  # クリップ内のVAD発話秒数
    parent: Optional[int] = None  # リトライで分割された場合の元クリップ
    cut: bool = False  # 右の境界がハード切り(長い発話を強制分割)か

    @property
    def dur(self) -> float:
        return self.end - self.start

    def to_dict(self) -> Dict[str, Any]:
        d = asdict(self)
        d["own_start"] = None if self.own_start <= -1e17 else round(self.own_start, 3)
        d["own_end"] = None if self.own_end >= 1e17 else round(self.own_end, 3)
        d["start"], d["end"], d["speech"] = round(self.start, 3), round(self.end, 3), round(self.speech, 3)
        return d

    @staticmethod
    def from_dict(d: Dict[str, Any]) -> "Clip":
        return Clip(
            id=int(d["id"]),
            start=float(d["start"]),
            end=float(d["end"]),
            own_start=-1e18 if d.get("own_start") is None else float(d["own_start"]),
            own_end=1e18 if d.get("own_end") is None else float(d["own_end"]),
            speech=float(d.get("speech") or 0.0),
            parent=d.get("parent"),
            cut=bool(d.get("cut", False)),
        )


def find_cut(db: Optional[np.ndarray], lo: float, hi: float, hop: float = HOP) -> float:
    """[lo, hi] の中でいちばん静かな時刻を返す(エネルギーが無ければ中央)"""
    if hi <= lo:
        return lo
    if db is None or db.size == 0:
        return (lo + hi) / 2
    a = max(0, int(lo / hop))
    b = min(db.size, int(math.ceil(hi / hop)))
    if b - a < 3:
        return (lo + hi) / 2
    seg = db[a:b]
    # 5フレーム(100ms)平滑してから最小を取る(瞬間的な無音より“間”を優先)
    k = min(5, seg.size)
    sm = np.convolve(seg, np.ones(k, np.float32) / k, mode="same")
    i = int(np.argmin(sm))
    return (a + i + 0.5) * hop


def _speech_in(segs: Sequence[Tuple[float, float]], a: float, b: float) -> float:
    tot = 0.0
    for s, e in segs:
        if e <= a:
            continue
        if s >= b:
            break
        tot += min(b, e) - max(a, s)
    return tot


def build_clips(
    segs: Sequence[Tuple[float, float]],
    total: float,
    *,
    max_clip: float = 30.0,
    max_gap: float = 6.0,
    pad: float = 0.5,
    overlap: float = 1.0,
    db: Optional[np.ndarray] = None,
    hop: float = HOP,
) -> List[Clip]:
    """VAD 区間を「余白込みで max_clip 秒以下」のクリップにまとめる。

    - モデルには前後を少し余分に聞かせる(ふつうの境界は pad 秒・ハード切りは overlap 秒)
    - そのかわり単語は「担当区間」(own_start〜own_end)に中点があるものだけ採用する
      → 境界で語頭/語尾が欠けるのを防ぎつつ、重なった部分の二重転写は出ない
      担当区間の境目は、ふつうの境界なら無音の真ん中、ハード切りなら切った時刻
    - 隣り合う発話は「余白を除いた長さが上限以内」かつ「無音が max_gap 以下」なら 1 クリップに結合
    - 1つの発話が長すぎるときは、なるべく静かな所でハード切り(v1 にあって v2 で抜けていた処理)
    """
    segs = normalize_segments(segs, total)
    if not segs:
        return []
    max_clip = max(2.0, float(max_clip))
    pad = max(0.0, min(float(pad), max_clip * 0.1))
    overlap = max(0.0, min(float(overlap), max_clip * 0.15))
    body = max_clip - 2 * max(pad, overlap)  # 余白を足しても max_clip を超えない本体の長さ

    # 1) 近い発話を結合
    groups: List[List[float]] = []
    cur = [segs[0][0], segs[0][1]]
    for s, e in segs[1:]:
        if e - cur[0] <= body and s - cur[1] <= max_gap:
            cur[1] = e
        else:
            groups.append(cur)
            cur = [s, e]
    groups.append(cur)

    # 2) 長すぎるグループをハード切り → pieces: (start, end, hard_left, hard_right)
    pieces: List[Tuple[float, float, bool, bool]] = []
    for gs, ge in groups:
        if ge - gs <= body:
            pieces.append((gs, ge, False, False))
            continue
        start = gs
        hard_left = False
        while ge - start > body:
            remain = ge - start
            n = math.ceil(remain / body)
            ideal = remain / n
            lo = start + max(ideal * 0.6, min(5.0, body * 0.5))
            hi = start + min(body, ideal * 1.25)
            cut = find_cut(db, lo, hi, hop)
            cut = min(max(cut, start + 1.0), start + body)
            pieces.append((start, cut, hard_left, True))
            start, hard_left = cut, True
        pieces.append((start, ge, hard_left, False))

    # 3) 余白と担当区間
    clips: List[Clip] = []
    for i, (s, e, hl, hr) in enumerate(pieces):
        if i == 0:
            own_s = -1e18
        elif hl:
            own_s = s
        else:
            own_s = (pieces[i - 1][1] + s) / 2  # 無音の真ん中
        if i + 1 == len(pieces):
            own_e = 1e18
        elif hr:
            own_e = e
        else:
            own_e = (e + pieces[i + 1][0]) / 2
        a = max(0.0, s - (overlap if hl else pad))
        b = min(total, e + (overlap if hr else pad))
        clips.append(Clip(id=i, start=a, end=b, own_start=own_s, own_end=own_e, speech=_speech_in(segs, a, b), cut=hr))
    return clips


def split_clip(c: Clip, db: Optional[np.ndarray], next_id: int, overlap: float = 0.5, hop: float = HOP) -> List[Clip]:
    """リトライ用: クリップを静かな所で2つに割る(担当区間も割る)"""
    mid_lo = c.start + c.dur * 0.3
    mid_hi = c.start + c.dur * 0.7
    cut = find_cut(db, mid_lo, mid_hi, hop)
    own_cut_lo = max(cut, c.own_start)
    own_cut_hi = min(cut, c.own_end)
    left = Clip(next_id, c.start, min(c.end, cut + overlap), c.own_start, own_cut_hi, 0.0, parent=c.id, cut=True)
    right = Clip(next_id + 1, max(c.start, cut - overlap), c.end, own_cut_lo, c.own_end, 0.0, parent=c.id, cut=c.cut)
    left.speech = c.speech * (left.dur / max(c.dur, 1e-6))
    right.speech = c.speech * (right.dur / max(c.dur, 1e-6))
    return [left, right]


# =====================================================================
# テキスト処理(句読点の付け直し・品質チェック・辞書置換・フィラー・CER)
# =====================================================================

OPENERS = set("「『（(［[｛{〈《【〔“‘«")
_SENT_END_JA = re.compile(r"[。！？!?…‥]+[」』）)\]】〕”’\"']*\s*$")
_PERIOD_END = re.compile(r"[.．][」』）)\]】〕”’\"']*\s+$")
_COMMA_END = re.compile(r"[、，,;；:：][」』）)\]】〕”’\"']*\s*$")
_CJK = r"぀-ヿ㐀-䶿一-鿿豈-﫿　-〿！-｠"
_JA_SPACE = re.compile(rf"(?<=[{_CJK}])[ \t　]+(?=[{_CJK}])")


def is_kept(ch: str) -> bool:
    """アライナー(qwen-asr)と同じ基準: 文字(L*)・数字(N*)・アポストロフィだけ残す"""
    if ch == "'":
        return True
    return unicodedata.category(ch)[:1] in ("L", "N")


def core_len(text: str) -> int:
    return sum(1 for ch in text if is_kept(ch))


def _match_token(text: str, tok: str, pos: int, max_skip: int = 400) -> Optional[Tuple[int, int]]:
    """tok を text[pos:] の中で探す。tok からは記号が抜けていることがあるので
    「残す文字は一致・それ以外は読み飛ばし可」の部分列マッチで対応を取る"""
    tok = "".join(ch for ch in tok if not ch.isspace())
    if not tok:
        return None
    limit = min(len(text), pos + max_skip + len(tok) * 4)
    first = tok[0]
    start = pos
    while True:
        a = text.find(first, start, limit)
        if a < 0:
            return None
        i, j = a, 0
        while i < len(text) and j < len(tok):
            if text[i] == tok[j]:
                i += 1
                j += 1
            elif not is_kept(text[i]):
                i += 1
            else:
                break
        if j == len(tok):
            return a, i
        start = a + 1


def _split_gap(gap: str) -> Tuple[str, str]:
    """語と語のあいだの文字列を「前の語のうしろ(句読点・空白)」と「次の語の頭(開き括弧)」に分ける"""
    for i, ch in enumerate(gap):
        if ch in OPENERS:
            return gap[:i], gap[i:]
    return gap, ""


def restore_display(text: str, tokens: Sequence[Sequence[Any]]) -> List[Dict[str, Any]]:
    """アライナーの語(句読点なし)を元の文章に対応づけて、句読点・記号・空白を語に付け直す。

    tokens: [(語, start, end), ...]   返り値: [{"word", "core", "start", "end"}, ...]
    対応が取れた範囲では "".join(word) が元の文章と一致する(SRT に句読点が戻る)。
    """
    text = text or ""
    toks = [(str(t[0]), float(t[1]), float(t[2])) for t in tokens if str(t[0]).strip()]
    if not toks:
        return []
    spans: List[Optional[Tuple[int, int]]] = []
    pos = 0
    for tok, _s, _e in toks:
        sp = _match_token(text, tok, pos)
        if sp is not None:
            pos = sp[1]
        spans.append(sp)
    idx = [i for i, sp in enumerate(spans) if sp is not None]
    if not idx:
        return [{"word": t, "core": t, "start": s, "end": max(s, e)} for t, s, e in toks]

    out: List[Dict[str, Any]] = []
    lead = text[: spans[idx[0]][0]]
    for k, i in enumerate(idx):
        a, b = spans[i]
        nxt = spans[idx[k + 1]][0] if k + 1 < len(idx) else len(text)
        trail, next_lead = _split_gap(text[b:nxt])
        if k + 1 == len(idx):
            trail, next_lead = text[b:], ""
        tok, s, e = toks[i]
        out.append({"word": lead + text[a:b] + trail, "core": tok, "start": s, "end": max(s, e)})
        lead = next_lead
    return out


def approx_tokens(text: str, start: float, end: float, speech: Sequence[Tuple[float, float]] = ()) -> List[Tuple[str, float, float]]:
    """タイムスタンプが全く無いモデル用の概算。文字数に比例して発話区間に割り当てる"""
    parts = [p for p in re.findall(r"[^、。！？!?,.，．\s]+[、。！？!?,.，．]*\s*", text or "") if p.strip()]
    if not parts:
        return []
    spans = [(max(start, s), min(end, e)) for s, e in speech if min(end, e) > max(start, s)] or [(start, end)]
    total_sp = sum(e - s for s, e in spans)
    weights = [max(1, core_len(p)) for p in parts]
    tot_w = float(sum(weights))

    def at(x: float) -> float:  # 発話時間上の位置 x(0..total_sp) → 実時刻
        for s, e in spans:
            if x <= e - s:
                return s + x
            x -= e - s
        return spans[-1][1]

    out, acc = [], 0.0
    for p, w in zip(parts, weights):
        a = at(acc / tot_w * total_sp)
        acc += w
        b = at(acc / tot_w * total_sp)
        out.append((p, a, max(a, b)))
    return out


def fix_ja_spaces(s: str) -> str:
    """日本語の文字どうしの間の余計な空白を消す(英単語の間の空白は残す)"""
    return _JA_SPACE.sub("", s)


_FW_ALNUM = {c: c - 0xFEE0 for c in list(range(0xFF10, 0xFF1A)) + list(range(0xFF21, 0xFF3B)) + list(range(0xFF41, 0xFF5B))}


def halfwidth_alnum(s: str) -> str:
    """全角英数字だけ半角に(！？（）などの全角記号はそのまま)"""
    return s.translate(_FW_ALNUM)


def parse_terms(s: str) -> List[str]:
    """context 用の語リスト: スペース/読点/カンマ/中黒/スラッシュ/改行区切り(重複除去・順序維持)"""
    seen, out = set(), []
    for w in re.split(r"[\s、，,・･/／;；]+", (s or "").strip()):
        w = w.strip()
        if w and w not in seen:
            seen.add(w)
            out.append(w)
    return out


def build_context(label: str, terms: Sequence[str], extra: str = "") -> str:
    """issue #321 方式:「見出し: A、B、C。」のフレームで語を渡す(英語見出しは , と .)"""
    parts = []
    if terms:
        sep, end = (", ", ".") if (label or "").isascii() else ("、", "。")
        parts.append((f"{label}: " if label else "") + sep.join(terms) + end)
    if extra and extra.strip():
        parts.append(extra.strip())
    return "\n".join(parts)


def compression_ratio(text: str) -> float:
    b = (text or "").encode("utf-8")
    if not b:
        return 0.0
    return len(b) / max(1, len(zlib.compress(b)))


_LOOP = re.compile(r"(.{1,20}?)\1{5,}", re.S)  # 同じ塊が6回以上連続
# 日本語のはずなのに出てきたらおかしい文字(ハングル・キリル・タイ・アラビア・デーヴァナーガリー)
_FOREIGN = {
    "Japanese": re.compile(r"[\uac00-\ud7af\u1100-\u11ff\u3130-\u318f\u0400-\u04ff\u0e00-\u0e7f\u0600-\u06ff\u0900-\u097f]"),
    "English": re.compile(r"[\u3040-\u30ff\uac00-\ud7af\u4e00-\u9fff]"),
}
_PUNCT = re.compile(r"[、。，．,.!?！？]")


def quality_flags(
    text: str,
    dur: float,
    speech: float,
    *,
    ctx_terms: Sequence[str] = (),
    ctx_label: str = "",
    punctuates: bool = True,
    max_cps: float = 18.0,
    lang: str = "",
) -> List[str]:
    """幻聴・ループ・context の復唱などの“怪しさ”を判定してフラグ名のリストを返す"""
    flags: List[str] = []
    t = (text or "").strip()
    if not t:
        if speech >= 2.0:
            flags.append("empty")  # 2秒以上しゃべっているのに空
        return flags
    core = core_len(t)
    if ctx_label and len(ctx_label) >= 2 and ctx_label in t:
        flags.append("context_label")  # 見出しの復唱
    if ctx_terms:
        residual, hits = t, 0
        for w in sorted(ctx_terms, key=len, reverse=True):
            if w and w in residual:
                hits += residual.count(w)
                residual = residual.replace(w, "")
        rest = core_len(residual)
        if hits >= 2 and rest <= max(3, int(core * 0.1)):
            flags.append("context_echo")  # 語を除くと中身がない=復唱してるだけ
    if dur > 0 and core / max(dur, 1.0) > max_cps:
        flags.append("too_dense")  # しゃべれる速さを超えている
    if _LOOP.search(t) or (len(t) >= 80 and compression_ratio(t) > 3.2):
        flags.append("repetition")
    if punctuates and core > 150 and len(_PUNCT.findall(t)) < 3:
        flags.append("no_punct")  # 句読点なしの長文(v1 からの判定)
    rx = _FOREIGN.get(lang)
    if rx is not None and rx.search(t):
        flags.append("foreign_script")  # 日本語指定なのにハングル等が混ざった
    return flags


def collapse_repeats(text: str, keep: int = 2) -> str:
    """6回以上つづく繰り返しを keep 回に縮める"""
    return _LOOP.sub(lambda m: m.group(1) * keep, text or "")


def parse_replacements(spec: str) -> List[Tuple[Any, str]]:
    """置換辞書。1行1ルール: 「誤 => 正」「誤→正」「誤<TAB>正」。re: で始めると正規表現"""
    rules: List[Tuple[Any, str]] = []
    for line in (spec or "").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r"^(.*?)\s*(?:=>|→|⇒|\t)\s*(.*)$", line)
        if not m:
            continue
        src, dst = m.group(1), m.group(2)
        if not src:
            continue
        if src.startswith("re:"):
            try:
                rules.append((re.compile(src[3:]), dst))
            except re.error:
                continue
        else:
            rules.append((src, dst))
    return rules


def apply_replacements(s: str, rules: Sequence[Tuple[Any, str]]) -> str:
    for src, dst in rules:
        s = src.sub(dst, s) if hasattr(src, "sub") else s.replace(src, dst)
    return s


FILLERS_SAFE = [
    "えーっと", "えーと", "えっと", "えーー", "えー", "ええと", "あのー", "あのう", "あー", "うーん", "うーむ",
    "んー", "そのー", "まー", "えぇ",
]
FILLERS_MORE = ["あの", "その", "まあ", "なんか", "ええ", "うん", "はい"]


def make_filler_regex(words: Sequence[str]) -> Optional["re.Pattern[str]"]:
    words = sorted({w for w in words if w}, key=len, reverse=True)
    if not words:
        return None
    alt = "|".join(re.escape(w) + "ー*" for w in words)
    # 文頭/句読点/空白の直後にあって、うしろに読点/空白/句点/文末が続くときだけ消す
    return re.compile(rf"(?:(?<=^)|(?<=[、。，．,.!?！？\s「『]))(?:{alt})(?:[、，,]\s*|\s+|(?=[。．.!?！？」』]|$))")


def remove_fillers(s: str, rx: Optional["re.Pattern[str]"]) -> str:
    if rx is None:
        return s
    prev = None
    while prev != s:  # 「えー、あのー、」のような連続も消す
        prev = s
        s = rx.sub("", s)
    s = re.sub(r"^[、，,\s]+", "", s)
    s = re.sub(r"[、，,]\s*([。．.!?！？])", r"\1", s)
    return s


def normalize_for_cer(s: str) -> str:
    s = unicodedata.normalize("NFKC", s or "").lower()
    return "".join(ch for ch in s if is_kept(ch) and ch != "'")


def edit_distance(a: str, b: str) -> int:
    try:
        from rapidfuzz.distance import Levenshtein  # type: ignore

        return int(Levenshtein.distance(a, b))
    except Exception:
        pass
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]


_NUM = re.compile(r"[0-9０-９]+(?:[.,．，][0-9０-９]+)?|[〇一二三四五六七八九十百千万億兆]+")


def term_recall(ref: str, hyp: str, terms: Sequence[str]) -> Tuple[int, int]:
    """正解文に出てくる用語のうち、認識結果にも出てきた数 (hit, total)"""
    r, h = unicodedata.normalize("NFKC", ref or ""), unicodedata.normalize("NFKC", hyp or "")
    tot = hit = 0
    for t in terms:
        t = unicodedata.normalize("NFKC", t)
        n = r.count(t)
        if n:
            tot += n
            hit += min(n, h.count(t))
    return hit, tot


def number_recall(ref: str, hyp: str) -> Tuple[int, int]:
    """正解文の数字(算用数字・漢数字)が認識結果にも同じ形で出てきた数 (hit, total)。金額や日付の取り違えの目安"""
    rn = Counter(_NUM.findall(unicodedata.normalize("NFKC", ref or "")))
    hn = Counter(_NUM.findall(unicodedata.normalize("NFKC", hyp or "")))
    tot = sum(rn.values())
    hit = sum(min(c, hn.get(k, 0)) for k, c in rn.items())
    return hit, tot


def align_opcodes(ref: str, hyp: str) -> List[Tuple[str, int, int, int, int]]:
    """ref → hyp の編集手順 (tag, ref の範囲, hyp の範囲)。tag は equal / replace / delete / insert"""
    try:
        from rapidfuzz.distance import Levenshtein  # type: ignore

        return [(o.tag, o.src_start, o.src_end, o.dest_start, o.dest_end) for o in Levenshtein.opcodes(ref, hyp)]
    except Exception:
        import difflib

        return [tuple(x) for x in difflib.SequenceMatcher(None, ref, hyp, autojunk=False).get_opcodes()]  # type: ignore


def _ref_status(r: str, h: str) -> List[str]:
    """正解の1文字ごとに「そのまま残った(=) / 別の字になった(s) / 抜けた(d)」"""
    st = ["d"] * len(r)
    for tag, a0, a1, b0, b1 in align_opcodes(r, h):
        if tag == "equal":
            st[a0:a1] = ["="] * (a1 - a0)
        elif tag == "replace":
            k = min(a1 - a0, b1 - b0)
            st[a0:a0 + k] = ["s"] * k  # 長さが違う置き換えは、あまった正解側を「抜けた」とみなす
    return st


def drop_runs(ref: str, hyp: str, min_len: int = 8, bridge: int = 2) -> Tuple[int, int]:
    """正解にあるのに文字起こしからまとめて抜けた箇所の (数, 字数)。発話の読み飛ばしの目安。
    偶然一致した数文字(bridge 字まで)をはさんでいても、ひと続きの抜けとして数える"""
    r, h = normalize_for_cer(ref), normalize_for_cer(hyp)
    st = _ref_status(r, h)
    n = chars = 0
    i = 0
    while i < len(st):
        if st[i] != "d":
            i += 1
            continue
        j, lost, gap = i, 0, 0
        while j < len(st) and gap <= bridge:
            if st[j] == "d":
                lost, gap = lost + 1, 0
            else:
                gap += 1
            j += 1
        if lost >= min_len:
            n += 1
            chars += lost
        i = j
    return n, chars


_NEG = re.compile(r"ない|なかっ|なく|ません|ずに")


def negation_check(ref: str, hyp: str) -> Tuple[int, int, int]:
    """否定の言い回し(ない・なかった・なく・ません・ずに)が文字起こしでも同じ所に残ったか。
    (残った数, 正解の数, 正解に無いのに文字起こしに出た数)。「しない」→「する」のような意味の反転の目安"""
    r, h = normalize_for_cer(ref), normalize_for_cer(hyp)
    ops = align_opcodes(r, h)
    rs = _ref_status(r, h)
    hs = ["d"] * len(h)  # 文字起こし側: 正解と一致した字か
    for tag, a0, a1, b0, b1 in ops:
        if tag == "equal":
            hs[b0:b1] = ["="] * (b1 - b0)
    ref_neg = [m.span() for m in _NEG.finditer(r)]
    hit = sum(1 for a, b in ref_neg if all(x == "=" for x in rs[a:b]))
    extra = sum(1 for a, b in (m.span() for m in _NEG.finditer(h)) if not all(x == "=" for x in hs[a:b]))
    return hit, len(ref_neg), extra


def cer(ref: str, hyp: str) -> float:
    r, h = normalize_for_cer(ref), normalize_for_cer(hyp)
    if not r:
        return float("nan") if h else 0.0
    return edit_distance(r, h) / len(r)


# =====================================================================
# ASR をクリップに流す(リトライつき)。エンジン非依存
# =====================================================================


@dataclass
class RetryPolicy:
    enabled: bool = True
    without_context: bool = True  # 1) context を外して再推論
    split: bool = True  # 2) それでもダメなら静かな所で2分割して再推論
    split_min_sec: float = 8.0
    punctuates: bool = True
    max_cps: float = 18.0
    lang: str = ""  # 指定言語(文字種チェック用)


@dataclass
class ClipResult:
    clip: Clip
    text: str = ""
    language: str = ""
    flags: List[str] = field(default_factory=list)
    attempts: List[Dict[str, Any]] = field(default_factory=list)
    words: Optional[List[List[Any]]] = None  # エンジン固有のタイムスタンプ [(語, s, e)] (クリップ先頭基準)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "clip": self.clip.to_dict(),
            "text": self.text,
            "language": self.language,
            "flags": list(self.flags),
            "attempts": self.attempts,
            "words": self.words,
        }

    @staticmethod
    def from_dict(d: Dict[str, Any]) -> "ClipResult":
        return ClipResult(
            clip=Clip.from_dict(d["clip"]),
            text=d.get("text") or "",
            language=d.get("language") or "",
            flags=list(d.get("flags") or []),
            attempts=list(d.get("attempts") or []),
            words=d.get("words"),
        )


TranscribeFn = Callable[[List[Tuple[np.ndarray, str]]], List[Dict[str, Any]]]


def run_asr(
    clips: Sequence[Clip],
    get_audio: Callable[[float, float], np.ndarray],
    transcribe: TranscribeFn,
    *,
    context: str = "",
    ctx_terms: Sequence[str] = (),
    ctx_label: str = "",
    policy: Optional[RetryPolicy] = None,
    batch_size: int = 8,
    db: Optional[np.ndarray] = None,
    on_progress: Optional[Callable[[str, int, int, List[ClipResult]], None]] = None,
    done: Optional[Dict[int, ClipResult]] = None,
) -> List[ClipResult]:
    """クリップ群を ASR にかける。

    pass1: 長い順に batch_size ずつ(パディングの無駄が減る & OOM は最初に出る)
    pass2: 怪しいクリップを context なしで再推論
    pass3: まだ怪しいクリップを2分割して再推論
    最後まで怪しいものは一番マシな結果を採用して flags を残す(JSON で確認できる)
    done: 途中再開用。pass1 が済んでいるクリップの結果(clip.id → 結果)。pass1 を飛ばす
    """
    policy = policy or RetryPolicy()
    clips = list(clips)
    if not clips:
        return []
    done = dict(done or {})

    def flags_of(text: str, c: Clip) -> List[str]:
        return quality_flags(
            text, c.dur, c.speech, ctx_terms=ctx_terms, ctx_label=ctx_label,
            punctuates=policy.punctuates, max_cps=policy.max_cps, lang=policy.lang,
        )

    def run_batch(items: List[Tuple[Clip, str]]) -> List[Dict[str, Any]]:
        auds = [(get_audio(c.start, c.end), ctx) for c, ctx in items]
        outs = transcribe(auds)
        if len(outs) != len(items):
            raise RuntimeError(f"エンジンの返り値の数が合いません: {len(outs)} != {len(items)}")
        return outs

    results: Dict[int, ClipResult] = {}
    for c in clips:
        if c.id in done:
            r = done[c.id]
            r.clip = c
            r.flags = flags_of(r.text, c)  # 判定基準が変わっていても大丈夫なように付け直す
            results[c.id] = r
    order = sorted((c for c in clips if c.id not in results), key=lambda c: c.dur, reverse=True)
    bs = max(1, int(batch_size))
    n_done = len(results)
    for i in range(0, len(order), bs):
        batch = order[i: i + bs]
        outs = run_batch([(c, context) for c in batch])
        part = []
        for c, o in zip(batch, outs):
            t = (o.get("text") or "").strip()
            f = flags_of(t, c)
            r = ClipResult(c, t, o.get("language") or "", f, [{"ctx": bool(context), "text": t, "flags": f}], o.get("words"))
            results[c.id] = r
            part.append(r)
        n_done += len(batch)
        if on_progress:
            on_progress("asr", n_done, len(clips), part)

    if not policy.enabled:
        return sorted(results.values(), key=lambda r: r.clip.start)

    # pass2: context を外す
    bad = [r for r in results.values() if r.flags]
    if bad and context and policy.without_context:
        for i in range(0, len(bad), bs):
            chunk = bad[i: i + bs]
            outs = run_batch([(r.clip, "") for r in chunk])
            for r, o in zip(chunk, outs):
                t = (o.get("text") or "").strip()
                f = flags_of(t, r.clip)
                r.attempts.append({"ctx": False, "text": t, "flags": f})
                if len(f) < len(r.flags) or not f:
                    r.text, r.flags, r.words = t, f, o.get("words")
                    r.language = o.get("language") or r.language
            if on_progress:
                on_progress("retry", min(i + bs, len(bad)), len(bad), chunk)

    # pass3: 2分割
    bad = [r for r in results.values() if r.flags and policy.split and r.clip.dur >= policy.split_min_sec]
    if bad:
        next_id = max(c.id for c in clips) + 1
        subs: List[Tuple[ClipResult, List[Clip]]] = []
        for r in bad:
            sc = split_clip(r.clip, db, next_id)
            next_id += 2
            subs.append((r, sc))
        # 分割後は、より良かった方の context 設定を使う
        items: List[Tuple[Clip, str]] = []
        for r, sc in subs:
            use_ctx = context if (r.attempts and r.attempts[0]["ctx"] and not r.attempts[0]["flags"]) else ""
            for c in sc:
                items.append((c, use_ctx))
        outs_all: List[Dict[str, Any]] = []
        for i in range(0, len(items), bs):
            outs_all.extend(run_batch(items[i: i + bs]))
        k = 0
        for r, sc in subs:
            sub_results = []
            for c in sc:
                o = outs_all[k]
                k += 1
                t = (o.get("text") or "").strip()
                sub_results.append(ClipResult(c, t, o.get("language") or "", flags_of(t, c), [{"ctx": bool(items[k - 1][1]), "text": t, "flags": flags_of(t, c)}], o.get("words")))
            n_bad_sub = sum(len(s.flags) for s in sub_results)
            if n_bad_sub < len(r.flags):
                del results[r.clip.id]
                for s in sub_results:
                    s.attempts = r.attempts + [{"split_from": r.clip.id}] + s.attempts
                    results[s.clip.id] = s
            else:
                r.attempts.append({"split": [s.text for s in sub_results], "flags": [s.flags for s in sub_results]})
        if on_progress:
            on_progress("split", len(subs), len(subs), [])

    # 最後まで残ったループは縮める
    for r in results.values():
        if "repetition" in r.flags:
            fixed = collapse_repeats(r.text)
            if fixed != r.text:
                r.attempts.append({"collapsed": True})
                r.text = fixed
                r.words = None  # テキストが変わったのでエンジン側タイムスタンプは使わない
    return sorted(results.values(), key=lambda r: (r.clip.start, r.clip.id))


# =====================================================================
# 単語列の組み立て(句読点復元・重複除去)
# =====================================================================


@dataclass
class Word:
    word: str  # 表示用(句読点・空白込み)
    start: float
    end: float
    core: str = ""
    speaker: Optional[str] = None
    clip: int = -1

    def to_dict(self) -> Dict[str, Any]:
        d = {"word": self.word, "start": round(self.start, 3), "end": round(self.end, 3)}
        if self.speaker is not None:
            d["speaker"] = self.speaker
        return d


def compose_words(
    results: Sequence[ClipResult],
    aligned: Dict[int, List[List[Any]]],
    speech_segs: Sequence[Tuple[float, float]] = (),
) -> Tuple[List[Word], Dict[str, int]]:
    """クリップごとのテキスト + タイムスタンプ → 全体の単語列。

    タイムスタンプの優先順位: アライナー > エンジン固有 > 文字数からの概算
    担当区間(ハード切りのオーバーラップ)の外に中点がある単語は捨てる。
    """
    words: List[Word] = []
    stats: Counter = Counter()
    for r in results:
        c = r.clip
        if not r.text:
            continue
        toks = aligned.get(c.id)
        src = "aligner"
        if toks:
            abs_toks = [(t[0], c.start + float(t[1]), c.start + float(t[2])) for t in toks]
        elif r.words:
            abs_toks = [(t[0], c.start + float(t[1]), c.start + float(t[2])) for t in r.words]
            src = "engine"
        else:
            abs_toks = approx_tokens(r.text, c.start, c.end, speech_segs)
            src = "approx"
        stats[src] += 1
        disp = restore_display(r.text, abs_toks)
        if not disp:  # タイムスタンプが全く取れなかった → 本文を落とさないよう概算で入れる
            disp = restore_display(r.text, approx_tokens(r.text, c.start, c.end, speech_segs))
            stats["rescued"] += 1
        kept = 0
        for d in disp:
            s = min(max(d["start"], c.start), c.end)
            e = min(max(d["end"], s), c.end)
            if e - s > 4.0:
                stats["long_words"] += 1  # 不自然に長い単語(アライナーの失敗の目安)
            mid = (s + e) / 2
            if c.own_start <= mid < c.own_end:
                words.append(Word(d["word"], s, e, d["core"], None, c.id))
                kept += core_len(d["word"])
        total_chars = core_len(r.text)
        # 余白の重なりで削れるのは普通だが、半分以上消えたらアライメントの失敗を疑う
        if total_chars >= 10 and kept < total_chars * 0.5:
            stats["trimmed_much"] += 1
            if "trimmed" not in r.flags:
                r.flags.append("trimmed")
    words.sort(key=lambda w: (w.start, w.end))
    # 時刻の逆転をならす(アライナーの誤差対策)
    for i in range(1, len(words)):
        if words[i].start < words[i - 1].start:
            words[i].start = words[i - 1].start
        if words[i].end < words[i].start:
            words[i].end = words[i].start
    return words, dict(stats)


# =====================================================================
# 話者分離の結果を単語に割り当てる
# =====================================================================


def assign_speakers(words: Sequence[Word], turns: Sequence[Sequence[Any]], max_dist: float = 1.0) -> None:
    """各単語に、単語と重なっている時間が最も長い話者を付ける(重ならなければ max_dist 秒以内の最寄り)。

    話者区間どうしが重なっていても(A:0〜10秒 の中に B:1〜2秒 がある等)取りこぼさないよう、
    「いちばん長い区間の長さ」ぶん手前から候補を集める(v2 は開始時刻の近い2区間しか見ておらず誤判定があった)。
    """
    turns = sorted(((float(s), float(e), str(k)) for s, e, k in turns if float(e) > float(s)), key=lambda x: x[0])
    if not turns:
        return
    starts = [t[0] for t in turns]
    max_len = max(e - s for s, e, _ in turns)
    for w in words:
        lo = bisect.bisect_left(starts, w.start - max_len - max_dist)
        hi = bisect.bisect_right(starts, w.end + max_dist)
        best, best_ov = None, 0.0
        near, near_d = None, float("inf")
        mid = (w.start + w.end) / 2
        for j in range(lo, hi):
            s, e, k = turns[j]
            ov = min(e, w.end) - max(s, w.start)
            if ov > best_ov:
                best, best_ov = k, ov
            d = 0.0 if s <= mid <= e else min(abs(mid - s), abs(mid - e))
            if d < near_d:
                near, near_d = k, d
        if best is None and near is not None and near_d <= max_dist:
            best = near  # 長さ0の単語や無音部分 → 最寄りの話者
        w.speaker = best
    # 割り当てられなかった単語は前後の話者で埋める
    last = None
    for w in words:
        if w.speaker is None:
            w.speaker = last
        else:
            last = w.speaker
    nxt = None
    for w in reversed(words):
        if w.speaker is None:
            w.speaker = nxt
        else:
            nxt = w.speaker


def overlap_regions(turns: Sequence[Sequence[Any]], min_dur: float = 0.2) -> List[Tuple[float, float, List[str]]]:
    """(重なりありの)話者区間から、2人以上が同時に話している時間帯を出す"""
    ev: List[Tuple[float, int, str]] = []
    for s, e, k in turns:
        if float(e) > float(s):
            ev.append((float(s), 1, str(k)))
            ev.append((float(e), -1, str(k)))
    ev.sort(key=lambda x: (x[0], x[1]))
    active: Counter = Counter()
    out: List[Tuple[float, float, List[str]]] = []
    start: Optional[float] = None
    for t, d, k in ev:
        before = sum(1 for v in active.values() if v > 0)
        active[k] += d
        after = sum(1 for v in active.values() if v > 0)
        if before < 2 <= after:
            start = t
        elif before >= 2 > after and start is not None:
            if t - start >= min_dur:
                out.append((start, t, sorted(x for x, v in active.items() if v > 0) or []))
            start = None
    # 話者名は「重なっていた人たち」を入れ直す
    res = []
    for s, e, _ in out:
        who = sorted({str(k) for ts, te, k in turns if min(float(te), e) - max(float(ts), s) > 0})
        res.append((round(s, 3), round(e, 3), who))
    return res


def smooth_speakers(words: Sequence[Word], min_share: float = 0.6, max_pause: float = 1.0) -> int:
    """文単位の多数決で「文の途中で話者がチラつく」のを直す。直した単語数を返す"""
    changed = 0
    for sent in split_sentences(words, max_pause=max_pause, split_on_speaker=False):
        dur: Counter = Counter()
        for w in sent:
            dur[w.speaker] += max(0.02, w.end - w.start)
        if not dur:
            continue
        spk, d = dur.most_common(1)[0]
        if spk is not None and d / sum(dur.values()) >= min_share:
            for w in sent:
                if w.speaker != spk:
                    w.speaker = spk
                    changed += 1
    # 1〜2語だけの割り込み(前後が同じ話者)も吸収
    ws = list(words)
    i = 0
    while i < len(ws):
        j = i
        while j < len(ws) and ws[j].speaker == ws[i].speaker:
            j += 1
        if 0 < i and j < len(ws) and (j - i) <= 2 and ws[i - 1].speaker == ws[j].speaker != ws[i].speaker:
            if ws[j - 1].end - ws[i].start < 0.8:
                for k in range(i, j):
                    ws[k].speaker = ws[i - 1].speaker
                    changed += 1
        i = j
    return changed


def speaker_names(words: Sequence[Word], style: str = "話者A", mapping_spec: str = "") -> Dict[str, str]:
    """登場順に 話者A, 話者B… (style) の名前を付け、mapping_spec で上書きする。

    mapping_spec: 「SPEAKER_00=田中, SPEAKER_01=佐藤」または「話者A=田中」または「田中, 佐藤」(登場順)
    """
    order: List[str] = []
    for w in words:
        if w.speaker is not None and w.speaker not in order:
            order.append(w.speaker)
    names: Dict[str, str] = {}
    for i, spk in enumerate(order):
        if style == "SPEAKER_00":
            names[spk] = spk
        elif style == "S1":
            names[spk] = f"S{i + 1}"
        else:
            names[spk] = "話者" + (chr(ord("A") + i) if i < 26 else str(i + 1))
    spec = (mapping_spec or "").strip()
    if spec:
        items = [x.strip() for x in re.split(r"[,、，\n]+", spec) if x.strip()]
        if all(("=" in x or "＝" in x) for x in items):
            for x in items:
                k, v = re.split(r"[=＝]", x, maxsplit=1)
                k, v = k.strip(), v.strip()
                for spk, nm in list(names.items()):
                    if k in (spk, nm):
                        names[spk] = v
        else:
            for spk, v in zip(order, items):
                names[spk] = v
    return names


# =====================================================================
# 文・段落・字幕の組み立て
# =====================================================================


def _ends_sentence(w: Word, nxt: Optional[Word]) -> bool:
    t = w.word
    if _SENT_END_JA.search(t):
        return True
    if _PERIOD_END.search(t):  # 英語のピリオドは後ろに空白があるときだけ(3.5 などを割らない)
        return True
    if nxt is None and re.search(r"[.．][」』）)\]】〕”’\"']*$", t.rstrip()):
        return True
    return False


def split_sentences(
    words: Sequence[Word], max_pause: float = 1.5, max_chars: int = 150, split_on_speaker: bool = True
) -> List[List[Word]]:
    sents: List[List[Word]] = []
    cur: List[Word] = []
    n = len(words)
    for i, w in enumerate(words):
        if cur and ((w.start - cur[-1].end > max_pause) or (split_on_speaker and w.speaker != cur[-1].speaker)):
            sents.append(cur)
            cur = []
        cur.append(w)
        nxt = words[i + 1] if i + 1 < n else None
        if _ends_sentence(w, nxt) or sum(len(x.word) for x in cur) >= max_chars:
            sents.append(cur)
            cur = []
    if cur:
        sents.append(cur)
    return sents


@dataclass
class TextPost:
    """出力テキストに最後にかける整形"""

    replacements: List[Tuple[Any, str]] = field(default_factory=list)
    filler_rx: Optional[Any] = None
    fix_spaces: bool = True
    halfwidth: bool = True

    def __call__(self, s: str) -> str:
        if self.fix_spaces:
            s = fix_ja_spaces(s)
        if self.halfwidth:
            s = halfwidth_alnum(s)
        if self.replacements:
            s = apply_replacements(s, self.replacements)
        if self.filler_rx is not None:
            s = remove_fillers(s, self.filler_rx)
        return s.strip()


def join_words(ws: Sequence[Word]) -> str:
    return "".join(w.word for w in ws).strip()


@dataclass
class Segment:
    start: float
    end: float
    text: str
    speaker: Optional[str] = None
    words: List[Word] = field(default_factory=list)


def make_sentences(words: Sequence[Word], post: TextPost, max_pause: float = 1.5) -> List[Segment]:
    out = []
    for s in split_sentences(words, max_pause=max_pause):
        txt = post(join_words(s))
        if not txt:
            continue
        out.append(Segment(s[0].start, s[-1].end, txt, s[0].speaker, list(s)))
    return out


def make_paragraphs(sents: Sequence[Segment], para_gap: float = 3.0, max_chars: int = 400) -> List[Segment]:
    paras: List[Segment] = []
    for s in sents:
        p = paras[-1] if paras else None
        if p and p.speaker == s.speaker and s.start - p.end <= para_gap and len(p.text) + len(s.text) <= max_chars:
            p.text += s.text if (p.text[-1:] in "。！？!?」』" or not p.text[-1:].isascii()) else " " + s.text
            p.end = s.end
            p.words.extend(s.words)
        else:
            paras.append(Segment(s.start, s.end, s.text, s.speaker, list(s.words)))
    return paras


@dataclass
class CueRules:
    max_chars: int = 30
    max_dur: float = 6.0
    gap: float = 0.8
    min_dur: float = 0.6
    tail_pad: float = 0.2
    punct_split_min: int = 12  # 句点で区切るのはこの文字数以上たまってから


def make_cues(words: Sequence[Word], post: TextPost, rules: Optional[CueRules] = None) -> List[Segment]:
    """単語列 → 字幕(SRT/VTT)のキュー。話者交代・無音・長さ・文字数・句読点で区切る"""
    rules = rules or CueRules()
    cues: List[Segment] = []
    cur: List[Word] = []

    def length(ws: Sequence[Word]) -> int:
        return len(join_words(ws))

    def flush() -> None:
        nonlocal cur
        if cur:
            txt = post(join_words(cur))
            if txt:
                cues.append(Segment(cur[0].start, cur[-1].end, txt, cur[0].speaker, list(cur)))
        cur = []

    n = len(words)
    # 各単語から「その文の終わり」までの文字数(読点で割るかどうかの先読み用)
    rest_len = [0] * n
    acc = 0
    for i in range(n - 1, -1, -1):
        nxt = words[i + 1] if i + 1 < n else None
        if _ends_sentence(words[i], nxt) or nxt is None or nxt.start - words[i].end >= rules.gap:
            acc = 0
        rest_len[i] = acc
        acc += len(words[i].word)

    def flush_at_break() -> None:
        """長さ/時間の上限で切るときは、なるべく直前の句読点で切って残りを次へ回す(「す。」だけの字幕を防ぐ)"""
        nonlocal cur
        k = None
        for j in range(len(cur) - 1, max(0, len(cur) // 3) - 1, -1):
            nx = cur[j + 1] if j + 1 < len(cur) else None
            if _ends_sentence(cur[j], nx) or _COMMA_END.search(cur[j].word):
                k = j
                break
        if k is None or k == len(cur) - 1:
            flush()
            return
        rest = cur[k + 1:]
        cur = cur[: k + 1]
        flush()
        cur = rest

    for i, w in enumerate(words):
        if cur:
            if w.start - cur[-1].end >= rules.gap or w.speaker != cur[0].speaker:
                flush()
            elif w.end - cur[0].start > rules.max_dur or length(cur) + len(w.word.strip()) > rules.max_chars:
                flush_at_break()
                # 残りを足してもまだ上限を超えるなら、そこで切る
                if cur and (w.end - cur[0].start > rules.max_dur or length(cur) + len(w.word.strip()) > rules.max_chars):
                    flush()
        cur.append(w)
        nxt = words[i + 1] if i + 1 < n else None
        L = length(cur)
        if _ends_sentence(w, nxt) and L >= rules.punct_split_min:
            flush()
        elif _COMMA_END.search(w.word) and L + rest_len[i] > rules.max_chars and L >= max(4, int(rules.max_chars * 0.35)):
            flush()  # 文の残りが入りきらないときだけ読点で割る
    flush()

    # タイミングの後処理: 最短表示時間・少し余韻・次のキューと重ねない
    for i, c in enumerate(cues):
        nxt_start = cues[i + 1].start if i + 1 < len(cues) else float("inf")
        end = max(c.end + rules.tail_pad, c.start + rules.min_dur)
        end = min(end, nxt_start - 0.001) if nxt_start < float("inf") else end
        c.end = max(end, c.start + 0.05)
    return cues


# =====================================================================
# 書き出し
# =====================================================================


def _label(seg: Segment, names: Dict[str, str]) -> str:
    return names.get(seg.speaker, seg.speaker or "") if seg.speaker is not None else ""


def to_srt(cues: Sequence[Segment], names: Dict[str, str], speaker_fmt: str = "{speaker}: {text}") -> str:
    out = []
    for i, c in enumerate(cues, 1):
        spk = _label(c, names)
        body = speaker_fmt.format(speaker=spk, text=c.text) if spk else c.text
        out.append(f"{i}\n{fmt_srt_time(c.start)} --> {fmt_srt_time(c.end)}\n{body}\n")
    return "\n".join(out)


def to_vtt(cues: Sequence[Segment], names: Dict[str, str]) -> str:
    out = ["WEBVTT", ""]
    for c in cues:
        spk = _label(c, names)
        body = f"<v {spk}>{c.text}" if spk else c.text
        out.append(f"{fmt_vtt_time(c.start)} --> {fmt_vtt_time(c.end)}\n{body}\n")
    return "\n".join(out)


def to_txt(paras: Sequence[Segment], names: Dict[str, str], timestamps: bool = True) -> str:
    lines = []
    for p in paras:
        spk = _label(p, names)
        head = f"[{fmt_hms(p.start)}] " if timestamps else ""
        lines.append(f"{head}{spk + ': ' if spk else ''}{p.text}")
    return "\n".join(lines) + ("\n" if lines else "")


def to_markdown(paras: Sequence[Segment], names: Dict[str, str], title: str, meta: Dict[str, Any]) -> str:
    lines = [f"# {title}", ""]
    for k, v in meta.items():
        lines.append(f"- {k}: {v}")
    lines += ["", "## 文字起こし", ""]
    for p in paras:
        spk = _label(p, names)
        who = f" **{spk}**" if spk else ""
        lines.append(f"**[{fmt_hms(p.start)}]**{who}  ")
        lines.append(p.text)
        lines.append("")
    return "\n".join(lines)


def to_csv(sents: Sequence[Segment], names: Dict[str, str]) -> str:
    bio = io.StringIO()
    w = csv.writer(bio, lineterminator="\n")
    w.writerow(["start", "end", "start_hms", "speaker", "text"])
    for s in sents:
        w.writerow([f"{s.start:.3f}", f"{s.end:.3f}", fmt_hms(s.start), _label(s, names), s.text])
    return bio.getvalue()


def to_rttm(turns: Sequence[Sequence[Any]], file_id: str, names: Optional[Dict[str, str]] = None) -> str:
    fid = re.sub(r"\s+", "_", file_id) or "audio"
    lines = []
    for s, e, k in sorted(turns, key=lambda x: float(x[0])):
        spk = (names or {}).get(k, k)
        spk = re.sub(r"\s+", "_", str(spk))
        lines.append(f"SPEAKER {fid} 1 {float(s):.3f} {float(e) - float(s):.3f} <NA> <NA> {spk} <NA> <NA>")
    return "\n".join(lines) + ("\n" if lines else "")


def to_json(
    sents: Sequence[Segment],
    names: Dict[str, str],
    results: Sequence[ClipResult],
    meta: Dict[str, Any],
) -> str:
    segs = []
    for i, s in enumerate(sents):
        d: Dict[str, Any] = {"id": i, "start": round(s.start, 3), "end": round(s.end, 3), "text": s.text}
        if s.speaker is not None:
            d["speaker"] = _label(s, names)
            d["speaker_id"] = s.speaker
        d["words"] = [w.to_dict() for w in s.words]
        segs.append(d)
    clips = []
    for r in results:
        d = r.clip.to_dict()
        d.update({"text": r.text, "flags": r.flags})
        if len(r.attempts) > 1 or r.flags:
            d["attempts"] = r.attempts
        clips.append(d)
    doc = dict(meta)
    doc["speakers"] = names
    doc["segments"] = segs
    doc["clips"] = clips
    return json.dumps(doc, ensure_ascii=False, indent=1)


FLAG_JA = {
    "empty": "発話があるのに文字が出なかった",
    "context_label": "context の見出しをそのまま出力(復唱)",
    "context_echo": "用語リストをくり返しているだけに見える(復唱)",
    "too_dense": "しゃべれる速さを超える文字数(幻聴?)",
    "repetition": "同じ言葉のループ",
    "no_punct": "句読点のない長文",
    "trimmed": "タイムスタンプ付けで大きく欠けた(アライメント失敗?)",
    "foreign_script": "指定した言語にない文字(ハングル等)が混ざった",
}


def review_items(results: Sequence[ClipResult]) -> List[ClipResult]:
    """要確認リストに載せるクリップ: 最後まで怪しいもの + 自動で差し替えたもの"""
    out = []
    for r in results:
        changed = any(("split_from" in a) or ("adopted" in a) for a in r.attempts) or (
            len([a for a in r.attempts if "text" in a]) > 1 and r.attempts[0].get("text") != r.text)
        if r.flags or changed:
            out.append(r)
    return out


def _attempt_line(a: Dict[str, Any]) -> str:
    src = f"別モデル {a['model']}" if "model" in a else ("context あり" if a.get("ctx") else "context なし")
    fl = a.get("flags") or []
    mark = f"  ⚠ {'、'.join(FLAG_JA.get(f, f) for f in fl)}" if fl else ""
    return f"  - 候補({src}): {a['text'][:400] or '(空)'}{mark}"


def to_review_md(results: Sequence[ClipResult], title: str) -> str:
    items = review_items(results)
    lines = [f"# 要確認リスト: {title}", "",
             "自動チェックで怪しいと判定された区間と、自動で再推論して差し替えた区間です。"
             "元の結果も残してあるので、音声を聞いて確かめてください。", ""]
    # 2分割で置き換わったクリップは、元のクリップごとにまとめる
    groups: List[List[ClipResult]] = []
    for r in items:
        if r.clip.parent is not None and groups and groups[-1][0].clip.parent == r.clip.parent:
            groups[-1].append(r)
        else:
            groups.append([r])
    for g in groups:
        first, last = g[0], g[-1]
        flags = [f for r in g for f in r.flags]
        why = "、".join(dict.fromkeys(FLAG_JA.get(f, f) for f in flags)) if flags else "自動で差し替え済み"
        split = first.clip.parent is not None
        lines.append(f"## [{fmt_hms(first.clip.start)} 〜 {fmt_hms(last.clip.end)}] {why}"
                     + (f"(静かな所で {len(g)} つに分けて読み直し)" if split and len(g) > 1 else ""))
        lines.append(f"- **採用した結果**: {''.join(r.text for r in g) or '(空)'}")
        shown = set()
        for r in g:
            for a in r.attempts:
                if "text" not in a:
                    continue
                key = (a.get("model"), a.get("ctx"), a["text"])
                if key in shown:
                    continue
                shown.add(key)
                lines.append(_attempt_line(a))
        lines.append("")
    return "\n".join(lines)


def diff_html(a: str, b: str, max_len: int = 20000) -> str:
    """2つのテキストの差分を HTML で(削除=赤, 追加=緑)。モデル比較用"""
    import difflib
    import html

    a, b = a[:max_len], b[:max_len]
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    out = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            out.append(html.escape(a[i1:i2]))
        if op in ("delete", "replace"):
            out.append(f"<del style='background:#fdd;text-decoration:line-through'>{html.escape(a[i1:i2])}</del>")
        if op in ("insert", "replace"):
            out.append(f"<ins style='background:#dfd;text-decoration:none'>{html.escape(b[j1:j2])}</ins>")
    return "".join(out)
'''
_FILES['runtime.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.runtime — 環境(venv)づくり・ワーカープロセス・vLLMサーバー・GPU/Colab まわり

ノートブックのカーネル側で動く。モデルごとに依存がぶつかるので、エンジンは
それぞれ専用の venv の中の「ワーカー」プロセスで動かす(カーネルは汚さない=再起動いらず)。
"""
from __future__ import annotations

import hashlib
import json
import os
import queue
import shutil
import signal
import subprocess
import sys
import threading
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np

from . import core

PKG_DIR = os.path.dirname(os.path.abspath(__file__))
PKG_PARENT = os.path.dirname(PKG_DIR)
BASE_DIR = os.environ.get("ASR_V3_HOME", "/content/asr_v3")
ENV_ROOT = os.path.join(BASE_DIR, "envs")
LOG_DIR = os.path.join(BASE_DIR, "logs")
WORK_DIR = os.path.join(BASE_DIR, "work")


def _mkdirs() -> None:
    for d in (BASE_DIR, ENV_ROOT, LOG_DIR, WORK_DIR):
        os.makedirs(d, exist_ok=True)


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# =====================================================================
# GPU / Colab
# =====================================================================


@dataclass
class GPUInfo:
    name: str = ""
    mem_gb: float = 0.0
    cc: Tuple[int, int] = (0, 0)
    driver: str = ""
    cuda: str = ""  # ドライバが対応する CUDA バージョン(nvidia-smi 表示)

    @property
    def ok(self) -> bool:
        return bool(self.name)

    @property
    def ampere_plus(self) -> bool:
        return self.cc[0] >= 8


def gpu_info() -> GPUInfo:
    try:
        p = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap,driver_version", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30,
        )
        line = (p.stdout or "").strip().splitlines()[0]
        name, mem, cc, drv = [x.strip() for x in line.split(",")[:4]]
        maj, mnr = (cc.split(".") + ["0"])[:2]
        info = GPUInfo(name, float(mem) / 1024.0, (int(maj), int(mnr)), drv)
        q = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout
        import re

        m = re.search(r"CUDA Version:\s*([\d.]+)", q or "")
        info.cuda = m.group(1) if m else ""
        return info
    except Exception:
        return GPUInfo()


def auto_batch_size(g: GPUInfo, model_gb: float = 4.0, per_item_gb: float = 0.5, cap: int = 128) -> int:
    """VRAM からざっくりバッチサイズを決める(30秒クリップ想定)。T4≒8 / L4≒16 / A100-40G≒32 / 80G≒64"""
    if not g.ok:
        return 1
    free = max(1.0, g.mem_gb * 0.8 - model_gb)
    bs = int(free / per_item_gb)
    for p in (128, 64, 32, 16, 8, 4, 2, 1):
        if bs >= p:
            return min(p, cap)
    return 1


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except Exception:
        return False


def get_secret(name: str) -> Optional[str]:
    """Colab のシークレット → 環境変数 の順で探す"""
    try:
        from google.colab import userdata  # type: ignore

        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or None


# =====================================================================
# venv の用意
# =====================================================================


@dataclass
class EnvSpec:
    name: str
    packages: List[str]
    check: str = "print('ok')"  # インストール確認用の python コード(最後の行を表示)
    isolated: bool = False  # True: システムの torch を使わない完全分離(vLLM など torch を固定するもの)
    python: str = "3.12"  # isolated のときの Python
    pip_args: List[str] = field(default_factory=list)
    post: List[List[str]] = field(default_factory=list)  # 追加の pip コマンド(引数リスト)
    note: str = ""

    def digest(self) -> str:
        d = asdict(self)
        d.pop("note", None)
        return hashlib.sha1(json.dumps(d, sort_keys=True).encode()).hexdigest()[:12]


def env_dir(name: str) -> str:
    return os.path.join(ENV_ROOT, name)


def env_python(name: str) -> str:
    return os.path.join(env_dir(name), "bin", "python")


def _uv_version(uv: str) -> Tuple[int, ...]:
    try:
        out = subprocess.run([uv, "--version"], capture_output=True, text=True, timeout=30).stdout
        return tuple(int(x) for x in out.split()[1].split(".")[:3])
    except Exception:
        return (0,)


def _uv() -> str:
    """uv を返す(無いか古ければ入れる。--torch-backend などを使うので 0.8 以上)"""
    uv = shutil.which("uv")
    if uv and _uv_version(uv) >= (0, 8, 0):
        return uv
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
    cand = os.path.join(os.path.dirname(sys.executable), "uv")
    return cand if os.path.exists(cand) else (shutil.which("uv") or "uv")


def _run_logged(cmd: List[str], logf, env: Optional[Dict[str, str]] = None, timeout: Optional[float] = None) -> int:
    logf.write(f"\n$ {' '.join(cmd)}\n")
    logf.flush()
    p = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, timeout=timeout)
    return p.returncode


@dataclass
class EnvStatus:
    name: str
    ok: bool
    seconds: float = 0.0
    info: str = ""
    log: str = ""
    skipped: bool = False


def ensure_env(spec: EnvSpec, force: bool = False) -> EnvStatus:
    """venv を作って packages を入れる。前回と同じ内容なら確認だけしてスキップ"""
    _mkdirs()
    d = env_dir(spec.name)
    marker = os.path.join(d, ".asr_v3_env.json")
    logp = os.path.join(LOG_DIR, f"env_{spec.name}.log")
    t0 = time.time()
    if not force and os.path.exists(marker):
        try:
            if json.load(open(marker)).get("digest") == spec.digest():
                ok, info = check_env(spec)
                if ok:
                    return EnvStatus(spec.name, True, time.time() - t0, info, logp, skipped=True)
        except Exception:
            pass
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)
    uv = _uv()
    env = dict(os.environ)
    env.setdefault("UV_LINK_MODE", "copy")
    with open(logp, "a", encoding="utf-8") as logf:
        logf.write(f"\n===== {time.ctime()} {spec.name} =====\n")
        if spec.isolated:
            rc = _run_logged([uv, "venv", "--seed", "-p", spec.python, d], logf, env)
            if rc == 0:
                rc = _run_logged([uv, "pip", "install", "--python", env_python(spec.name), *spec.pip_args, *spec.packages], logf, env)
        else:
            # システムの torch 等をそのまま使う venv(pip はシステムにある物を「入っている」とみなしてくれる)
            rc = _run_logged([uv, "venv", "--seed", "--system-site-packages", "-p", sys.executable, d], logf, env)
            if rc == 0:
                rc = _run_logged([env_python(spec.name), "-m", "pip", "install", "--progress-bar", "off",
                                  *spec.pip_args, *spec.packages], logf, env)
        for extra in spec.post:
            if rc != 0:
                break
            # post は失敗しても致命的でない(flash-attn など)
            _run_logged([env_python(spec.name), "-m", "pip", "install", "--progress-bar", "off", *extra], logf, env)
    if rc != 0:
        return EnvStatus(spec.name, False, time.time() - t0, _tail(logp), logp)
    if not spec.isolated:
        _fix_google_namespace(spec.name)
    ok, info = check_env(spec)
    if ok:
        with open(marker, "w") as f:
            json.dump({"digest": spec.digest(), "time": time.time(), "info": info}, f)
    # 失敗時: インストールログの末尾 → 確認(import)のエラー の順(表示は末尾 3000 字なので、肝心のエラーを最後に)
    return EnvStatus(spec.name, ok, time.time() - t0,
                     info if ok else _tail(logp, 15) + "\n--- 確認(import)でのエラー ---\n" + info, logp)


_GOOGLE_NS_PTH = (
    "import sys, os, importlib.util, importlib.machinery; "
    "_d = sys._getframe(1).f_locals.get('sitedir') or ''; "
    "_s = importlib.machinery.PathFinder.find_spec('google', [_d]) if os.path.isdir(os.path.join(_d, 'google')) else None; "
    "_s and ('google' not in sys.modules) and sys.modules.__setitem__('google', importlib.util.module_from_spec(_s))\n"
)


def _fix_google_namespace(name: str) -> None:
    """システムの site-packages を見る venv で、venv に入れた google.* (protobuf など)が隠れないようにする

    Colab には google_generativeai の *-nspkg.pth があり、起動時に `google` をシステム側のパスだけで作ってしまう。
    すると venv に入れた新しい protobuf ではなくシステムの古い protobuf が読まれ、NeMo(onnx)が
    「gencode 6.x / runtime 5.x」で落ちる。venv 側の .pth で先に venv の google/ を登録しておく
    (システム側の nspkg.pth はそこへ自分のパスを足すだけになる)。
    """
    import glob

    for sp in glob.glob(os.path.join(env_dir(name), "lib", "python3*", "site-packages")):
        pth = os.path.join(sp, "_asr_v3_google_ns.pth")
        if os.path.isdir(os.path.join(sp, "google")) and not os.path.exists(os.path.join(sp, "google", "__init__.py")):
            with open(pth, "w") as f:
                f.write(_GOOGLE_NS_PTH)
        elif os.path.exists(pth):
            os.remove(pth)


def check_env(spec: EnvSpec) -> Tuple[bool, str]:
    py = env_python(spec.name)
    if not os.path.exists(py):
        return False, "python がありません"
    p = subprocess.run([py, "-c", spec.check], capture_output=True, text=True, timeout=600)
    out = (p.stdout or "").strip().splitlines()
    if p.returncode != 0:
        return False, (p.stderr or "")[-1500:]
    return True, out[-1] if out else "ok"


def _tail(path: str, n: int = 40) -> str:
    try:
        with open(path, encoding="utf-8", errors="replace") as f:
            return "".join(deque(f, maxlen=n))
    except Exception:
        return ""


def ensure_envs(specs: Sequence[EnvSpec], parallel: int = 3, force: bool = False) -> List[EnvStatus]:
    """複数の venv を並列に用意(経過を表示)"""
    specs = list(specs)
    if not specs:
        return []
    res: Dict[str, EnvStatus] = {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max(1, parallel)) as ex:
        futs = {ex.submit(ensure_env, s, force): s for s in specs}
        pending = set(futs)
        last = 0.0
        while pending:
            done = [f for f in pending if f.done()]
            for f in done:
                s = futs[f]
                pending.discard(f)
                try:
                    st = f.result()
                except Exception as e:  # pragma: no cover
                    st = EnvStatus(s.name, False, 0, repr(e))
                res[s.name] = st
                mark = "✅" if st.ok else "❌"
                how = "(前回のまま)" if st.skipped else f"({core.fmt_dur(st.seconds)})"
                log(f"{mark} 環境 {s.name} {how} {st.info.splitlines()[0] if st.ok and st.info else ''}")
                if not st.ok:
                    print(st.info[-3000:])
            if pending and time.time() - last > 20:
                last = time.time()
                names = ", ".join(futs[f].name for f in pending)
                log(f"… インストール中: {names} (経過 {core.fmt_dur(time.time() - t0)})")
            time.sleep(0.5)
    return [res[s.name] for s in specs]


# =====================================================================
# ワーカー(venv の中で asrkit.worker を動かし、JSON 1行ずつでやりとり)
# =====================================================================


class WorkerError(RuntimeError):
    pass


class Worker:
    def __init__(self, env_name: str, extra_env: Optional[Dict[str, str]] = None, echo: bool = False):
        _mkdirs()
        self.env_name = env_name
        py = env_python(env_name)
        if not os.path.exists(py):
            raise WorkerError(f"環境 {env_name} がありません。セットアップのセルで入れてください。")
        env = dict(os.environ)
        env.update(extra_env or {})
        env["PYTHONPATH"] = PKG_PARENT + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
        env["PYTHONUNBUFFERED"] = "1"
        env.setdefault("TOKENIZERS_PARALLELISM", "false")
        env.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
        self.log_path = os.path.join(LOG_DIR, f"worker_{env_name}.log")
        self._log = open(self.log_path, "a", encoding="utf-8")
        self._log.write(f"\n===== start {time.ctime()} =====\n")
        self.proc = subprocess.Popen(
            [py, "-u", "-m", "asrkit.worker"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, encoding="utf-8", errors="replace", bufsize=1, env=env,
        )
        self.q: "queue.Queue[Optional[Dict[str, Any]]]" = queue.Queue()
        self.err: deque = deque(maxlen=400)
        self.echo = echo
        self._id = 0
        self._lock = threading.Lock()
        threading.Thread(target=self._read_out, daemon=True).start()
        threading.Thread(target=self._read_err, daemon=True).start()
        self.loaded: Dict[str, Dict[str, Any]] = {}

    def _read_out(self) -> None:
        assert self.proc.stdout is not None
        for line in self.proc.stdout:
            line = line.strip()
            if not line:
                continue
            try:
                self.q.put(json.loads(line))
            except Exception:
                self.err.append("[stdout] " + line)
        self.q.put(None)

    def _read_err(self) -> None:
        assert self.proc.stderr is not None
        for line in self.proc.stderr:
            self.err.append(line.rstrip("\n"))
            try:
                self._log.write(line)
                self._log.flush()
            except Exception:
                pass
            if self.echo:
                print(line, end="")

    def alive(self) -> bool:
        return self.proc.poll() is None

    def tail(self, n: int = 60) -> str:
        return "\n".join(list(self.err)[-n:])

    def death_hint(self) -> str:
        rc = self.proc.poll()
        if rc in (-9, 137):
            return ("(強制終了されました。メモリ不足(OOM)の可能性が高いです。"
                    "ランタイムをハイメモリにするか、⑧で VRAM を解放してから、読み込むモデルを減らしてください)")
        if rc in (-11, 139):
            return "(セグメンテーション違反で落ちました。ライブラリの組み合わせの問題の可能性があります)"
        return f"(終了コード {rc})"

    def call(self, cmd: str, *, on_event: Optional[Callable[[Dict[str, Any]], None]] = None,
             timeout: Optional[float] = None, **payload: Any) -> Dict[str, Any]:
        with self._lock:
            self._id += 1
            rid = self._id
            msg = {"id": rid, "cmd": cmd, **payload}
            if not self.alive():
                raise WorkerError(f"ワーカー({self.env_name})は終了しています\n{self.tail()}")
            assert self.proc.stdin is not None
            self.proc.stdin.write(json.dumps(msg, ensure_ascii=False) + "\n")
            self.proc.stdin.flush()
            t0 = time.time()
            try:
                while True:
                    try:
                        m = self.q.get(timeout=1.0)
                    except queue.Empty:
                        if not self.alive():
                            raise WorkerError(f"ワーカー({self.env_name})が落ちました {self.death_hint()}\n{self.tail()}")
                        if timeout and time.time() - t0 > timeout:
                            raise WorkerError(f"ワーカー({self.env_name})がタイムアウトしました")
                        continue
                    if m is None:
                        try:
                            self.proc.wait(timeout=5)
                        except Exception:
                            pass
                        raise WorkerError(f"ワーカー({self.env_name})が落ちました {self.death_hint()}\n{self.tail()}")
                    if m.get("id") != rid:
                        continue
                    if m.get("event"):
                        if on_event:
                            on_event(m)
                        continue
                    if m.get("ok"):
                        return m
                    err = m.get("error", "?")
                    if err == "cancelled":
                        raise KeyboardInterrupt
                    raise WorkerError(f"{err}\n{m.get('traceback', '')}\n--- ワーカーのログ(末尾) ---\n{self.tail(30)}")
            except KeyboardInterrupt:
                self._cancel(rid)
                raise

    def _cancel(self, rid: int) -> None:
        """セルの停止ボタン → ワーカーにも SIGINT を送って今の処理だけ止める(モデルは残す)"""
        if not self.alive():
            return
        try:
            self.proc.send_signal(signal.SIGINT)
        except Exception:
            return
        t0 = time.time()
        while time.time() - t0 < 20:
            try:
                m = self.q.get(timeout=1.0)
            except queue.Empty:
                continue
            if m is None or (m.get("id") == rid and not m.get("event")):
                return
        log(f"⚠️ ワーカー({self.env_name})が止まらないので終了させます")
        self.close(force=True)

    def close(self, force: bool = False) -> None:
        if self.alive() and not force:
            try:
                assert self.proc.stdin is not None
                self.proc.stdin.write(json.dumps({"id": 0, "cmd": "exit"}) + "\n")
                self.proc.stdin.flush()
                self.proc.wait(timeout=20)
            except Exception:
                pass
        if self.alive():
            self.proc.kill()
            try:
                self.proc.wait(timeout=10)
            except Exception:
                pass
        try:
            self._log.close()
        except Exception:
            pass


# =====================================================================
# vLLM サーバー(OpenAI 互換の /v1/audio/transcriptions を叩く)
# =====================================================================


class VLLMServer:
    def __init__(self, env_name: str, model: str, *, port: int = 8791, gpu_mem: float = 0.5,
                 max_model_len: Optional[int] = None, extra_args: Sequence[str] = (),
                 extra_env: Optional[Dict[str, str]] = None):
        self.env_name, self.model, self.port = env_name, model, port
        self.gpu_mem, self.max_model_len = gpu_mem, max_model_len
        self.extra_args = list(extra_args)
        self.extra_env = dict(extra_env or {})
        self.proc: Optional[subprocess.Popen] = None
        self.start_sec: Optional[float] = None  # 起動にかかった秒数
        self.log_path = os.path.join(LOG_DIR, f"vllm_{model.replace('/', '__')}.log")
        self.base = f"http://127.0.0.1:{port}"

    @property
    def key(self) -> str:
        return json.dumps([self.model, self.gpu_mem, self.max_model_len, self.extra_args])

    def alive(self) -> bool:
        return self.proc is not None and self.proc.poll() is None

    def start(self, timeout: float = 2400) -> None:
        import urllib.request

        _mkdirs()
        exe = os.path.join(env_dir(self.env_name), "bin", "vllm")
        if not os.path.exists(exe):
            raise WorkerError("vLLM の環境がありません。セットアップのセルで vLLM にチェックを入れてください。")
        args = [exe, "serve", self.model, "--host", "127.0.0.1", "--port", str(self.port),
                "--served-model-name", "asr", "--gpu-memory-utilization", f"{self.gpu_mem:.2f}"]
        if self.max_model_len:
            args += ["--max-model-len", str(self.max_model_len)]
        args += self.extra_args
        env = dict(os.environ)
        env.update(self.extra_env)
        env.setdefault("VLLM_LOGGING_LEVEL", "INFO")
        # venv の bin(ninja など)を PATH に。FlashInfer のサンプラーは初回に JIT ビルドするが、
        # Colab の nvcc(12.8)と vLLM の torch(cu13x)が合わず失敗しやすいので、PyTorch 版のサンプラーを使う
        env["PATH"] = os.path.join(env_dir(self.env_name), "bin") + os.pathsep + env.get("PATH", "")
        env.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
        logf = open(self.log_path, "a", encoding="utf-8")
        logf.write(f"\n===== {time.ctime()} {' '.join(args)} =====\n")
        logf.flush()
        # 親(カーネル)が死んだら道連れにする prctl をしてから exec するラッパー経由で起動
        wrap = ("import ctypes,os,signal,sys\n"
                "try: ctypes.CDLL('libc.so.6').prctl(1, signal.SIGTERM)\n"
                "except Exception: pass\n"
                "os.execvp(sys.argv[1], sys.argv[1:])")
        self.proc = subprocess.Popen([sys.executable, "-c", wrap, *args], stdout=logf, stderr=subprocess.STDOUT, env=env)
        t0, last = time.time(), 0.0
        while True:
            if not self.alive():
                raise WorkerError(f"vLLM サーバーが起動に失敗しました。\n{_tail(self.log_path, 60)}")
            try:
                with urllib.request.urlopen(self.base + "/health", timeout=5) as r:
                    if r.status == 200:
                        self.start_sec = round(time.time() - t0, 1)
                        log(f"✅ vLLM サーバー起動 ({core.fmt_dur(self.start_sec)}): {self.model}")
                        return
            except Exception:
                pass
            if time.time() - t0 > timeout:
                self.stop()
                raise WorkerError(f"vLLM サーバーの起動がタイムアウトしました\n{_tail(self.log_path, 40)}")
            if time.time() - last > 30:
                last = time.time()
                lines = [l for l in _tail(self.log_path, 5).splitlines() if l.strip()]
                log(f"… vLLM 起動待ち {core.fmt_dur(time.time() - t0)}: {lines[-1][-160:] if lines else ''}")
            time.sleep(2)

    def _one(self, audio: np.ndarray, ctx: str, language: Optional[str], temperature: float) -> Dict[str, Any]:
        import requests

        data = {"model": "asr", "response_format": "json", "temperature": str(temperature)}
        if language:
            data["language"] = language
        if ctx:
            data["prompt"] = ctx
        body = core.wav_bytes(audio)
        err: Optional[Exception] = None
        for i in range(4):
            try:
                r = requests.post(self.base + "/v1/audio/transcriptions", data=data,
                                  files={"file": ("clip.wav", body, "audio/wav")}, timeout=900)
                if r.status_code >= 400:
                    raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
                return {"text": (r.json().get("text") or "").strip(), "language": language or ""}
            except Exception as e:  # 一時的な失敗は少し待って再送
                err = e
                if not self.alive():
                    break
                time.sleep(1.5 * (i + 1))
        raise WorkerError(f"vLLM への送信に失敗: {err}\n{_tail(self.log_path, 20)}")

    def transcribe_fn(self, language: Optional[str], concurrency: int = 64, temperature: float = 0.0) -> core.TranscribeFn:
        def fn(items: List[Tuple[np.ndarray, str]]) -> List[Dict[str, Any]]:
            with ThreadPoolExecutor(max_workers=max(1, min(concurrency, len(items)))) as ex:
                futs = [ex.submit(self._one, a, c, language, temperature) for a, c in items]
                return [f.result() for f in futs]

        return fn

    def stop(self) -> None:
        if self.proc is not None and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=30)
            except Exception:
                self.proc.kill()
        self.proc = None


# =====================================================================
# ハンドルの管理(モデルは使い回す / 切り替え時は解放)
# =====================================================================


class Handles:
    """ワーカーと vLLM サーバーをまとめて持つ。カーネルに1つだけ置いて使い回す"""

    def __init__(self) -> None:
        self.workers: Dict[str, Worker] = {}
        self.vllm: Optional[VLLMServer] = None

    def worker(self, env_name: str, extra_env: Optional[Dict[str, str]] = None) -> Worker:
        w = self.workers.get(env_name)
        if w is not None and w.alive():
            return w
        if w is not None:
            log(f"⚠️ ワーカー({env_name})が落ちていたので起動しなおします。末尾ログ:\n{w.tail(15)}")
        w = Worker(env_name, extra_env)
        self.workers[env_name] = w
        return w

    def ensure_loaded(self, env_name: str, key: str, kind: str, options: Dict[str, Any],
                      extra_env: Optional[Dict[str, str]] = None, exclusive_group: Optional[str] = None) -> Dict[str, Any]:
        """ワーカーにエンジンを読み込ませる(同じ設定なら何もしない)。
        exclusive_group が同じエンジンは同時に1つだけ(ASR モデルの切り替えで前のを解放)"""
        w = self.worker(env_name, extra_env)
        cur = w.loaded.get(key)
        if cur is not None and cur.get("options") == options and cur.get("kind") == kind:
            return dict(cur.get("info", {}), reused=True)  # 読み込み済み(load_sec は前に読み込んだときの値)
        if exclusive_group:
            for k, v in list(w.loaded.items()):
                if v.get("group") == exclusive_group and k != key:
                    self.unload(env_name, k)
        if cur is not None:
            self.unload(env_name, key)
        t0 = time.time()
        log(f"モデル読み込み中: {options.get('model', kind)} ({env_name})")
        r = w.call("load", key=key, kind=kind, options=options)
        info = r.get("info", {})
        w.loaded[key] = {"kind": kind, "options": options, "info": info, "group": exclusive_group}
        log(f"✅ 読み込み完了 ({core.fmt_dur(time.time() - t0)}) {info.get('summary', '')}")
        return info

    def unload(self, env_name: str, key: str) -> None:
        w = self.workers.get(env_name)
        if w is None or not w.alive():
            return
        if key in w.loaded:
            try:
                w.call("unload", key=key)
            except Exception:
                pass
            w.loaded.pop(key, None)

    def vllm_server(self, env_name: str, model: str, **kw: Any) -> VLLMServer:
        s = VLLMServer(env_name, model, **kw)
        if self.vllm is not None and self.vllm.alive() and self.vllm.key == s.key:
            return self.vllm
        if self.vllm is not None:
            self.vllm.stop()
        s.start()
        self.vllm = s
        return s

    def stop_vllm(self) -> None:
        if self.vllm is not None:
            self.vllm.stop()
            self.vllm = None

    def close_all(self) -> None:
        self.stop_vllm()
        for w in list(self.workers.values()):
            w.close()
        self.workers.clear()

    def status(self) -> List[str]:
        out = []
        for name, w in self.workers.items():
            out.append(f"{name}: {'稼働中' if w.alive() else '停止'} / " + ", ".join(v['options'].get('model', k) for k, v in w.loaded.items()))
        if self.vllm is not None:
            out.append(f"vLLM: {'稼働中' if self.vllm.alive() else '停止'} / {self.vllm.model}")
        return out
'''
_FILES['engines.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.engines — ワーカー(各 venv)の中で動く、モデルごとのアダプタ

どのエンジンも transcribe(items, language) -> [{"text", "language", "words"?}] をそろえる。
items は [(float32 16kHz の波形, context 文字列), ...]。words はクリップ先頭基準の [(語, 開始, 終了)]。
重い import (torch など) はクラスの中でだけ行う。
"""
from __future__ import annotations

import gc
import glob
import os
import sys
import tempfile
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

from . import core

# Qwen3-ASR/アライナーの言語名 ⇔ ISO 639-1
LANG_CODES = {
    "Japanese": "ja", "English": "en", "Chinese": "zh", "Cantonese": "yue", "Korean": "ko", "French": "fr",
    "German": "de", "Spanish": "es", "Portuguese": "pt", "Italian": "it", "Russian": "ru", "Arabic": "ar",
    "Indonesian": "id", "Thai": "th", "Vietnamese": "vi", "Turkish": "tr", "Hindi": "hi", "Malay": "ms",
    "Dutch": "nl", "Swedish": "sv", "Danish": "da", "Finnish": "fi", "Polish": "pl", "Czech": "cs",
    "Filipino": "fil", "Persian": "fa", "Greek": "el", "Romanian": "ro", "Hungarian": "hu", "Macedonian": "mk",
}
CODE_LANGS = {v: k for k, v in LANG_CODES.items()}
ALIGNER_LANGS = {"Chinese", "English", "Cantonese", "French", "German", "Italian", "Japanese", "Korean",
                 "Portuguese", "Russian", "Spanish"}


def lang_code(language: Optional[str]) -> Optional[str]:
    if not language:
        return None
    return LANG_CODES.get(language, language if len(language) <= 3 else None)


def lang_name(code_or_name: Optional[str]) -> str:
    if not code_or_name:
        return ""
    s = str(code_or_name).split(",")[0].strip()
    if s in LANG_CODES:
        return s
    return CODE_LANGS.get(s.lower(), s[:1].upper() + s[1:].lower())


def _pad_min(a: np.ndarray, min_sec: float = 0.5) -> np.ndarray:
    n = int(min_sec * core.SR)
    a = np.asarray(a, np.float32)
    if a.shape[0] < n:
        a = np.pad(a, (0, n - a.shape[0]))
    return a


# ---------------------------------------------------------------- torch まわり


def torch_mod():
    import torch

    return torch


def pick_dtype(dtype: str = "auto"):
    torch = torch_mod()
    if dtype in (None, "", "auto"):
        if torch.cuda.is_available():
            return torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        return torch.float32
    return {"bf16": torch.bfloat16, "bfloat16": torch.bfloat16, "fp16": torch.float16,
            "float16": torch.float16, "fp32": torch.float32, "float32": torch.float32}[dtype]


def pick_attn(attn: str = "auto") -> str:
    if attn and attn != "auto":
        return attn
    torch = torch_mod()
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
        try:
            import flash_attn  # noqa: F401

            return "flash_attention_2"
        except Exception:
            pass
    return "sdpa"


def device_str() -> str:
    torch = torch_mod()
    return "cuda:0" if torch.cuda.is_available() else "cpu"


def free_cuda() -> None:
    gc.collect()
    try:
        torch = torch_mod()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def is_oom(e: BaseException) -> bool:
    msg = str(e).lower()
    return "out of memory" in msg or "cuda error: out of memory" in msg or type(e).__name__ == "OutOfMemoryError"


def reset_gpu_peak() -> None:
    """VRAM 峰の計測をリセット(同じワーカーでモデルを入れ替えても、前のモデルの峰を引きずらないように)

    torch をまだ読んでいないワーカーでは何もしない(faster-whisper などの CUDA ライブラリの読み込み順を変えないため)
    """
    torch = sys.modules.get("torch")
    if torch is None:
        return
    try:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass


def gpu_peak_gb() -> float:
    try:
        torch = torch_mod()
        if torch.cuda.is_available():
            return round(torch.cuda.max_memory_allocated() / 1e9, 2)
    except Exception:
        pass
    return 0.0


def with_oom_backoff(fn, items: List[Any], bs: int, set_bs=None, min_bs: int = 1, log=None) -> List[Any]:
    """CUDA OOM が出たらバッチを半分にしてやり直す(T4 などで便利)"""
    out: List[Any] = []
    i = 0
    while i < len(items):
        chunk = items[i: i + bs]
        try:
            if set_bs:
                set_bs(bs)
            out.extend(fn(chunk))
            i += len(chunk)
        except Exception as e:
            if not is_oom(e) or bs <= min_bs:
                raise
            free_cuda()
            bs = max(min_bs, bs // 2)
            if log:
                log(f"CUDA OOM → バッチを {bs} に下げて再試行")
    return out


def _log(msg: str) -> None:
    print(f"[engine] {msg}", file=sys.stderr, flush=True)


# =====================================================================
# エンジン本体
# =====================================================================


class Engine:
    kind = "base"
    punctuates = True
    native_timestamps = False
    uses_context = True

    def __init__(self, **opt: Any) -> None:
        self.opt = opt
        self.batch_size = int(opt.get("batch_size") or 8)

    def info(self) -> Dict[str, Any]:
        return {"summary": f"{self.kind}:{self.opt.get('model')}", "punctuates": self.punctuates,
                "native_timestamps": self.native_timestamps, "uses_context": self.uses_context,
                "gpu_peak_gb": gpu_peak_gb()}

    def transcribe(self, items: List[Tuple[np.ndarray, str]], language: Optional[str]) -> List[Dict[str, Any]]:
        raise NotImplementedError

    def close(self) -> None:
        free_cuda()


class QwenASREngine(Engine):
    """Qwen3-ASR (qwen-asr パッケージ / transformers バックエンド)"""

    kind = "qwen"

    def __init__(self, model: str = "Qwen/Qwen3-ASR-1.7B", dtype: str = "auto", attn: str = "auto",
                 batch_size: int = 8, max_new_tokens: int = 1024, **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        from qwen_asr import Qwen3ASRModel

        self.attn = pick_attn(attn)
        self.dtype = pick_dtype(dtype)
        kw = dict(dtype=self.dtype, device_map=device_str(), max_inference_batch_size=self.batch_size,
                  max_new_tokens=int(max_new_tokens))
        if self.attn:
            kw["attn_implementation"] = self.attn
        try:
            self.m = Qwen3ASRModel.from_pretrained(model, **kw)
        except Exception as e:
            if self.attn == "flash_attention_2":
                _log(f"flash_attention_2 で失敗 → sdpa で再試行: {e}")
                kw["attn_implementation"] = self.attn = "sdpa"
                self.m = Qwen3ASRModel.from_pretrained(model, **kw)
            else:
                raise

    def info(self) -> Dict[str, Any]:
        d = super().info()
        d["summary"] = f"{self.opt.get('model')} attn={self.attn} dtype={str(self.dtype).replace('torch.', '')}"
        return d

    def _run(self, chunk: List[Tuple[np.ndarray, str]], language: Optional[str]) -> List[Dict[str, Any]]:
        auds = [(_pad_min(a), core.SR) for a, _ in chunk]
        ctxs = [c or "" for _, c in chunk]
        langs = [language] * len(chunk) if language else None
        res = self.m.transcribe(audio=auds, context=ctxs, language=langs, return_time_stamps=False)
        return [{"text": r.text or "", "language": r.language or (language or "")} for r in res]

    def transcribe(self, items, language):
        def set_bs(b):
            self.m.max_inference_batch_size = b

        return with_oom_backoff(lambda ch: self._run(ch, language), items, self.batch_size, set_bs, log=_log)

    def close(self) -> None:
        del self.m
        super().close()


class QwenAligner:
    """Qwen3-ForcedAligner-0.6B。どのASRの結果にも単語タイムスタンプを付けられる(11言語)"""

    kind = "aligner"

    def __init__(self, model: str = "Qwen/Qwen3-ForcedAligner-0.6B", dtype: str = "auto", attn: str = "auto",
                 batch_size: int = 8, **opt: Any) -> None:
        from qwen_asr import Qwen3ForcedAligner

        self.opt = dict(model=model, **opt)
        self.batch_size = int(batch_size or 8)
        self.attn = pick_attn(attn)
        kw = dict(dtype=pick_dtype(dtype), device_map=device_str())
        if self.attn:
            kw["attn_implementation"] = self.attn
        try:
            self.m = Qwen3ForcedAligner.from_pretrained(model, **kw)
        except Exception as e:
            if self.attn == "flash_attention_2":
                _log(f"aligner: flash_attention_2 で失敗 → sdpa: {e}")
                kw["attn_implementation"] = self.attn = "sdpa"
                self.m = Qwen3ForcedAligner.from_pretrained(model, **kw)
            else:
                raise

    def info(self) -> Dict[str, Any]:
        return {"summary": f"{self.opt.get('model')} attn={self.attn}", "gpu_peak_gb": gpu_peak_gb()}

    def align(self, items: List[Tuple[np.ndarray, str, str]]) -> List[List[Tuple[str, float, float]]]:
        """items: [(波形, テキスト, 言語名)] → [[(語, 開始, 終了)], ...](クリップ先頭基準)"""

        def run(chunk):
            res = self.m.align(audio=[(_pad_min(a), core.SR) for a, _, _ in chunk],
                               text=[t for _, t, _ in chunk], language=[l for _, _, l in chunk])
            return [[(it.text, float(it.start_time), float(it.end_time)) for it in r.items] for r in res]

        return with_oom_backoff(run, items, self.batch_size, log=_log)

    def close(self) -> None:
        del self.m
        free_cuda()


def _preload_nvidia_libs() -> None:
    """pip の nvidia-*-cu12 に入っている cuBLAS/cuDNN を先に読み込む(CTranslate2 用)"""
    import ctypes

    for sp in sys.path:
        for pat in ("nvidia/cublas/lib/libcublas*.so*", "nvidia/cudnn/lib/libcudnn*.so*", "nvidia/cuda_runtime/lib/libcudart*.so*"):
            for f in sorted(glob.glob(os.path.join(sp, pat))):
                try:
                    ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass


class FasterWhisperEngine(Engine):
    """Whisper 系 (faster-whisper / CTranslate2)。kotoba-whisper の -faster 版もこれ"""

    kind = "faster-whisper"
    native_timestamps = True

    def __init__(self, model: str = "large-v3-turbo", compute_type: str = "auto", beam_size: int = 5,
                 batch_size: int = 8, word_timestamps: Any = "auto", chunk_length: Optional[int] = None,
                 **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        # kotoba-whisper は公式の使い方が chunk_length=15(15 秒の窓で読む)
        self.chunk_length = int(chunk_length) if chunk_length else None
        # 蒸留モデル(kotoba-whisper / distil-whisper。デコーダが 2 層)は、変換時に入った large-v3 用の
        # alignment_heads が存在しない層を指していて、単語タイムスタンプ(find_alignment)で segfault する。
        # その場合は単語時刻を出さず、Qwen3-ForcedAligner でタイムスタンプを付ける
        if word_timestamps == "auto":
            word_timestamps = not any(k in model.lower() for k in ("kotoba", "distil"))
        self.word_ts = bool(word_timestamps)
        self.native_timestamps = self.word_ts
        _preload_nvidia_libs()
        from faster_whisper import WhisperModel

        torch = None
        try:
            torch = torch_mod()
        except Exception:
            pass
        cuda = bool(torch and torch.cuda.is_available())
        if compute_type == "auto":
            compute_type = "float16" if cuda else "int8"
        self.beam = int(beam_size)
        self.m = WhisperModel(model, device="cuda" if cuda else "cpu", compute_type=compute_type)
        self.ct = compute_type

    def transcribe(self, items, language):
        out = []
        code = lang_code(language)
        for a, ctx in items:
            segs, info = self.m.transcribe(
                np.asarray(a, np.float32), language=code, beam_size=self.beam, initial_prompt=(ctx or None),
                word_timestamps=self.word_ts, vad_filter=False, condition_on_previous_text=False,
                temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0), compression_ratio_threshold=2.4,
                log_prob_threshold=-1.0, no_speech_threshold=0.6,
                **({"chunk_length": self.chunk_length} if self.chunk_length else {}),
            )
            segs = list(segs)
            text = "".join(s.text for s in segs).strip()
            words = [[w.word, float(w.start), float(w.end)] for s in segs for w in (s.words or [])]
            out.append({"text": text, "language": lang_name(getattr(info, "language", "") or code or ""), "words": words})
        return out

    def close(self) -> None:
        del self.m
        super().close()


class NemoEngine(Engine):
    """NVIDIA NeMo の ASR (例: nvidia/parakeet-tdt_ctc-0.6b-ja)。文字単位のタイムスタンプあり"""

    kind = "nemo"
    native_timestamps = True
    uses_context = False

    def __init__(self, model: str = "nvidia/parakeet-tdt_ctc-0.6b-ja", batch_size: int = 16, **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        import nemo.collections.asr as nemo_asr

        self.m = nemo_asr.models.ASRModel.from_pretrained(model_name=model)
        torch = torch_mod()
        if torch.cuda.is_available():
            self.m = self.m.cuda()
        self.m.eval()
        self.tmp = tempfile.mkdtemp(prefix="nemo_clips_")

    def _run(self, chunk):
        paths = []
        for i, (a, _) in enumerate(chunk):
            p = os.path.join(self.tmp, f"{i}.wav")
            core.write_wav16(p, _pad_min(a))
            paths.append(p)
        try:
            hyps = self.m.transcribe(paths, batch_size=len(paths), timestamps=True, verbose=False)
        except TypeError:
            hyps = self.m.transcribe(paths, batch_size=len(paths), return_hypotheses=True)
        if isinstance(hyps, tuple):  # 古い RNNT 系は (best, all)
            hyps = hyps[0]
        out = []
        for h in hyps:
            text = getattr(h, "text", h if isinstance(h, str) else "") or ""
            ts = getattr(h, "timestamp", None) or {}
            words = None
            unit = ts.get("word") if isinstance(ts, dict) else None
            if unit and " " in text.strip() and len(unit) > 1:  # 空白で区切る言語は単語単位
                words = [[u.get("word", ""), float(u.get("start", 0)), float(u.get("end", 0))] for u in unit]
            elif isinstance(ts, dict) and ts.get("char"):  # 日本語などは文字単位
                words = [[u.get("char", ""), float(u.get("start", 0)), float(u.get("end", 0))] for u in ts["char"]]
            out.append({"text": text, "language": "", "words": words})
        return out

    def transcribe(self, items, language):
        return with_oom_backoff(self._run, items, self.batch_size, log=_log)

    def close(self) -> None:
        del self.m
        super().close()


class HFPipelineEngine(Engine):
    """transformers の automatic-speech-recognition パイプライン(Whisper 系・kotoba-whisper など)"""

    kind = "hf-pipeline"

    def __init__(self, model: str, dtype: str = "auto", batch_size: int = 8, trust_remote_code: bool = False,
                 whisper: bool = True, **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        from transformers import pipeline

        torch = torch_mod()
        self.whisper = whisper
        kw: Dict[str, Any] = dict(model=model, device=device_str(), trust_remote_code=trust_remote_code)
        try:
            self.p = pipeline("automatic-speech-recognition", dtype=pick_dtype(dtype), **kw)
        except TypeError:
            self.p = pipeline("automatic-speech-recognition", torch_dtype=pick_dtype(dtype), **kw)
        self.torch = torch

    def _gen_kwargs(self, language, ctx):
        gk: Dict[str, Any] = {}
        if self.whisper:
            if language:
                gk["language"] = lang_code(language) or language
            gk["task"] = "transcribe"
            if ctx:
                try:
                    ids = self.p.tokenizer.get_prompt_ids(ctx, return_tensors="pt").to(self.p.device)
                    gk["prompt_ids"] = ids
                except Exception:
                    pass
        return gk

    def transcribe(self, items, language):
        # context ごとにまとめて流す(Whisper の prompt はバッチ内で共通)
        out: List[Optional[Dict[str, Any]]] = [None] * len(items)
        groups: Dict[str, List[int]] = {}
        for i, (_, c) in enumerate(items):
            groups.setdefault(c or "", []).append(i)
        for ctx, idxs in groups.items():
            auds = [{"raw": _pad_min(items[i][0]), "sampling_rate": core.SR} for i in idxs]

            def run(chunk):
                res = self.p(chunk, batch_size=len(chunk), generate_kwargs=self._gen_kwargs(language, ctx))
                return [r.get("text", "") if isinstance(r, dict) else str(r) for r in res]

            texts = with_oom_backoff(run, auds, self.batch_size, log=_log)
            for i, t in zip(idxs, texts):
                out[i] = {"text": (t or "").strip(), "language": language or ""}
        return [o or {"text": ""} for o in out]

    def close(self) -> None:
        del self.p
        super().close()


class CohereASREngine(Engine):
    """Cohere Transcribe (CohereLabs/cohere-transcribe-03-2026, 2B, 14言語・日本語あり)。
    transformers>=5.4 のネイティブ実装を使う(qwen-asr とは transformers のバージョンがぶつかるので hf 環境で)。
    タイムスタンプも context も無いので、時刻は Qwen3-ForcedAligner で付ける。"""

    kind = "cohere"
    uses_context = False

    def __init__(self, model: str = "CohereLabs/cohere-transcribe-03-2026", dtype: str = "auto", batch_size: int = 16,
                 max_new_tokens: int = 448, punctuation: bool = True, revision: str = "", **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        import transformers
        from transformers import AutoProcessor

        self.max_new_tokens = int(max_new_tokens)
        self.punctuation = bool(punctuation)
        dt = pick_dtype(dtype)
        cls = getattr(transformers, "CohereAsrForConditionalGeneration", None)
        last: Optional[Exception] = None
        self.m = None
        # 公開直後は HF 版の重みが PR ブランチ(refs/pr/6)にあったので、だめなら順に試す
        for rev in ([revision] if revision else []) + [None, "refs/pr/6"]:
            try:
                kw: Dict[str, Any] = {"revision": rev} if rev else {}
                self.proc = AutoProcessor.from_pretrained(model, **kw)
                if cls is not None:
                    self.m = cls.from_pretrained(model, dtype=dt, device_map=device_str(), **kw)
                else:
                    from transformers import AutoModelForSpeechSeq2Seq

                    self.m = AutoModelForSpeechSeq2Seq.from_pretrained(model, dtype=dt, device_map=device_str(),
                                                                      trust_remote_code=True, **kw)
                break
            except Exception as e:  # 次の候補へ
                last = e
                _log(f"cohere: revision={rev} で失敗: {type(e).__name__}: {str(e)[:300]}")
        if self.m is None:
            raise RuntimeError(f"Cohere Transcribe を読み込めませんでした(HF で規約に同意し HF_TOKEN を登録しましたか?): {last}")
        self.m.eval()

    def _run(self, chunk, code):
        torch = torch_mod()
        auds = [_pad_min(a) for a, _ in chunk]
        inputs = self.proc(auds, sampling_rate=core.SR, return_tensors="pt", language=code, punctuation=self.punctuation)
        idx = inputs.get("audio_chunk_index")
        inputs = inputs.to(self.m.device, dtype=self.m.dtype)
        with torch.inference_mode():
            out = self.m.generate(**inputs, max_new_tokens=self.max_new_tokens)
        try:
            texts = self.proc.decode(out, skip_special_tokens=True, audio_chunk_index=idx, language=code)
        except TypeError:
            texts = self.proc.batch_decode(out, skip_special_tokens=True)
        if isinstance(texts, str):
            texts = [texts]
        return [{"text": (t or "").strip(), "language": lang_name(code)} for t in texts]

    def transcribe(self, items, language):
        code = lang_code(language) or "ja"
        return with_oom_backoff(lambda ch: self._run(ch, code), items, self.batch_size, log=_log)

    def close(self) -> None:
        del self.m
        super().close()


class GraniteSpeechEngine(Engine):
    """IBM Granite Speech 4.1 2B(英・仏・独・西・葡・日)。キーワード(固有名詞)を渡せる。
    プロンプトは英語で書く決まり(モデルカードより)。transformers の hf 環境で動かす"""

    kind = "granite"

    def __init__(self, model: str = "ibm-granite/granite-speech-4.1-2b", dtype: str = "auto", batch_size: int = 8,
                 max_new_tokens: int = 448, **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

        self.proc = AutoProcessor.from_pretrained(model)
        self.tok = self.proc.tokenizer
        self.m = AutoModelForSpeechSeq2Seq.from_pretrained(model, device_map=device_str(), dtype=pick_dtype(dtype))
        self.m.eval()
        self.max_new_tokens = int(max_new_tokens)
        self.ctx_terms: List[str] = []

    def _prompt(self, use_kw: bool) -> str:
        if use_kw and self.ctx_terms:
            q = "transcribe the speech to text. Keywords: " + ", ".join(self.ctx_terms)
        else:
            q = "transcribe the speech with proper punctuation and capitalization."
        return self.tok.apply_chat_template([{"role": "user", "content": "<|audio|>" + q}], tokenize=False,
                                            add_generation_prompt=True)

    def _run(self, chunk: List[Tuple[np.ndarray, str]]) -> List[Dict[str, Any]]:
        torch = torch_mod()
        prompt = self._prompt(bool(chunk[0][1]))  # chunk の中はプロンプトが同じ
        wavs = [_pad_min(a) for a, _ in chunk]  # 長さが違っても processor が詰めてくれる(トークナイザは左詰め)
        inp = self.proc([prompt] * len(wavs), wavs, device=device_str(), return_tensors="pt").to(device_str())
        with torch.inference_mode():
            ids = self.m.generate(**inp, max_new_tokens=self.max_new_tokens, do_sample=False, num_beams=1)
        n = inp["input_ids"].shape[-1]
        texts = self.tok.batch_decode(ids[:, n:], add_special_tokens=False, skip_special_tokens=True)
        return [{"text": (t or "").strip()} for t in texts]

    def transcribe(self, items, language):
        # キーワードあり/なしでプロンプトが違うので、それぞれまとめてバッチ処理(以前は1件ずつで遅かった)
        out: List[Dict[str, Any]] = [{} for _ in items]
        for use_kw in (True, False):
            idx = [i for i, (_, c) in enumerate(items) if bool(c) == use_kw]
            if not idx:
                continue
            res = with_oom_backoff(self._run, [items[i] for i in idx], self.batch_size, log=_log)
            for i, r in zip(idx, res):
                out[i] = {"text": r["text"], "language": language or ""}
        return out

    def close(self) -> None:
        del self.m
        super().close()


class VibeVoiceEngine(Engine):
    """Microsoft VibeVoice-ASR(8B、50以上の言語、context を渡せる)。transformers ネイティブ版(-HF)を使う。
    本来は60分を一気に読んで話者も付けられるモデルだが、ここでは他のモデルと同じくクリップ単位で本文だけ使う"""

    kind = "vibevoice"

    def __init__(self, model: str = "microsoft/VibeVoice-ASR-HF", dtype: str = "auto", batch_size: int = 4,
                 max_new_tokens: int = 2048, **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        import transformers
        from transformers import AutoProcessor

        cls = getattr(transformers, "VibeVoiceAsrForConditionalGeneration")
        self.proc = AutoProcessor.from_pretrained(model)
        self.m = cls.from_pretrained(model, device_map=device_str(), dtype=pick_dtype(dtype))
        self.m.eval()
        self.max_new_tokens = int(max_new_tokens)
        self.tmp = tempfile.mkdtemp(prefix="vibevoice_")

    def _run(self, chunk):
        torch = torch_mod()
        paths = []
        for i, (a, _) in enumerate(chunk):
            p = os.path.join(self.tmp, f"{i}.wav")
            core.write_wav16(p, _pad_min(a))
            paths.append(p)
        prompts = [c or None for _, c in chunk]
        inputs = self.proc.apply_transcription_request(paths, prompt=prompts).to(self.m.device, self.m.dtype)
        with torch.inference_mode():
            out = self.m.generate(**inputs, max_new_tokens=self.max_new_tokens)
        gen = out[:, inputs["input_ids"].shape[1]:]
        texts = self.proc.decode(gen, return_format="transcription_only")
        if isinstance(texts, str):
            texts = [texts]
        return [{"text": (t or "").strip(), "language": ""} for t in texts]

    def transcribe(self, items, language):
        return with_oom_backoff(self._run, items, self.batch_size, log=_log)

    def close(self) -> None:
        del self.m
        super().close()


class PyannoteDiarizer:
    """pyannote.audio 4.x の話者分離(community-1 は exclusive 出力もあり)"""

    kind = "pyannote"

    def __init__(self, model: str = "pyannote/speaker-diarization-community-1", **opt: Any) -> None:
        from pyannote.audio import Pipeline

        torch = torch_mod()
        token = os.environ.get("HF_TOKEN") or None
        try:
            p = Pipeline.from_pretrained(model, token=token)
        except TypeError:
            p = Pipeline.from_pretrained(model, use_auth_token=token)
        if p is None:
            raise RuntimeError(
                f"{model} を読み込めませんでした。Hugging Face でモデルの利用規約に同意し、"
                "Colab のシークレットに HF_TOKEN を登録してください。")
        if torch.cuda.is_available():
            p.to(torch.device("cuda"))
        self.p = p
        self.opt = dict(model=model, **opt)

    def info(self) -> Dict[str, Any]:
        return {"summary": self.opt.get("model"), "gpu_peak_gb": gpu_peak_gb()}

    def diarize(self, wav_path: str, num_speakers: int = 0, min_speakers: int = 0, max_speakers: int = 0) -> Dict[str, Any]:
        torch = torch_mod()
        w = core.Wav16(wav_path)
        x = torch.from_numpy(w.get(0, w.duration)).unsqueeze(0)
        kw: Dict[str, Any] = {}
        if num_speakers and num_speakers > 0:
            kw["num_speakers"] = int(num_speakers)
        else:
            if min_speakers and min_speakers > 0:
                kw["min_speakers"] = int(min_speakers)
            if max_speakers and max_speakers > 0:
                kw["max_speakers"] = int(max_speakers)
        out = self.p({"waveform": x, "sample_rate": w.sr}, **kw)
        ann = getattr(out, "speaker_diarization", out)
        excl = getattr(out, "exclusive_speaker_diarization", None)

        def tracks(a):
            return [[round(float(t.start), 3), round(float(t.end), 3), str(k)] for t, _, k in a.itertracks(yield_label=True)]

        return {"turns": tracks(ann), "exclusive": tracks(excl) if excl is not None else None}

    def close(self) -> None:
        del self.p
        free_cuda()


class HFSpeechLMEngine(Engine):
    """transformers の「音声→テキスト」系 LLM (AutoProcessor + AutoModelForSpeechSeq2Seq/ImageTextToText 等)
    の汎用アダプタ。processor が apply_transcription_request を持つモデル(Voxtral など)と、
    chat template で音声を渡すモデルの両方をなるべく吸収する(実験的)。"""

    kind = "hf-speechlm"

    def __init__(self, model: str, dtype: str = "auto", batch_size: int = 4, max_new_tokens: int = 1024,
                 trust_remote_code: bool = True, prompt: str = "", **opt: Any) -> None:
        super().__init__(model=model, batch_size=batch_size, **opt)
        import transformers
        from transformers import AutoProcessor

        torch = torch_mod()
        self.torch = torch
        self.dtype = pick_dtype(dtype)
        self.max_new_tokens = int(max_new_tokens)
        self.prompt = prompt
        self.proc = AutoProcessor.from_pretrained(model, trust_remote_code=trust_remote_code)
        last: Optional[Exception] = None
        self.m = None
        for cls_name in ("AutoModelForSpeechSeq2Seq", "AutoModelForImageTextToText", "AutoModelForCausalLM", "AutoModel"):
            cls = getattr(transformers, cls_name, None)
            if cls is None:
                continue
            try:
                self.m = cls.from_pretrained(model, dtype=self.dtype, device_map=device_str(), trust_remote_code=trust_remote_code)
                break
            except Exception as e:  # 次の Auto クラスを試す
                last = e
        if self.m is None:
            raise RuntimeError(f"{model} を読み込めませんでした: {last}")
        self.m.eval()

    def _inputs(self, audio: np.ndarray, ctx: str, language: Optional[str]):
        p = self.proc
        code = lang_code(language)
        if hasattr(p, "apply_transcription_request"):
            kw: Dict[str, Any] = {"audio": audio, "model_id": self.opt.get("model")}
            if code:
                kw["language"] = code
            try:
                return p.apply_transcription_request(**kw, sampling_rate=core.SR, format=["wav"])
            except TypeError:
                return p.apply_transcription_request(**kw)
        text = self.prompt or ("以下の音声を日本語で正確に書き起こしてください。" if code == "ja" else "Transcribe the audio.")
        if ctx:
            text = f"{ctx}\n{text}"
        msgs = [{"role": "user", "content": [{"type": "audio", "audio": audio}, {"type": "text", "text": text}]}]
        try:
            return p.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                         return_tensors="pt", sampling_rate=core.SR)
        except Exception:
            return p(audio=audio, text=text, sampling_rate=core.SR, return_tensors="pt")

    def transcribe(self, items, language):
        torch = self.torch
        out = []
        for a, ctx in items:  # モデルごとに入力形式が違うので 1 件ずつ(確実さ優先)
            inp = self._inputs(_pad_min(a), ctx, language)
            inp = inp.to(self.m.device)
            for k, v in list(inp.items()):
                if hasattr(v, "is_floating_point") and v.is_floating_point():
                    inp[k] = v.to(self.dtype)
            with torch.inference_mode():
                ids = self.m.generate(**inp, max_new_tokens=self.max_new_tokens, do_sample=False)
            n_in = inp["input_ids"].shape[1] if "input_ids" in inp else 0
            gen = ids[:, n_in:] if n_in and ids.shape[1] > n_in else ids
            dec = getattr(self.proc, "batch_decode", None) or self.proc.tokenizer.batch_decode
            text = dec(gen, skip_special_tokens=True)[0]
            out.append({"text": (text or "").strip(), "language": language or ""})
        return out

    def close(self) -> None:
        del self.m
        super().close()


class DummyEngine(Engine):
    """テスト用(GPU なしで配管だけ確かめる)。長さに応じた決まった文章を返す"""

    kind = "dummy"

    def transcribe(self, items, language):
        out = []
        for a, ctx in items:
            sec = len(a) / core.SR
            if ctx and "ECHO" in ctx:
                out.append({"text": ctx, "language": language or "Japanese"})
            else:
                n = max(1, int(sec // 3))
                out.append({"text": "".join(f"これは{i + 1}番目の文です。" for i in range(n)), "language": language or "Japanese"})
        return out


class DummyAligner:
    kind = "dummy-aligner"

    def __init__(self, **opt: Any) -> None:
        self.opt = opt

    def info(self) -> Dict[str, Any]:
        return {"summary": "dummy aligner"}

    def align(self, items):
        out = []
        for a, text, _ in items:
            dur = len(a) / core.SR
            chars = [ch for ch in text if core.is_kept(ch)]
            n = max(1, len(chars))
            out.append([(ch, dur * i / n, dur * (i + 1) / n) for i, ch in enumerate(chars)])
        return out

    def close(self) -> None:
        pass


class DummyDiarizer:
    kind = "dummy-diar"

    def __init__(self, **opt: Any) -> None:
        self.opt = opt

    def info(self) -> Dict[str, Any]:
        return {"summary": "dummy diarizer"}

    def diarize(self, wav_path, num_speakers=0, min_speakers=0, max_speakers=0):
        w = core.Wav16(wav_path)
        turns, t, k = [], 0.0, 0
        while t < w.duration:
            turns.append([round(t, 3), round(min(w.duration, t + 10.0), 3), f"SPEAKER_{k % 2:02d}"])
            t += 10.0
            k += 1
        return {"turns": turns, "exclusive": turns}

    def close(self) -> None:
        pass


ENGINE_CLASSES = {
    "dummy": DummyEngine,
    "dummy-aligner": DummyAligner,
    "dummy-diar": DummyDiarizer,
    "qwen": QwenASREngine,
    "faster-whisper": FasterWhisperEngine,
    "nemo": NemoEngine,
    "hf-pipeline": HFPipelineEngine,
    "hf-speechlm": HFSpeechLMEngine,
    "cohere": CohereASREngine,
    "granite": GraniteSpeechEngine,
    "vibevoice": VibeVoiceEngine,
    "aligner": QwenAligner,
    "pyannote": PyannoteDiarizer,
}


def make_engine(kind: str, options: Dict[str, Any]):
    cls = ENGINE_CLASSES.get(kind)
    if cls is None:
        raise ValueError(f"未知のエンジン: {kind}")
    return cls(**options)
'''
_FILES['worker.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.worker — 各 venv の中で動くワーカー。stdin/stdout で JSON を1行ずつやりとりする

  → {"id": 1, "cmd": "load", "key": "asr", "kind": "qwen", "options": {...}}
  ← {"id": 1, "ok": true, "info": {...}}
  → {"id": 2, "cmd": "transcribe", "key": "asr", "wav": "...", "clips": [...], ...}
  ← {"id": 2, "event": "progress", "stage": "asr", "done": 8, "total": 120, "results": [...]}
  ← {"id": 2, "ok": true, "results": [...]}

stdout はこのやりとり専用。ライブラリのログや print は全部 stderr に流す。
"""
from __future__ import annotations

import json
import os
import signal
import sys
import time
import traceback
from typing import Any, Dict


def _setup_io():
    proto = os.fdopen(os.dup(1), "w", encoding="utf-8", buffering=1)
    os.dup2(2, 1)  # 以降の print は stderr へ
    sys.stdout = sys.stderr
    return proto


def _pdeathsig() -> None:
    try:
        import ctypes

        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGTERM)
    except Exception:
        pass


def main() -> None:
    proto = _setup_io()
    _pdeathsig()
    from . import core, engines

    objs: Dict[str, Any] = {}

    def send(obj: Dict[str, Any]) -> None:
        proto.write(json.dumps(obj, ensure_ascii=False) + "\n")
        proto.flush()

    wav_cache: Dict[str, core.Wav16] = {}

    def wav(path: str) -> core.Wav16:
        w = wav_cache.get(path)
        if w is None:
            wav_cache.clear()
            w = wav_cache[path] = core.Wav16(path)
        return w

    while True:
        try:
            line = sys.stdin.readline()
        except KeyboardInterrupt:
            continue
        if not line:
            break  # 親が閉じた
        line = line.strip()
        if not line:
            continue
        try:
            msg = json.loads(line)
        except Exception:
            continue
        rid, cmd = msg.get("id"), msg.get("cmd")
        try:
            if cmd == "exit":
                send({"id": rid, "ok": True})
                break
            elif cmd == "ping":
                info: Dict[str, Any] = {"python": sys.version.split()[0], "loaded": list(objs)}
                try:
                    import torch

                    info.update(torch=torch.__version__, cuda=torch.cuda.is_available(),
                                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
                except Exception as e:
                    info["torch_error"] = repr(e)
                send({"id": rid, "ok": True, "info": info})
            elif cmd == "load":
                key = msg["key"]
                if key in objs:
                    try:
                        objs.pop(key).close()
                    except Exception:
                        pass
                    engines.free_cuda()
                engines.reset_gpu_peak()
                t0 = time.time()
                obj = engines.make_engine(msg["kind"], dict(msg.get("options") or {}))
                objs[key] = obj
                info = obj.info() if hasattr(obj, "info") else {}
                info["load_sec"] = round(time.time() - t0, 1)
                send({"id": rid, "ok": True, "info": info})
            elif cmd == "unload":
                obj = objs.pop(msg["key"], None)
                if obj is not None:
                    obj.close()
                engines.free_cuda()
                send({"id": rid, "ok": True})
            elif cmd == "transcribe":
                eng = objs[msg["key"]]
                w = wav(msg["wav"])
                clips = [core.Clip.from_dict(d) for d in msg["clips"]]
                done = {int(d["clip"]["id"]): core.ClipResult.from_dict(d) for d in (msg.get("done") or [])}
                language = msg.get("language") or None
                pol = core.RetryPolicy(**(msg.get("policy") or {}))
                pol.punctuates = bool(getattr(eng, "punctuates", True)) and pol.punctuates
                db = core.frame_db(w) if pol.enabled and pol.split else None
                if hasattr(eng, "ctx_terms"):  # キーワードを別に受け取るモデル(Granite など)
                    eng.ctx_terms = list(msg.get("ctx_terms") or [])

                def on_progress(stage, n, total, part):
                    send({"id": rid, "event": "progress", "stage": stage, "done": n, "total": total,
                          "results": [r.to_dict() for r in part]})

                engines.reset_gpu_peak()  # 峰 = 読み込み済みの重み + この文字起こし中の最大
                t0 = time.time()
                res = core.run_asr(
                    clips, w.get, lambda items: eng.transcribe(items, language),
                    context=msg.get("context") or "", ctx_terms=msg.get("ctx_terms") or [],
                    ctx_label=msg.get("ctx_label") or "", policy=pol,
                    batch_size=int(msg.get("batch_size") or getattr(eng, "batch_size", 8)),
                    db=db, on_progress=on_progress, done=done,
                )
                send({"id": rid, "ok": True, "results": [r.to_dict() for r in res],
                      "seconds": round(time.time() - t0, 2), "gpu_peak_gb": engines.gpu_peak_gb()})
            elif cmd == "align":
                al = objs[msg["key"]]
                w = wav(msg["wav"])
                items = msg["items"]  # [{"id", "start", "end", "text", "language"}]
                out: Dict[str, Any] = {}
                bs = int(msg.get("batch_size") or 8)
                total = len(items)
                # 長さ順に並べてまとめて流す
                order = sorted(items, key=lambda d: d["end"] - d["start"], reverse=True)
                for i in range(0, len(order), bs * 4):
                    chunk = order[i: i + bs * 4]
                    toks = al.align([(w.get(d["start"], d["end"]), d["text"], d["language"]) for d in chunk])
                    for d, t in zip(chunk, toks):
                        out[str(d["id"])] = [[a, round(b, 3), round(c, 3)] for a, b, c in t]
                    send({"id": rid, "event": "progress", "stage": "align", "done": min(i + bs * 4, total), "total": total})
                send({"id": rid, "ok": True, "aligned": out, "gpu_peak_gb": engines.gpu_peak_gb()})
            elif cmd == "diarize":
                d = objs[msg["key"]]
                t0 = time.time()
                r = d.diarize(msg["wav"], int(msg.get("num_speakers") or 0), int(msg.get("min_speakers") or 0),
                              int(msg.get("max_speakers") or 0))
                r["seconds"] = round(time.time() - t0, 2)
                send({"id": rid, "ok": True, **r})
            else:
                raise ValueError(f"未知のコマンド: {cmd}")
        except KeyboardInterrupt:
            engines.free_cuda()
            send({"id": rid, "ok": False, "error": "cancelled"})
        except BaseException as e:  # noqa: BLE001
            engines.free_cuda()
            send({"id": rid, "ok": False, "error": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc()})
            if isinstance(e, (SystemExit,)):
                break


if __name__ == "__main__":
    main()
'''
_FILES['presets.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.presets — モデルのプリセットと、エンジンごとの venv の中身

新しいモデルを足したいときは PRESETS に1行足すだけ。フォームのドロップダウンにも自動で出る。
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from .runtime import EnvSpec

# ---------------------------------------------------------------------------------------------
# venv の中身
#   qwen      : Qwen3-ASR / Qwen3-ForcedAligner(タイムスタンプ付け)。システムの torch を使う
#   fw        : faster-whisper (Whisper large-v3 / turbo / kotoba-whisper-faster)
#   nemo      : NVIDIA NeMo (parakeet-tdt_ctc-0.6b-ja など)
#   hf        : 最新 transformers(新しめの音声LLM・kotoba-whisper の HF 版など)
#   pyannote  : 話者分離
#   vllm      : vLLM サーバー(A100/H100 で爆速。torch ごと別に入れるので分離環境)
# ---------------------------------------------------------------------------------------------

ENV_SPECS: Dict[str, EnvSpec] = {
    "qwen": EnvSpec(
        name="qwen",
        packages=["qwen-asr==0.0.6"],
        check="import qwen_asr, transformers, torch; print(f'qwen-asr OK / transformers {transformers.__version__} / torch {torch.__version__}')",
        note="Qwen3-ASR と タイムスタンプ用アライナー(必須)",
    ),
    "fw": EnvSpec(
        name="fw",
        packages=["faster-whisper>=1.2.1", "nvidia-cublas-cu12"],
        check="import faster_whisper, ctranslate2; print(f'faster-whisper {faster_whisper.__version__} / ctranslate2 {ctranslate2.__version__}')",
        note="Whisper 系(faster-whisper)",
    ),
    "nemo": EnvSpec(
        name="nemo",
        packages=["nemo_toolkit[asr]>=2.4"],
        check="import nemo, nemo.collections.asr; print(f'nemo {nemo.__version__}')",
        note="NVIDIA NeMo(parakeet 日本語モデル)。インストールに数分かかる",
    ),
    "hf": EnvSpec(
        name="hf",
        packages=["transformers>=5.5", "accelerate", "librosa", "soundfile", "sentencepiece"],
        check="import transformers, torch; print(f'transformers {transformers.__version__} / torch {torch.__version__}')",
        note="最新 transformers の音声モデル(Cohere / Granite / VibeVoice / カスタム)",
    ),
    "pyannote": EnvSpec(
        name="pyannote",
        packages=["pyannote.audio>=4.0.4,<5"],
        check="import pyannote.audio, torch; print(f'pyannote.audio {pyannote.audio.__version__} / torch {torch.__version__}')",
        note="話者分離(Hugging Face の HF_TOKEN と規約同意が必要)",
    ),
    "vllm": EnvSpec(
        name="vllm",
        packages=["vllm[audio]"],
        # uv がドライバの CUDA を見て合う torch を選ぶ(vLLM 公式の推奨)
        pip_args=["--torch-backend=auto"],
        isolated=True,
        check=("import vllm, torch; ok = torch.cuda.is_available(); "
               "print(f'vllm {vllm.__version__} / torch {torch.__version__} (cuda {torch.version.cuda}, GPU {ok})')"),
        note="vLLM(T4 でも動くが A100/H100/L4 で真価。インストール数分)",
    ),
}


@dataclass
class Preset:
    key: str
    label: str
    engine: str  # qwen / vllm / faster-whisper / nemo / hf-pipeline / hf-speechlm
    env: str
    model: str
    options: Dict[str, Any] = field(default_factory=dict)
    vllm_args: List[str] = field(default_factory=list)
    ja: str = "◎"  # 日本語の相性(目安)
    punctuates: bool = True
    context: bool = True  # context(固有名詞のヒント)を使えるか
    native_ts: bool = False
    # クリップの区切り方のおすすめ(None = ふつう: 30 秒まで・6 秒以下の無音はつなぐ)。
    # 長いクリップや途中の無音で発話を読み飛ばしやすいモデルは短めにする。④ で「自動」のときだけ使う
    max_clip: Optional[float] = None
    max_gap: Optional[float] = None
    gpu: str = "T4〜"  # 目安
    license: str = ""
    note: str = ""


PRESETS: List[Preset] = [
    Preset("qwen3-1.7b", "Qwen3-ASR 1.7B（標準・おすすめ）", "qwen", "qwen", "Qwen/Qwen3-ASR-1.7B",
           license="Apache-2.0", note="v1/v2 と同じ。context で固有名詞に強い"),
    Preset("qwen3-0.6b", "Qwen3-ASR 0.6B（軽量・速い）", "qwen", "qwen", "Qwen/Qwen3-ASR-0.6B",
           ja="○", license="Apache-2.0", note="1.7B より少し精度が落ちるぶん速い"),
    Preset("qwen3-1.7b-ja", "Qwen3-ASR 1.7B JA（neosophie・固有名詞に強い日本語調整版）", "qwen", "qwen",
           "neosophie/Qwen3-ASR-1.7B-JA", license="Apache-2.0",
           note="IT・ビジネス系の固有名詞 F1 が 0.59→0.65。全体の CER は 8.23%→8.92% と少し悪化の報告"),
    Preset("qwen3-1.7b-vllm", "Qwen3-ASR 1.7B × vLLM（A100/H100で爆速）", "vllm", "vllm", "Qwen/Qwen3-ASR-1.7B",
           gpu="T4〜(A100/H100 で真価)", license="Apache-2.0", note="中身は標準と同じモデル。大量バッチで数倍〜十数倍速い"),
    Preset("cohere-transcribe", "Cohere Transcribe 2B（別系統の有力候補）", "cohere", "hf",
           "CohereLabs/cohere-transcribe-03-2026", options={"batch_size": 16}, context=False, gpu="T4〜",
           license="Apache-2.0", note="日本語 CER: FLEURS 2.89% / CV 20.2%(Qwen 5.28% / 26.3%)。HF で規約同意が必要"),
    Preset("cohere-transcribe-vllm", "Cohere Transcribe 2B × vLLM", "vllm", "vllm", "CohereLabs/cohere-transcribe-03-2026",
           context=False, gpu="T4〜(A100/H100 で真価)", license="Apache-2.0", note="Cohere を vLLM で高速に"),
    Preset("whisper-large-v3-turbo", "Whisper large-v3-turbo（faster-whisper）", "faster-whisper", "fw", "large-v3-turbo",
           options={"beam_size": 5}, ja="○", native_ts=True, license="MIT", note="定番。速い"),
    Preset("whisper-large-v3", "Whisper large-v3（faster-whisper）", "faster-whisper", "fw", "large-v3",
           options={"beam_size": 5}, ja="○", native_ts=True, license="MIT", note="定番の最高精度版"),
    Preset("kotoba-whisper-v2", "kotoba-whisper v2.0（日本語特化 Whisper）", "faster-whisper", "fw",
           "kotoba-tech/kotoba-whisper-v2.0-faster", options={"beam_size": 5, "word_timestamps": False, "chunk_length": 15},
           max_clip=15.0, license="MIT",
           note="ReazonSpeech で学習した日本語特化の蒸留モデル。公式のおすすめどおり 15 秒ずつ読む"
                "(タイムスタンプはアライナーで付ける。faster-whisper の単語時刻はこのモデルだと segfault するため)"),
    Preset("parakeet-ja", "Parakeet TDT-CTC 0.6B ja（NVIDIA・日本語特化）", "nemo", "nemo", "nvidia/parakeet-tdt_ctc-0.6b-ja",
           options={"batch_size": 16}, context=False, native_ts=True, punctuates=False, license="CC-BY-4.0",
           note="とても速い日本語専用モデル(JSUT の CER 6.60% の報告)。句読点は少なめ"),
    Preset("granite-speech-4.1", "Granite Speech 4.1 2B（IBM・実験的）", "granite", "hf", "ibm-granite/granite-speech-4.1-2b",
           options={"batch_size": 8}, ja="○", punctuates=False, license="Apache-2.0",
           note="日本語対応の2B(2026年4月)。context の語をキーワードとして渡すが、関係ない所にも入れがちなので語は少なめに"),
    Preset("vibevoice-asr", "VibeVoice-ASR 8B（Microsoft・実験的）", "vibevoice", "hf", "microsoft/VibeVoice-ASR-HF",
           options={"batch_size": 4}, ja="○", gpu="A100/H100/L4(24GB〜)", license="MIT",
           note="50以上の言語・context 対応の8B。本来は60分一気読み＋話者付けのモデル"),
]


# flash-attn の配布済み whl(torch のバージョン, Python タグ) → URL。無ければ sdpa で動く
# (torch 2.11 用は公式の whl が無いので、CUDA 12.8 でビルドされたコミュニティ版。Colab のカーネルは 3.12 → 3.13 に変わった)
_FA_211 = "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/"
FLASH_WHEELS: Dict[tuple, str] = {
    ("2.11", tag): f"{_FA_211}flash_attn-2.8.3%2Bcu12torch2.11cxx11abiTRUE-{tag}-{tag}-linux_x86_64.whl"
    for tag in ("cp312", "cp313")
}


def flash_attn_wheel() -> Optional[str]:
    """いまのカーネルの torch / Python に合う flash-attn の whl の URL(無ければ None)"""
    import sys

    try:
        import torch

        tv = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    except Exception:
        return None
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    return FLASH_WHEELS.get((tv, tag))


def preset_labels() -> List[str]:
    return [p.label for p in PRESETS]


def find_preset(key_or_label: str) -> Optional[Preset]:
    s = (key_or_label or "").strip()
    for p in PRESETS:
        if s in (p.key, p.label):
            return p
    return None


def custom_preset(engine: str, model: str, env: Optional[str] = None) -> Preset:
    """フォームの「カスタム」用: エンジン名と HF のモデル ID から作る"""
    engine = engine.strip()
    env = env or {"qwen": "qwen", "vllm": "vllm", "faster-whisper": "fw", "nemo": "nemo",
                  "hf-pipeline": "hf", "hf-speechlm": "hf", "cohere": "hf", "granite": "hf",
                  "vibevoice": "hf"}.get(engine, "hf")
    return Preset(f"custom:{engine}:{model}", f"カスタム {engine}: {model}", engine, env, model.strip(),
                  native_ts=engine == "nemo" or (engine == "faster-whisper" and not any(
                      k in model.lower() for k in ("kotoba", "distil"))), ja="?", note="カスタム")


def envs_for(preset: Preset, diarize: bool, aligner: bool) -> List[str]:
    need = [preset.env]
    if aligner and "qwen" not in need:
        need.append("qwen")
    if diarize:
        need.append("pyannote")
    return need
'''
_FILES['pipeline.py'] = r'''# -*- coding: utf-8 -*-
"""asrkit.pipeline — ノートブックから呼ぶ高レベルの処理

  sess = Session()
  sess.run(inputs, settings)            # 文字起こし(複数ファイル可)
  sess.compare(src, [プリセット...], settings, start=0, duration=300, reference="")  # モデル比較
"""
from __future__ import annotations

import dataclasses
import glob
import hashlib
import html
import json
import os
import time
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

from . import core, runtime
from .engines import ALIGNER_LANGS, lang_code, lang_name
from .presets import Preset, custom_preset, find_preset

VERSION = "3.0.0"
log = runtime.log


# =====================================================================
# 設定
# =====================================================================


@dataclass
class Settings:
    # 入出力
    output_dir: str = ""  # 空: 音源と同じフォルダ / フォルダ: その中 / ファイル名: そのパス(拡張子は無視)
    formats: Tuple[str, ...] = ("txt", "srt", "json")  # txt srt vtt json csv md plain rttm
    skip_existing: bool = False
    # モデル
    preset: str = "qwen3-1.7b"
    custom_engine: str = ""
    custom_model: str = ""
    language: str = "Japanese"  # "auto" で自動判定
    batch_size: int = 0  # 0 = GPU から自動
    max_new_tokens: int = 1024
    dtype: str = "auto"
    attn: str = "auto"
    # タイムスタンプ
    aligner: bool = True
    aligner_model: str = "Qwen/Qwen3-ForcedAligner-0.6B"
    prefer_aligner: bool = True  # モデル自身のタイムスタンプがあってもアライナーを優先
    # 音声
    channel: str = "mix"  # mix / left / right
    af: str = ""  # ffmpeg フィルタ
    # 区間検出
    vad: str = "fireredvad"  # fireredvad / silero / energy / none
    vad_threshold: float = 0.4
    vad_min_speech: float = 0.2
    vad_min_silence: float = 0.2
    vad_pad: float = 0.2
    energy_top_db: float = 45.0
    # クリップの区切り方。None = 自動(モデルのおすすめ。ふつうは 30 秒まで・6 秒以下の無音はつなぐ)
    max_clip: Optional[float] = None
    max_gap: Optional[float] = None
    overlap: float = 1.0
    # context
    context_label: str = "固有名詞・専門用語"
    context_terms: List[str] = field(default_factory=list)
    context_extra: str = ""
    # リトライ
    retry: bool = True
    # テキスト整形
    replacements: str = ""
    fillers: str = "off"  # off / safe / more
    halfwidth: bool = True
    # 字幕
    cue_max_chars: int = 30
    cue_max_dur: float = 6.0
    cue_gap: float = 0.8
    speaker_fmt: str = "{speaker}: {text}"
    # 話者分離
    diarize: bool = False
    diar_model: str = "pyannote/speaker-diarization-community-1"
    num_speakers: int = 0
    min_speakers: int = 0
    max_speakers: int = 0
    speaker_names: str = ""
    speaker_style: str = "話者A"
    smooth_speakers: bool = True
    # vLLM
    vllm_gpu_mem: float = 0.0  # 0 = 自動
    vllm_concurrency: int = 64
    # セカンドオピニオン: 最後まで怪しいクリップだけ別モデルでも読んで、良い方を採用(元の結果も JSON に残す)
    second_opinion: str = ""  # プリセットのキー or 表示名。空ならしない
    # その他
    cache: bool = True
    review: bool = True  # 要確認リスト(_review.md)を書き出し、ノートブックに音声つきで表示

    def replace(self, **kw: Any) -> "Settings":
        return dataclasses.replace(self, **kw)


DEFAULT_MAX_CLIP = 30.0
DEFAULT_MAX_GAP = 6.0


def auto_num(v: Any) -> Optional[float]:
    """フォームの「自動」/空欄 → None、それ以外は数値"""
    s = str(v if v is not None else "").strip()
    if s in ("", "自動", "auto", "Auto", "AUTO", "None"):
        return None
    return float(s)


def clip_settings(st: "Settings", p: Preset) -> "Settings":
    """max_clip / max_gap が自動(None)なら、モデルのおすすめ値(無ければ 30 秒 / 6 秒)を入れた Settings を返す"""
    mc = st.max_clip if st.max_clip else (p.max_clip or DEFAULT_MAX_CLIP)
    mg = st.max_gap if st.max_gap is not None else (p.max_gap if p.max_gap is not None else DEFAULT_MAX_GAP)
    return st.replace(max_clip=float(mc), max_gap=float(mg))


def _sha(obj: Any, n: int = 16) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:n]


# =====================================================================
# 入力ファイルの解決・出力先
# =====================================================================


def resolve_inputs(spec: str, recursive: bool = False) -> List[str]:
    """ファイル / フォルダ / ワイルドカード / 改行・カンマ区切りの複数指定 → 音声ファイルの一覧"""
    out: List[str] = []
    parts: List[str] = []
    for line in (spec or "").splitlines():
        line = line.strip().strip('"').strip("'")
        if not line:
            continue
        # カンマ区切りの複数指定にも対応(ただしカンマを含む実在パスはそのまま)
        if "," in line and not os.path.exists(line):
            parts += [x.strip().strip('"').strip("'") for x in line.split(",") if x.strip()]
        else:
            parts.append(line)
    for p in parts:
        if os.path.isdir(p):
            pat = "**/*" if recursive else "*"
            cands = sorted(glob.glob(os.path.join(glob.escape(p), pat), recursive=recursive))
            out += [c for c in cands if os.path.isfile(c) and c.lower().endswith(core.AUDIO_EXTS)]
        elif any(ch in p for ch in "*?["):
            out += sorted(c for c in glob.glob(p, recursive=True) if os.path.isfile(c))
        elif os.path.isfile(p):
            out.append(p)
        else:
            raise FileNotFoundError(f"音源が見つかりません: {p}")
    seen, uniq = set(), []
    for p in out:
        if p not in seen:
            seen.add(p)
            uniq.append(p)
    return uniq


def output_base(src: str, output_dir: str, many: bool = False) -> str:
    stem = os.path.splitext(os.path.basename(src))[0]
    od = (output_dir or "").strip()
    if not od:
        return os.path.splitext(src)[0]
    if many or os.path.isdir(od) or od.endswith(("/", os.sep)) or not os.path.splitext(od)[1]:
        return os.path.join(od, stem)
    return os.path.splitext(od)[0]


# =====================================================================
# VAD(カーネル側で動かす。どれも軽い)
# =====================================================================


def vad_fireredvad(w: core.Wav16, st: Settings) -> List[Tuple[float, float]]:
    """FireRedVAD。長い音声でもメモリを食わないよう 300 秒ごとに特徴量→確率を出してから後処理"""
    import torch
    from fireredvad import FireRedVad, FireRedVadConfig
    from huggingface_hub import snapshot_download

    repo = snapshot_download("FireRedTeam/FireRedVAD", allow_patterns=["VAD/*"])
    cfg = FireRedVadConfig(
        use_gpu=False, smooth_window_size=5, speech_threshold=float(st.vad_threshold),
        min_speech_frame=max(1, int(round(st.vad_min_speech * 100))),
        max_speech_frame=max(200, int((st.max_clip or DEFAULT_MAX_CLIP) * 100)),  # 長い発話は“間”の所で VAD 自身に割らせる
        min_silence_frame=max(1, int(round(st.vad_min_silence * 100))),
        merge_silence_frame=0, extend_speech_frame=0, chunk_max_frame=30000,
    )
    vad = FireRedVad.from_pretrained(os.path.join(repo, "VAD"), cfg)
    x = w.int16()
    L = 30000 * 160  # 300 秒ぶん(フレームの位置がずれないよう 160 の倍数)
    probs: List[Any] = []
    with torch.no_grad():
        for a in range(0, len(x), L):
            seg = np.asarray(x[a: a + L + 240])
            if seg.shape[0] < 400:
                break
            feats, _ = vad.audio_feat.extract((seg, core.SR))
            if feats.shape[0] == 0:
                continue
            if a + L < len(x):
                feats = feats[:30000]
            p, _ = vad.vad_model.forward(feats.unsqueeze(0))
            probs.append(p.detach().cpu().reshape(-1))
    if not probs:
        return []
    pr = torch.cat(probs).tolist()
    dec = vad.vad_postprocessor.process(pr)
    return [(float(s), float(e)) for s, e in vad.vad_postprocessor.decision_to_segment(dec, w.duration)]


def vad_silero(w: core.Wav16, st: Settings) -> List[Tuple[float, float]]:
    import torch
    from silero_vad import get_speech_timestamps, load_silero_vad

    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
    model = load_silero_vad()
    x = torch.from_numpy(w.get(0, w.duration))
    ts = get_speech_timestamps(
        x, model, sampling_rate=core.SR, threshold=float(st.vad_threshold),
        min_speech_duration_ms=int(st.vad_min_speech * 1000), min_silence_duration_ms=int(st.vad_min_silence * 1000),
        speech_pad_ms=30, max_speech_duration_s=float(st.max_clip or DEFAULT_MAX_CLIP), return_seconds=True,
    )
    return [(float(t["start"]), float(t["end"])) for t in ts]


# =====================================================================
# セッション
# =====================================================================


class Session:
    """カーネルに1つだけ作って使い回す(モデルを読み込んだまま、設定だけ変えて何度でも回せる)"""

    def __init__(self) -> None:
        self.h = runtime.Handles()
        self.gpu = runtime.gpu_info()
        self.hf_token = runtime.get_secret("HF_TOKEN")
        self._vad_cache: Dict[str, List[Tuple[float, float]]] = {}
        self.last: Dict[str, Any] = {}
        # タイムスタンプ付けと話者分離のエンジン(テストではダミーに差し替える)
        self.aligner_env, self.aligner_kind = "qwen", "aligner"
        self.diar_env, self.diar_kind = "pyannote", "pyannote"

    # ---------------------------------------------------------- 小物
    def extra_env(self) -> Dict[str, str]:
        env = {}
        if self.hf_token:
            env["HF_TOKEN"] = self.hf_token
        return env

    def preset_of(self, st: Settings) -> Preset:
        if st.preset.startswith("カスタム") or st.preset == "custom":
            if not st.custom_model.strip():
                raise ValueError("カスタムを選んだときは custom_model に Hugging Face のモデル ID を入れてください")
            return custom_preset(st.custom_engine or "hf-pipeline", st.custom_model)
        p = find_preset(st.preset)
        if p is None:
            raise ValueError(f"プリセットが見つかりません: {st.preset}")
        return p

    def batch_size(self, st: Settings, p: Preset) -> int:
        if st.batch_size and st.batch_size > 0:
            return int(st.batch_size)
        model_gb = {"qwen": 5.0, "faster-whisper": 4.0, "nemo": 3.0, "hf-pipeline": 4.0, "hf-speechlm": 10.0,
                    "cohere": 5.0, "granite": 5.0, "vibevoice": 18.0}.get(p.engine, 5.0)
        per = 0.45 * max(1.0, (st.max_clip or DEFAULT_MAX_CLIP) / 30.0)
        bs = runtime.auto_batch_size(self.gpu, model_gb, per, cap=64 if p.engine == "qwen" else 32)
        if p.engine == "faster-whisper":
            bs = min(bs, 8)
        return bs

    def free(self, what: str = "all") -> None:
        if what in ("all", "vllm"):
            self.h.stop_vllm()
        if what == "all":
            self.h.close_all()
        log("🧹 VRAM を解放しました")

    # ---------------------------------------------------------- 前処理
    def prepare_audio(self, src: str, st: Settings, excerpt: Optional[Tuple[float, float]] = None) -> Tuple[str, str]:
        runtime._mkdirs()
        stt = os.stat(src)
        key = _sha([os.path.abspath(src), stt.st_size, int(stt.st_mtime), st.channel, st.af, excerpt])
        dst = os.path.join(runtime.WORK_DIR, f"{key}.wav")
        if not os.path.exists(dst):
            tmp = dst + ".tmp.wav"
            core.ffmpeg_to_wav16(src, tmp, channel=st.channel, af=st.af,
                                 start=excerpt[0] if excerpt else None, duration=excerpt[1] if excerpt else None)
            os.replace(tmp, dst)
            # 作業ファイルは新しい 6 個だけ残す(3時間で 350MB 程度あるため)
            olds = sorted(glob.glob(os.path.join(runtime.WORK_DIR, "*.wav")), key=os.path.getmtime)
            for p in olds[:-6]:
                if p != dst:
                    try:
                        os.remove(p)
                    except OSError:
                        pass
        return dst, key

    def vad(self, w: core.Wav16, wav_key: str, st: Settings, db: np.ndarray) -> List[Tuple[float, float]]:
        ck = _sha([wav_key, st.vad, st.vad_threshold, st.vad_min_speech, st.vad_min_silence, st.energy_top_db, st.max_clip])
        if ck in self._vad_cache:
            return self._vad_cache[ck]
        method = st.vad
        segs: List[Tuple[float, float]] = []
        try:
            if method == "fireredvad":
                segs = vad_fireredvad(w, st)
            elif method == "silero":
                segs = vad_silero(w, st)
            elif method == "energy":
                segs = core.energy_vad(db, top_db=st.energy_top_db, min_speech=st.vad_min_speech,
                                       min_silence=max(0.3, st.vad_min_silence))
            else:
                segs = [(0.0, w.duration)]
        except Exception as e:
            log(f"⚠️ VAD({method}) に失敗したので簡易VAD(energy)で続けます: {type(e).__name__}: {e}")
            segs = core.energy_vad(db, top_db=st.energy_top_db)
        speech = sum(e - s for s, e in segs)
        if w.duration > 5 and speech < 0.02 * w.duration:
            # 発話がほぼ見つからない(小さな声・遠いマイク・VAD の相性)→ 全体を静かな所で区切って全部読ませる
            log(f"⚠️ VAD({method}) で発話がほとんど見つかりませんでした({core.fmt_dur(speech)})。"
                "音声全体を静かな所で区切って文字起こしします(④の vad_threshold を下げるか、音声フィルタも試してください)")
            segs = [(0.0, w.duration)]
        self._vad_cache[ck] = segs
        return segs

    # ---------------------------------------------------------- 各ステージ
    def _asr(self, p: Preset, w: core.Wav16, wav_path: str, clips: List[core.Clip], st: Settings,
             lang: Optional[str], ctx: str, terms: List[str], db: np.ndarray, cache_path: Optional[str],
             bs: int, quiet: bool = False) -> Tuple[List[core.ClipResult], Dict[str, Any]]:
        from tqdm.auto import tqdm

        policy = core.RetryPolicy(enabled=st.retry, punctuates=p.punctuates, lang=lang or "")
        if not p.context:
            ctx, terms = "", []
        done: Dict[int, core.ClipResult] = {}
        if cache_path and os.path.exists(cache_path):
            try:
                with open(cache_path, encoding="utf-8") as f:
                    for line in f:
                        d = json.loads(line)
                        if d.get("final"):
                            res = [core.ClipResult.from_dict(x) for x in d["results"]]
                            log(f"♻️ キャッシュから文字起こし結果を復元しました ({len(res)} クリップ)")
                            return res, {"cached": True}
                        r = core.ClipResult.from_dict(d)
                        done[r.clip.id] = r
                if done:
                    log(f"♻️ 途中から再開します(済み {len(done)}/{len(clips)} クリップ)")
            except Exception:
                done = {}
        cache_f = open(cache_path, "a", encoding="utf-8") if cache_path else None
        bar = tqdm(total=len(clips), desc="文字起こし", unit="clip", disable=quiet, dynamic_ncols=True)
        bar.update(len(done))
        stage_msgs = {"retry": "怪しいクリップを context なしで再推論", "split": "まだ怪しいクリップを分割して再推論"}

        def on_progress(stage: str, n: int, total: int, part: List[core.ClipResult]) -> None:
            if stage == "asr":
                if cache_f:
                    for r in part:
                        cache_f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")
                    cache_f.flush()
                bar.n = n
                bar.refresh()
            elif not quiet:
                bar.set_postfix_str(f"{stage_msgs.get(stage, stage)} {n}/{total}")

        info: Dict[str, Any] = {}
        t0 = time.time()
        try:
            if p.engine == "vllm":
                g = self.gpu
                mem = st.vllm_gpu_mem or (min(0.6, max(0.3, (g.mem_gb - 14) / max(g.mem_gb, 1))) if g.ok else 0.5)
                t_load = time.time()
                server = self.h.vllm_server("vllm", p.model, gpu_mem=mem, extra_args=p.vllm_args, extra_env=self.extra_env())
                waited = time.time() - t_load
                info["load_sec"] = server.start_sec if server.start_sec is not None else round(waited, 1)
                info["reused"] = waited < 1.0  # 起動済みのサーバーを使い回した
                t0 = time.time()  # 速度比較はサーバー起動時間を除いて測る
                fn = server.transcribe_fn(lang_code(lang), concurrency=st.vllm_concurrency)
                res = core.run_asr(clips, w.get, fn, context=ctx, ctx_terms=terms, ctx_label=st.context_label,
                                   policy=policy, batch_size=st.vllm_concurrency, db=db, on_progress=on_progress, done=done)
            else:
                opts: Dict[str, Any] = dict(p.options)
                opts.update(model=p.model, batch_size=bs)
                if p.engine in ("qwen", "hf-pipeline", "hf-speechlm", "cohere", "granite", "vibevoice"):
                    opts.update(dtype=st.dtype)
                if p.engine == "qwen":
                    opts.update(attn=st.attn, max_new_tokens=st.max_new_tokens)
                info = dict(self.h.ensure_loaded(p.env, "asr", p.engine, opts, self.extra_env(), exclusive_group="asr"))
                t0 = time.time()  # 速度比較はモデルの読み込み時間を除いて測る
                wk = self.h.worker(p.env)
                r = wk.call(
                    "transcribe", key="asr", wav=wav_path, clips=[c.to_dict() for c in clips],
                    done=[d.to_dict() for d in done.values()], language=lang, context=ctx, ctx_terms=terms,
                    ctx_label=st.context_label, policy=asdict(policy), batch_size=bs,
                    on_event=lambda m: on_progress(m.get("stage", ""), int(m.get("done", 0)), int(m.get("total", 0)),
                                                   [core.ClipResult.from_dict(d) for d in m.get("results") or []]),
                )
                res = [core.ClipResult.from_dict(d) for d in r["results"]]
                info = dict(info, gpu_peak_gb=r.get("gpu_peak_gb"))
        finally:
            bar.close()
            if cache_f:
                cache_f.close()
        info["asr_sec"] = round(time.time() - t0, 2)
        if cache_path:
            with open(cache_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({"final": True, "results": [r.to_dict() for r in res]}, ensure_ascii=False) + "\n")
        return res, info

    def _align(self, p: Preset, results: List[core.ClipResult], wav_path: str, st: Settings, lang: Optional[str],
               cache_path: Optional[str], bs: int, quiet: bool = False) -> Dict[int, List[List[Any]]]:
        from tqdm.auto import tqdm

        if not st.aligner:
            return {}
        items = []
        for r in results:
            if not r.text or (r.words and not st.prefer_aligner):
                continue
            L = lang_name(r.language) or (lang or "")
            if L not in ALIGNER_LANGS:
                continue
            if r.clip.dur > 179:
                continue
            items.append({"id": r.clip.id, "start": r.clip.start, "end": r.clip.end, "text": r.text, "language": L})
        if not items:
            return {}
        key = _sha([st.aligner_model, [(d["id"], round(d["start"], 3), round(d["end"], 3), d["text"], d["language"]) for d in items]])
        if cache_path:
            cache_path = f"{cache_path}.{key}.align.json"
            if os.path.exists(cache_path):
                try:
                    return {int(k): v for k, v in json.load(open(cache_path, encoding="utf-8")).items()}
                except Exception:
                    pass
        if not os.path.exists(runtime.env_python(self.aligner_env)):
            log(f"⚠️ アライナー用の環境({self.aligner_env})がないので、タイムスタンプはモデル固有/概算になります")
            return {}
        self.h.ensure_loaded(self.aligner_env, "aligner", self.aligner_kind,
                             {"model": st.aligner_model, "dtype": st.dtype, "attn": st.attn, "batch_size": min(bs, 16)},
                             self.extra_env())
        bar = tqdm(total=len(items), desc="タイムスタンプ", unit="clip", disable=quiet, dynamic_ncols=True)
        try:
            r = self.h.worker(self.aligner_env).call(
                "align", key="aligner", wav=wav_path, items=items, batch_size=min(bs, 16),
                on_event=lambda m: (setattr(bar, "n", int(m.get("done", 0))), bar.refresh()),
            )
        finally:
            bar.close()
        aligned = {int(k): v for k, v in r["aligned"].items()}
        if cache_path:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump({str(k): v for k, v in aligned.items()}, f, ensure_ascii=False)
        return aligned

    def _second_opinion(self, results: List[core.ClipResult], w: core.Wav16, wav_path: str, st: Settings,
                        lang: Optional[str], ctx: str, terms: List[str], db: np.ndarray, bs: int, quiet: bool) -> int:
        """最後まで怪しいクリップだけ別のモデルで読み直し、怪しさが減るなら差し替える。元の結果は attempts に残す"""
        p2 = find_preset(st.second_opinion)
        main = self.preset_of(st)
        if p2 is None or p2.key == main.key:
            log(f"⚠️ セカンドオピニオンのモデルが見つからないか、本命と同じです: {st.second_opinion}")
            return 0
        if p2.engine == "vllm" and main.engine == "vllm":
            log("⚠️ vLLM どうしのセカンドオピニオンはできません(サーバーは1つだけ)")
            return 0
        if p2.engine != "vllm" and not os.path.exists(runtime.env_python(p2.env)):
            log(f"⚠️ セカンドオピニオン用の環境 {p2.env} が未インストールです(① で入れてください)")
            return 0
        bad = [r for r in results if r.flags]
        if not bad:
            return 0
        if not quiet:
            log(f"🩺 セカンドオピニオン: 怪しい {len(bad)} クリップを {p2.label} でも読みます")
        policy = core.RetryPolicy(enabled=False, punctuates=p2.punctuates, lang=lang or "")
        ctx_flags = {"context_label", "context_echo"}
        # context の復唱が疑われるクリップは context なしで、それ以外は context ありで読ませる
        groups = {
            (ctx if p2.context else ""): [r.clip for r in bad if not (set(r.flags) & ctx_flags)],
            "": [r.clip for r in bad if set(r.flags) & ctx_flags],
        } if ctx and p2.context else {"": [r.clip for r in bad]}
        res2: List[core.ClipResult] = []
        server = None
        if p2.engine == "vllm":
            server = self.h.vllm_server("vllm", p2.model, gpu_mem=st.vllm_gpu_mem or 0.4, extra_args=p2.vllm_args,
                                        extra_env=self.extra_env())
        else:
            opts: Dict[str, Any] = dict(p2.options)
            opts.update(model=p2.model, batch_size=bs)
            if p2.engine in ("qwen", "hf-pipeline", "hf-speechlm", "cohere", "granite", "vibevoice"):
                opts.update(dtype=st.dtype)
            self.h.ensure_loaded(p2.env, "asr2", p2.engine, opts, self.extra_env(), exclusive_group="asr2")
        for use_ctx, clips2 in groups.items():
            if not clips2:
                continue
            if server is not None:
                res2 += core.run_asr(clips2, w.get, server.transcribe_fn(lang_code(lang), st.vllm_concurrency),
                                     context=use_ctx, policy=policy, batch_size=st.vllm_concurrency)
            else:
                r = self.h.worker(p2.env).call("transcribe", key="asr2", wav=wav_path, clips=[c.to_dict() for c in clips2],
                                               language=lang, context=use_ctx, policy=asdict(policy), batch_size=bs)
                res2 += [core.ClipResult.from_dict(d) for d in r["results"]]
        by_id = {x.clip.id: x for x in res2}
        adopted = 0
        for r in bad:
            a = by_id.get(r.clip.id)
            if a is None:
                continue
            f2 = core.quality_flags(a.text, r.clip.dur, r.clip.speech, ctx_terms=terms, ctx_label=st.context_label,
                                    punctuates=p2.punctuates, lang=lang or "")
            r.attempts.append({"model": p2.key, "text": a.text, "flags": f2})
            if len(f2) < len(r.flags):
                r.attempts.append({"adopted": p2.key})
                r.text, r.flags, r.words = a.text, f2, a.words
                adopted += 1
        if not quiet:
            log(f"      → {adopted} クリップを {p2.label} の結果に差し替えました(元の結果は JSON と要確認リストに残っています)")
        return adopted

    def _diar_start(self, wav_path: str, st: Settings, cache_path: Optional[str]) -> Optional[Future]:
        if not st.diarize:
            return None
        if not self.hf_token and self.diar_kind == "pyannote":
            log("⚠️ HF_TOKEN が見つからないので話者分離をスキップします(Colab のシークレットに登録してください)")
            return None
        key = _sha([st.diar_model, st.num_speakers, st.min_speakers, st.max_speakers])
        fut: Future = Future()
        if cache_path:
            cache_path = f"{cache_path}.{key}.diar.json"
            if os.path.exists(cache_path):
                try:
                    fut.set_result(json.load(open(cache_path, encoding="utf-8")))
                    return fut
                except Exception:
                    pass
        if not os.path.exists(runtime.env_python(self.diar_env)):
            log(f"⚠️ 話者分離の環境({self.diar_env})がありません。セットアップのセルで入れてください")
            return None
        # ワーカーの起動とモデル読み込みはメインスレッドで(子プロセスの寿命がスレッドに縛られないように)
        self.h.ensure_loaded(self.diar_env, "diar", self.diar_kind, {"model": st.diar_model}, self.extra_env())
        wk = self.h.worker(self.diar_env)

        def job() -> Dict[str, Any]:
            r = wk.call("diarize", key="diar", wav=wav_path, num_speakers=st.num_speakers,
                        min_speakers=st.min_speakers, max_speakers=st.max_speakers)
            d = {"turns": r["turns"], "exclusive": r.get("exclusive"), "seconds": r.get("seconds")}
            if cache_path:
                with open(cache_path, "w", encoding="utf-8") as f:
                    json.dump(d, f)
            return d

        ex = ThreadPoolExecutor(max_workers=1)
        fut2 = ex.submit(job)
        ex.shutdown(wait=False)
        return fut2

    # ---------------------------------------------------------- 1ファイル
    def transcribe_file(self, src: str, st: Settings, *, excerpt: Optional[Tuple[float, float]] = None,
                        out_base: Optional[str] = None, many: bool = False, quiet: bool = False) -> Dict[str, Any]:
        T: Dict[str, float] = {}
        t_all = time.time()
        p = self.preset_of(st)
        st = clip_settings(st, p)  # クリップ長・無音の「自動」をモデルのおすすめ値に
        lang = None if (st.language or "").lower() in ("auto", "", "自動") else st.language
        base = out_base or output_base(src, st.output_dir, many)
        os.makedirs(os.path.dirname(base) or ".", exist_ok=True)
        stem = os.path.basename(base)
        if st.skip_existing and all(os.path.exists(base + _ext(f)) for f in st.formats):
            log(f"⏭️ 出力が揃っているのでスキップ: {stem}")
            return {"skipped": True, "base": base}
        cache_dir = os.path.join(os.path.dirname(base) or ".", ".asr_v3_cache") if st.cache else None
        if cache_dir:
            os.makedirs(cache_dir, exist_ok=True)

        # 1) 音声
        t = time.time()
        wav_path, wav_key = self.prepare_audio(src, st, excerpt)
        w = core.Wav16(wav_path)
        db = core.frame_db(w)
        T["audio"] = time.time() - t
        if not quiet:
            log(f"[1/6] 音声 {core.fmt_dur(w.duration)} を読み込み ({core.fmt_dur(T['audio'])})")

        # 2) 区間検出 → クリップ
        t = time.time()
        segs = self.vad(w, wav_key, st, db)
        max_clip = min(float(st.max_clip), 170.0) if st.aligner else float(st.max_clip)
        clips = core.build_clips(segs, w.duration, max_clip=max_clip, max_gap=st.max_gap, pad=st.vad_pad,
                                 overlap=st.overlap, db=db)
        T["vad"] = time.time() - t
        speech = sum(e - s for s, e in segs)
        if not quiet:
            n_hard = sum(1 for c in clips if c.cut)
            log(f"[2/6] 区間検出({st.vad}) 発話 {core.fmt_dur(speech)} → {len(clips)} クリップ"
                f"(上限 {max_clip:g}秒・無音 {st.max_gap:g}秒でつなぐ / 最長 {max((c.dur for c in clips), default=0):.1f}秒,"
                f" ハード切り {n_hard} か所)")

        # 3) 話者分離(裏で並行して走らせる)
        diar_key = _sha([wav_key])
        diar_cache = os.path.join(cache_dir, f"{stem}.{diar_key}") if cache_dir else None
        diar_fut = self._diar_start(wav_path, st, diar_cache)

        # 4) 文字起こし
        terms = list(st.context_terms)
        ctx = core.build_context(st.context_label, terms, st.context_extra)
        bs = self.batch_size(st, p)
        asr_key = _sha([VERSION, wav_key, [c.to_dict() for c in clips], p.engine, p.model, p.options, lang, ctx,
                        st.retry, st.max_new_tokens])
        asr_cache = os.path.join(cache_dir, f"{stem}.{asr_key}.asr.jsonl") if cache_dir else None
        if not quiet:
            log(f"[3/6] 文字起こし: {p.label} / 言語={lang or '自動'} / バッチ={st.vllm_concurrency if p.engine == 'vllm' else bs}"
                + (f" / context={ctx[:60]}{'…' if len(ctx) > 60 else ''}" if ctx and p.context else ""))
        t = time.time()
        results, asr_info = self._asr(p, w, wav_path, clips, st, lang, ctx, terms, db, asr_cache, bs, quiet)
        T["asr"] = time.time() - t
        n_retry = sum(1 for r in results if len(r.attempts) > 1)
        n_flag = sum(1 for r in results if r.flags)
        if not quiet:
            if asr_info.get("cached"):
                speed = "キャッシュから復元"
            else:  # 倍速はモデルの読み込み・vLLM の起動を除いた推論だけの時間で(⑥の表と同じ)
                infer = float(asr_info.get("asr_sec") or T["asr"])
                load = 0.0 if asr_info.get("reused") else float(asr_info.get("load_sec") or 0.0)
                speed = (f"推論 {core.fmt_dur(infer)}, x{w.duration / max(infer, 1e-6):.0f} 倍速"
                         + (f" / 読み込み {core.fmt_dur(load)}" if load >= 0.5 else ""))
            log(f"      → {len(results)} クリップ / 再推論 {n_retry} / 要確認 {n_flag} ({speed})")

        # 4.5) セカンドオピニオン(怪しいクリップだけ別モデルで)
        if st.second_opinion and n_flag:
            t = time.time()
            n_adopt = self._second_opinion(results, w, wav_path, st, lang, ctx, terms, db, bs, quiet)
            n_flag = sum(1 for r in results if r.flags)
            T["second_opinion"] = time.time() - t
            asr_info["second_opinion_adopted"] = n_adopt

        # 5) タイムスタンプ
        t = time.time()
        aligned = self._align(p, results, wav_path, st, lang, os.path.join(cache_dir, f"{stem}.{asr_key}") if cache_dir else None, bs, quiet)
        words, ts_stats = core.compose_words(results, aligned, segs)
        T["align"] = time.time() - t
        # 本文と字幕の文字数の照合(重なり部分の除去で少し減るのは正常。大きく減ったら警告)
        n_text = sum(core.core_len(r.text) for r in results)
        n_words = sum(core.core_len(w_.word) for w_ in words)
        coverage = n_words / n_text if n_text else 1.0
        ts_stats["coverage"] = round(coverage, 3)
        if n_text and coverage < 0.9:
            log(f"⚠️ 字幕に残った文字が本文の {coverage * 100:.0f}% です(タイムスタンプ付けで欠けた可能性。_review.md を確認してください)")
        n_flag = sum(1 for r in results if r.flags)  # タイムスタンプ付けで付いたフラグ(trimmed)も数える
        if not quiet:
            src_j = {"aligner": "アライナー", "engine": "モデル固有", "approx": "概算"}
            log(f"[4/6] タイムスタンプ {len(words)} 語 (" + ", ".join(f"{src_j.get(k, k)} {v}" for k, v in ts_stats.items()
                                                              if k in src_j) + f", 本文との一致 {coverage * 100:.0f}%)")

        # 6) 話者
        turns: List[List[Any]] = []
        diar_info: Dict[str, Any] = {}
        if diar_fut is not None:
            t = time.time()
            try:
                d = diar_fut.result()
                turns = d.get("turns") or []
                use = d.get("exclusive") or turns
                core.assign_speakers(words, use)
                changed = core.smooth_speakers(words) if st.smooth_speakers else 0
                diar_info = {"speakers": len({k for _, _, k in turns}), "turns": len(turns), "smoothed_words": changed,
                             "seconds": d.get("seconds")}
                if not quiet:
                    log(f"[5/6] 話者分離 {diar_info['speakers']} 人 / {len(turns)} ターン (文単位の補正 {changed} 語)")
            except Exception as e:
                log(f"⚠️ 話者分離に失敗しました(文字起こしは続けます): {e}")
            T["diar_wait"] = time.time() - t
        names = core.speaker_names(words, st.speaker_style, st.speaker_names)

        # 7) 書き出し
        meta = {
            "version": f"asr-v3 {VERSION}",
            "source": os.path.abspath(src),
            "excerpt": list(excerpt) if excerpt else None,
            "duration": round(w.duration, 3),
            "language": lang or "auto",
            "preset": p.key,
            "engine": p.engine,
            "model": p.model,
            "aligner": st.aligner_model if aligned else None,
            "diarization": st.diar_model if turns else None,
            "context": ctx if p.context else "",
            "vad": st.vad,
            "clips": {"n": len(clips), "max_clip": max_clip, "max_gap": st.max_gap},
            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
            "timing_sec": {k: round(v, 2) for k, v in T.items()},
            "asr": asr_info,
            "timestamps": ts_stats,
            "diar": diar_info,
            "gpu": self.gpu.name,
        }
        if turns:
            meta["overlaps"] = core.overlap_regions(turns)  # 同時にしゃべっている時間帯(重なりありの結果から)
        paths, texts = write_outputs(base, words, results, turns, names, st, meta)
        T["total"] = time.time() - t_all
        meta["timing_sec"]["total"] = round(T["total"], 2)
        if not quiet:
            log(f"[6/6] 書き出し完了 (合計 {core.fmt_dur(T['total'])}, x{w.duration / max(T['total'], 1e-6):.0f} 倍速)")
            for k, v in paths.items():
                print(f"   📄 {v}")
        out = {"base": base, "paths": paths, "text": texts.get("plain", ""), "meta": meta, "words": len(words),
               "results": results, "duration": w.duration, "timing": T, "flags": n_flag, "retries": n_retry,
               "wav": wav_path, "names": names}
        self.last = out
        return out

    # ---------------------------------------------------------- 複数ファイル
    def run(self, inputs: Sequence[str], st: Settings, preview_lines: int = 8) -> List[Dict[str, Any]]:
        outs = []
        many = len(inputs) > 1
        for i, src in enumerate(inputs, 1):
            if many:
                print(f"\n========== ({i}/{len(inputs)}) {os.path.basename(src)} ==========")
            else:
                print(f"🎧 {src}")
            try:
                o = self.transcribe_file(src, st, many=many)
            except KeyboardInterrupt:
                log("⏹️ 中断しました(途中までの結果はキャッシュに残っています。もう一度実行すると続きから)")
                raise
            except Exception as e:
                if not many:
                    raise
                log(f"❌ 失敗: {src}: {type(e).__name__}: {e}")
                outs.append({"error": str(e), "src": src})
                continue
            outs.append(o)
            if st.review and o.get("paths", {}).get("review"):
                try:
                    show_review(o, max_items=8)
                except Exception as e:  # 表示だけの失敗で止めない
                    log(f"(要確認の表示に失敗: {e})")
            if preview_lines and o.get("paths", {}).get("txt"):
                with open(o["paths"]["txt"], encoding="utf-8") as f:
                    lines = f.read().splitlines()
                print("\n--- プレビュー ---")
                for l in lines[:preview_lines]:
                    print(l[:200])
                if len(lines) > preview_lines:
                    print(f"… (全 {len(lines)} 行)")
        return outs

    # ---------------------------------------------------------- モデル比較
    def compare(self, src: str, preset_keys: Sequence[str], st: Settings, start: float = 0.0, duration: float = 300.0,
                reference: str = "", out_dir: str = "", keep_loaded: bool = False) -> List[Dict[str, Any]]:
        """同じ区間を複数モデルで文字起こしして、速度・VRAM・(正解があれば)CER を並べる"""
        from IPython.display import HTML, display

        ref = reference
        if ref and os.path.isfile(ref):
            ref = open(ref, encoding="utf-8").read()
        stem = os.path.splitext(os.path.basename(src))[0]
        od = out_dir or os.path.join(os.path.dirname(output_base(src, st.output_dir)) or ".", "compare")
        os.makedirs(od, exist_ok=True)
        rows = []
        for key in preset_keys:
            p = find_preset(key)
            if p is None:
                log(f"⚠️ プリセットが見つかりません: {key}")
                continue
            if not os.path.exists(runtime.env_python(p.env)) and not (p.engine == "vllm" and os.path.exists(runtime.env_dir("vllm"))):
                log(f"⚠️ {p.label}: 環境 {p.env} が未インストールなのでスキップ")
                continue
            print(f"\n===== {p.label} =====")
            st2 = st.replace(preset=p.key, diarize=False, formats=("txt", "srt", "json"))
            t0 = time.time()
            try:
                o = self.transcribe_file(src, st2, excerpt=(start, duration), out_base=os.path.join(od, f"{stem}.{p.key}"), quiet=False)
            except Exception as e:
                log(f"❌ {p.label}: {type(e).__name__}: {str(e)[:500]}")
                rows.append({"preset": p.key, "label": p.label, "error": str(e)[:300]})
                continue
            wall = time.time() - t0
            text = o["text"]
            row = {
                "preset": p.key, "label": p.label, "text": text, "chars": core.core_len(text),
                "asr_sec": o["meta"]["asr"].get("asr_sec") or o["timing"].get("asr"), "wall_sec": wall,
                "load_sec": (o["meta"]["asr"] or {}).get("load_sec"), "gpu_peak_gb": (o["meta"]["asr"] or {}).get("gpu_peak_gb"),
                "flags": o["flags"], "duration": o["duration"], "srt": o["paths"].get("srt"),
                "clips": o["meta"].get("clips"),
            }
            row["x_realtime"] = o["duration"] / max(row["asr_sec"] or wall, 1e-6)
            if ref:
                row["cer"] = core.cer(ref, text)
                row["terms"] = core.term_recall(ref, text, st.context_terms) if st.context_terms else None
                row["numbers"] = core.number_recall(ref, text)
                row["negations"] = core.negation_check(ref, text)
                row["drops"] = core.drop_runs(ref, text)
            elif st.context_terms:
                row["terms_found"] = sum(text.count(t) for t in st.context_terms)
            rows.append(row)
            main_key = self.preset_of(st).key if st.preset else ""
            if not keep_loaded and p.key != main_key:  # 本番で使うモデルは残しておく
                if p.engine == "vllm":
                    self.h.stop_vllm()
                else:
                    self.h.unload(p.env, "asr")
        # 表
        ok = [r for r in rows if "error" not in r]
        if ok and not ref and len(ok) >= 2:
            base_txt = ok[0]["text"]
            for r in ok[1:]:
                r["diff_vs_first"] = core.cer(base_txt, r["text"])
        display(HTML(compare_table_html(rows, bool(ref))))
        if len(ok) >= 2:
            a, b = (ref, ok[0]) if ref else (ok[0]["text"], ok[1])
            title = f"正解 → {b['label']}" if ref else f"{ok[0]['label']} → {b['label']}"
            display(HTML(f"<details><summary><b>差分: {html.escape(title)}</b>(赤=消えた / 緑=増えた)</summary>"
                         f"<div style='white-space:pre-wrap;line-height:1.7;font-size:14px'>{core.diff_html(a, b['text'])}</div></details>"))
        with open(os.path.join(od, f"{stem}.compare.json"), "w", encoding="utf-8") as f:
            json.dump(rows, f, ensure_ascii=False, indent=1, default=str)
        return rows


def _ratio(x: Optional[Tuple[int, int]]) -> str:
    if not x or not x[1]:
        return "-"
    return f"{x[0]}/{x[1]}"


def _neg(x: Optional[Sequence[int]]) -> str:
    if not x:
        return "-"
    hit, tot, extra = x
    return (f"{hit}/{tot}" if tot else "-") + (f" (+{extra})" if extra else "")


def _drops(x: Optional[Sequence[int]]) -> str:
    if not x:
        return "-"
    n, chars = x
    return "なし" if not n else f"<b>{n} か所</b>（{chars} 字）"


def compare_table_html(rows: List[Dict[str, Any]], has_ref: bool) -> str:
    th = ("<tr><th>モデル</th><th>推論</th><th>倍速</th><th>読み込み</th><th>VRAM峰</th>"
          + ("<th>CER</th><th>用語</th><th>数字</th><th>否定</th><th>抜け</th>" if has_ref
             else "<th>1行目との差</th><th>用語の出現</th>")
          + "<th>要確認</th><th>冒頭</th></tr>")
    trs = []
    for r in rows:
        if "error" in r:
            trs.append(f"<tr><td>{html.escape(r['label'])}</td><td colspan=11 style='color:#c00'>{html.escape(r['error'][:200])}</td></tr>")
            continue
        metric = r.get("cer") if has_ref else r.get("diff_vs_first")
        mtxt = "-" if metric is None else f"{metric * 100:.1f}%"
        extra = (f"<td>{_ratio(r.get('terms'))}</td><td>{_ratio(r.get('numbers'))}</td>"
                 f"<td>{_neg(r.get('negations'))}</td><td>{_drops(r.get('drops'))}</td>" if has_ref
                 else f"<td>{r.get('terms_found', '-')}</td>")
        cl = r.get("clips") or {}
        ctxt = (f"<br><small style='color:#666'>{cl.get('n')} クリップ（上限 {cl.get('max_clip'):g}秒・無音 {cl.get('max_gap'):g}秒でつなぐ）</small>"
                if cl.get("max_clip") else "")
        load = r.get("load_sec")
        trs.append(
            f"<tr><td>{html.escape(r['label'])}{ctxt}</td><td>{(r.get('asr_sec') or 0):.1f}秒</td><td>x{r['x_realtime']:.0f}</td>"
            f"<td>{'-' if load is None else f'{load:.0f}秒'}</td>"
            f"<td>{r.get('gpu_peak_gb') or '-'}</td><td><b>{mtxt}</b></td>{extra}<td>{r.get('flags', 0)}</td>"
            f"<td style='max-width:520px'>{html.escape(r['text'][:160])}…</td></tr>")
    return ("<table style='border-collapse:collapse;font-size:13px' border=1 cellpadding=4>" + th + "".join(trs) + "</table>"
            + "<div style='font-size:12px;color:#666'>推論・倍速はモデルの読み込み(vLLM はサーバーの起動)を除いた時間。"
              "読み込みはそのモデルを最後に読み込んだときの時間です。</div>"
            + ("<div style='font-size:12px;color:#666'>CER は句読点・空白を除き NFKC 正規化した文字誤り率(低いほど良い)。"
               "用語 = 正解に出てくる context の語を正しく書けた数、数字 = 正解の数字を同じ形で書けた数(金額・日付の取り違えの目安)、"
               "否定 = 正解の「ない・ません」などが同じ所に残った数(+ は正解に無い否定。意味の反転の目安)、"
               "抜け = 正解の 8 字以上がまとめて抜けた箇所(発話の読み飛ばしの目安)。</div>" if has_ref else
               "<div style='font-size:12px;color:#666'>正解テキストがないので、1行目のモデルとの文字の食い違い率を参考表示しています(どちらが正しいかは分かりません)。</div>"))


def _ext(fmt: str) -> str:
    return {"plain": "_plain.txt", "review": "_review.md"}.get(fmt, "." + fmt)


def write_outputs(base: str, words: List[core.Word], results: List[core.ClipResult], turns: List[List[Any]],
                  names: Dict[str, str], st: Settings, meta: Dict[str, Any]) -> Tuple[Dict[str, str], Dict[str, str]]:
    fill = {"off": None, "safe": core.make_filler_regex(core.FILLERS_SAFE),
            "more": core.make_filler_regex(core.FILLERS_SAFE + core.FILLERS_MORE)}.get(st.fillers)
    post = core.TextPost(core.parse_replacements(st.replacements), fill, True, st.halfwidth)
    sents = core.make_sentences(words, post)
    paras = core.make_paragraphs(sents)
    cues = core.make_cues(words, post, core.CueRules(max_chars=int(st.cue_max_chars), max_dur=float(st.cue_max_dur),
                                                    gap=float(st.cue_gap)))
    paths: Dict[str, str] = {}
    texts: Dict[str, str] = {"plain": core.to_txt(paras, names, timestamps=False)}

    def put(fmt: str, text: str, enc: str = "utf-8") -> None:
        pth = base + _ext(fmt)
        tmp = pth + ".tmp"
        with open(tmp, "w", encoding=enc, newline="") as f:
            f.write(text)
        os.replace(tmp, pth)
        paths[fmt] = pth

    stem = os.path.basename(base)
    fm = set(st.formats)
    if "txt" in fm:
        put("txt", core.to_txt(paras, names))
    if "srt" in fm:
        put("srt", core.to_srt(cues, names, st.speaker_fmt))
    if "vtt" in fm:
        put("vtt", core.to_vtt(cues, names))
    if "json" in fm:
        put("json", core.to_json(sents, names, results, meta))
    if "csv" in fm:
        put("csv", core.to_csv(sents, names), enc="utf-8-sig")
    if "md" in fm:
        info = {"音源": os.path.basename(meta.get("source", "")), "長さ": core.fmt_dur(meta.get("duration", 0)),
                "モデル": meta.get("model"), "作成": meta.get("created")}
        if names:
            info["話者"] = "、".join(dict.fromkeys(names.values()))
        put("md", core.to_markdown(paras, names, stem, info))
    if "plain" in fm:
        put("plain", texts["plain"])
    if "rttm" in fm and turns:
        put("rttm", core.to_rttm(turns, stem, names))
    rv = base + "_review.md"
    if st.review and core.review_items(results):
        put("review", core.to_review_md(results, stem))
    elif os.path.exists(rv):
        os.remove(rv)  # 前回の古い要確認リストが残らないように
    return paths, texts


# =====================================================================
# 要確認リストを音声つきで表示
# =====================================================================


def _clip_audio_html(wav_path: str, start: float, end: float) -> str:
    """区間を小さい Opus にして <audio> に埋め込む(ノートブックが重くならないよう 24kbps)"""
    import base64
    import subprocess

    cmd = ["ffmpeg", "-loglevel", "error", "-ss", f"{max(0.0, start):.2f}", "-t", f"{max(0.3, end - start):.2f}",
           "-i", wav_path, "-c:a", "libopus", "-b:a", "24k", "-f", "ogg", "pipe:1"]
    p = subprocess.run(cmd, capture_output=True)
    if p.returncode != 0 or not p.stdout:
        return ""
    b64 = base64.b64encode(p.stdout).decode()
    return f"<audio controls preload='none' src='data:audio/ogg;base64,{b64}'></audio>"


def show_review(out: Dict[str, Any], max_items: int = 8) -> None:
    from IPython.display import HTML, display

    items = core.review_items(out.get("results") or [])
    if not items:
        return
    groups: List[List[core.ClipResult]] = []  # 2分割したものは元のクリップごとにまとめる
    for r in items:
        if r.clip.parent is not None and groups and groups[-1][0].clip.parent == r.clip.parent:
            groups[-1].append(r)
        else:
            groups.append([r])
    rows = []
    for g in groups[:max_items]:
        a0, a1 = g[0].clip.start, g[-1].clip.end
        flags = [f for r in g for f in r.flags]
        why = "、".join(dict.fromkeys(core.FLAG_JA.get(f, f) for f in flags)) if flags else "自動で差し替え済み"
        adopted = "".join(r.text for r in g)
        first = next((a.get("text") for a in g[0].attempts if "text" in a), adopted) or ""
        body = (f"<div style='line-height:1.7'>{core.diff_html(first, adopted)}</div>"
                "<div style='color:#888;font-size:11px'>赤=最初の結果から消えた / 緑=差し替えで増えた</div>"
                if first != adopted else f"<div>{html.escape(adopted[:300] or '(空)')}</div>")
        rows.append(
            f"<tr><td style='white-space:nowrap'>{core.fmt_hms(a0)}〜{core.fmt_hms(a1)}</td>"
            f"<td>{_clip_audio_html(out['wav'], a0, a1)}</td><td><b>{html.escape(why)}</b>{body}</td></tr>")
    more = (f"<div>ほか {len(groups) - max_items} 件は {html.escape(out['paths'].get('review', ''))} を見てください</div>"
            if len(groups) > max_items else "")
    display(HTML(f"<details open><summary><b>要確認 {len(groups)} 件</b>(音声を聞いて確かめられます)</summary>"
                 f"<table border=1 cellpadding=4 style='border-collapse:collapse;font-size:13px'>{''.join(rows)}</table>{more}</details>"))


# =====================================================================
# 議事録づくり(プロンプトを作る / Claude API で作る)
# =====================================================================

MINUTES_SYSTEM = """あなたは、会議の文字起こしから正確で読みやすい日本語の議事録を作るアシスタントです。
- 文字起こしは音声認識の結果なので、同音異義語の誤変換・固有名詞の誤り・言い直しやフィラーが含まれます。文脈から明らかな誤りは正しく直してかまいませんが、発言にない内容を推測で足さないでください。
- 自信がない固有名詞・数字・日付は【要確認】と書き添えてください。
- 話者ラベル(「話者A」など)は、文字起こし内で名前が明らかな場合だけ名前に置き換えてください。
- 重要な内容には、根拠になった時刻を [hh:mm:ss] の形で添えてください。"""

MINUTES_STYLES = {
    "議事録（決定事項・TODO つき）": """<transcript> の内容から議事録を Markdown で作ってください。構成:
# 会議名(わからなければ「会議」)
- 日時・参加者(文字起こしから分かる範囲)
## 要約(3〜5行)
## 議題ごとの内容(議題ごとに ### 見出し。誰が何を言ったか、根拠の時刻)
## 決定事項
## TODO(| 担当 | 期限 | 内容 | の表。分からない欄は「未定」)
## 保留・次回への持ち越し
## 要確認(聞き取りが怪しい箇所・数字・固有名詞)""",
    "要約（3行＋詳細）": """<transcript> の内容を要約してください。最初に3行の要約、続けて論点ごとの詳細(箇条書き・根拠の時刻つき)、最後に要確認の点を書いてください。""",
    "発言者ごとの要点": """<transcript> について、発言者ごとに主張・提案・懸念・引き受けたことを箇条書きでまとめてください(根拠の時刻つき)。最後に、発言者どうしで意見が分かれた点を整理してください。""",
}


def _minutes_request(text: str, style: str, glossary: Sequence[str] = ()) -> Tuple[str, str]:
    instr = MINUTES_STYLES.get(style, style)
    if glossary:
        instr += "\n\n参考: この会議で出てくる固有名詞・専門用語の候補: " + "、".join(glossary)
    return instr, f"<transcript>\n{text}\n</transcript>"


def make_minutes(transcript: str, *, style: str = "議事録（決定事項・TODO つき）", mode: str = "プロンプトだけ作る",
                 model: str = "claude-opus-5", glossary: Sequence[str] = (), out_path: str = "",
                 gemini_model: str = "google/gemini-3.5-flash") -> Optional[str]:
    """文字起こし(ファイルパス or 本文)から議事録を作る。書き出したファイルのパスを返す

    mode: "プロンプトだけ作る" / "Gemini（Colab AI・無料）" / "Claude API"
    """
    src_path = transcript if os.path.isfile(transcript) else ""
    text = open(src_path, encoding="utf-8").read() if src_path else transcript
    if not text.strip():
        raise ValueError("文字起こしが空です")
    base = os.path.splitext(src_path)[0] if src_path else os.path.join(runtime.BASE_DIR, "minutes")
    instr, tr = _minutes_request(text, style, glossary)

    if mode.startswith("プロンプト"):
        pth = out_path or base + "_minutes_prompt.md"
        with open(pth, "w", encoding="utf-8") as f:
            f.write(MINUTES_SYSTEM + "\n\n" + instr + "\n\n" + tr + "\n")
        log(f"📝 プロンプトを書き出しました: {pth}\n   中身をまるごと Claude / ChatGPT に貼り付けてください(約 {len(text):,} 文字)")
        return pth

    if mode.startswith("Gemini"):
        # Colab に組み込みの google.colab.ai(API キー不要。2026年6月から全ユーザー無料)
        try:
            from google.colab import ai
        except Exception as e:
            raise RuntimeError(f"google.colab.ai が使えません(Colab の画面から実行してください): {e}")
        log(f"🤖 {gemini_model}(Colab AI)で作成中…")
        chunks: List[str] = []
        for piece in ai.generate_text(MINUTES_SYSTEM + "\n\n" + instr + "\n\n" + tr, model_name=gemini_model, stream=True):
            if piece:
                print(piece, end="", flush=True)
                chunks.append(piece)
        print()
        pth = out_path or base + "_minutes.md"
        with open(pth, "w", encoding="utf-8") as f:
            f.write("".join(chunks))
        log(f"✅ 議事録を書き出しました: {pth}")
        return pth

    # ---- Claude API ----
    try:
        import anthropic
    except ImportError:
        import subprocess
        import sys

        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
        import anthropic
    key = runtime.get_secret("ANTHROPIC_API_KEY")
    if not key and not os.environ.get("ANTHROPIC_AUTH_TOKEN"):
        raise RuntimeError("Colab のシークレットに ANTHROPIC_API_KEY を登録して、このノートブックのアクセスを ON にしてください")
    client = anthropic.Anthropic(api_key=key) if key else anthropic.Anthropic()
    req: Dict[str, Any] = dict(
        model=model, max_tokens=64000, system=MINUTES_SYSTEM,
        messages=[{"role": "user", "content": [
            # 文字起こしを先頭に置いてキャッシュ(同じ文字起こしで別のスタイルを試すとき安くなる)
            {"type": "text", "text": tr, "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": instr},
        ]}],
    )
    # 安全フィルタで断られたとき、サーバー側で別モデルに引き継ぐ(Opus 5 系・Fable 系で有効)
    use_fallback = model.startswith(("claude-opus-5", "claude-fable-5"))
    log(f"🤖 {model} で作成中…(長い会議だと数分かかります)")
    try:
        if use_fallback:
            cm = client.beta.messages.stream(**req, betas=["server-side-fallback-2026-07-01"], fallbacks="default")
        else:
            cm = client.messages.stream(**req)
        with cm as stream:
            for chunk in stream.text_stream:
                print(chunk, end="", flush=True)
            msg = stream.get_final_message()
    except anthropic.AuthenticationError:
        raise RuntimeError("API キーが正しくないようです(ANTHROPIC_API_KEY を確認してください)")
    except anthropic.RateLimitError:
        raise RuntimeError("レート制限にかかりました。少し待ってからもう一度実行してください")
    except anthropic.APIStatusError as e:
        raise RuntimeError(f"Claude API のエラー ({e.status_code}): {e.message}")
    except anthropic.APIConnectionError:
        raise RuntimeError("Claude API に接続できませんでした(ネットワークを確認してください)")
    print()
    if msg.stop_reason == "refusal":
        log("⚠️ モデルが応答を断りました。内容を確認して、別のモデルかプロンプトだけ作るモードを試してください")
        return None
    out = "".join(b.text for b in msg.content if b.type == "text")
    if msg.stop_reason == "max_tokens":
        log("⚠️ 出力が上限で途中まで切れています")
    pth = out_path or base + "_minutes.md"
    with open(pth, "w", encoding="utf-8") as f:
        f.write(out)
    u = msg.usage
    log(f"✅ 議事録を書き出しました: {pth}\n   トークン: 入力 {u.input_tokens:,} / キャッシュ読 {getattr(u, 'cache_read_input_tokens', 0) or 0:,}"
        f" / キャッシュ書 {getattr(u, 'cache_creation_input_tokens', 0) or 0:,} / 出力 {u.output_tokens:,}")
    return pth
'''
for _p, _src in _FILES.items():
    _fp = os.path.join(_LIB, "asrkit", _p)
    os.makedirs(os.path.dirname(_fp), exist_ok=True)
    with open(_fp, "w", encoding="utf-8") as _f:
        _f.write(_src)
if _LIB not in sys.path:
    sys.path.insert(0, _LIB)
if "SESS" in globals():  # 作り直す前に古いワーカーを止める
    try:
        SESS.free()
    except Exception:
        pass
    del SESS
import asrkit
from asrkit import core, runtime, engines, presets, pipeline
for _m in (asrkit, core, runtime, engines, presets, pipeline):
    importlib.reload(_m)
print(f"✅ asrkit {asrkit.__version__} を読み込みました ({_LIB})")

In [ ]:
#@title ① セットアップ（インストール）
#@markdown 使うものにチェック。入れたものは次から一瞬でスキップされます（ランタイムを削除するまで）。
#@markdown
#@markdown **Qwen3-ASR（必須）**: 標準モデル＋タイムスタンプ用のアライナー
qwen = True  #@param {type:"boolean"}
#@markdown **話者分離（pyannote）**: 「誰が話したか」を付ける。Hugging Face の `HF_TOKEN` が必要（下の説明）
pyannote = True  #@param {type:"boolean"}
#@markdown **vLLM**: Qwen3-ASR / Cohere を大幅に高速化（A100 / H100 で真価。インストール 3〜6 分）
vllm = False  #@param {type:"boolean"}
#@markdown **faster-whisper**: Whisper large-v3 / large-v3-turbo / kotoba-whisper を試す
faster_whisper = False  #@param {type:"boolean"}
#@markdown **NeMo**: NVIDIA Parakeet 日本語モデルを試す（インストール数分）
nemo = False  #@param {type:"boolean"}
#@markdown **最新 transformers**: Cohere Transcribe / Granite Speech / VibeVoice-ASR や「カスタム」を試す用
hf = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **flash-attn**: A100/H100/L4 で Qwen を少し速く・省メモリに（配布済み whl があるときだけ。なければ sdpa で動きます）
flash_attn = True  #@param {type:"boolean"}
#@markdown whl が無いときにソースからビルドする（20〜30分）
build_flash_if_missing = False  #@param {type:"boolean"}
#@markdown **Google Drive をマウント**（音声が Drive にあるとき）
mount_drive = True  #@param {type:"boolean"}
#@markdown ---
#@markdown **HF_TOKEN の準備（話者分離を使うとき・初回だけ）**
#@markdown 1. https://huggingface.co/settings/tokens で Read トークンを作る
#@markdown 2. https://huggingface.co/pyannote/speaker-diarization-community-1 で規約に同意
#@markdown 3. Colab 左の🔑（シークレット）に `HF_TOKEN` という名前で登録し、このノートブックのアクセスを ON

import os, sys, subprocess, time
g = runtime.gpu_info()
if g.ok:
    print(f"GPU: {g.name} / {g.mem_gb:.0f}GB / compute {g.cc[0]}.{g.cc[1]} / driver {g.driver} (CUDA {g.cuda})")
else:
    print("⚠️ GPU が見つかりません。メニュー「ランタイム → ランタイムのタイプを変更」で GPU を選んでください")

# --- カーネル側に VAD だけ入れる(軽い) ---
_need = []
for _mod, _pkg in (("fireredvad", "fireredvad"), ("silero_vad", "silero-vad"), ("rapidfuzz", "rapidfuzz")):
    try:
        __import__(_mod)
    except Exception:
        _need.append(_pkg)
if _need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=False)

# --- flash-attn (qwen 環境にだけ入れる) ---
_specs = []
_names = [n for n, on in (("qwen", qwen), ("pyannote", pyannote), ("vllm", vllm), ("fw", faster_whisper),
                          ("nemo", nemo), ("hf", hf)) if on]
for _n in _names:
    _s = presets.ENV_SPECS[_n]
    if _n == "qwen" and flash_attn and g.ampere_plus:
        _whl = presets.flash_attn_wheel()
        if _whl:
            _s = runtime.EnvSpec(**{**_s.__dict__, "post": [[_whl]]})
        elif build_flash_if_missing:
            print("flash-attn の配布 whl が無いのでソースからビルドします(20〜30分)…")
            _s = runtime.EnvSpec(**{**_s.__dict__, "post": [["ninja"], ["flash-attn", "--no-build-isolation"]]})
        else:
            print("ℹ️ いまの torch に合う flash-attn の whl が無いので sdpa で動かします(速度差は小さめ)")
    _specs.append(_s)
if vllm and g.ok and not g.ampere_plus:
    print("ℹ️ T4 でも vLLM は動きますが、速さの差が大きいのは A100 / L4 / H100 です")

t0 = time.time()
_st = runtime.ensure_envs(_specs)
print(f"\nセットアップ完了 ({core.fmt_dur(time.time() - t0)})" + ("" if all(s.ok for s in _st) else " ※失敗したものがあります(上のログ)"))

if pyannote and not runtime.get_secret("HF_TOKEN"):
    print("⚠️ HF_TOKEN が見つかりません。話者分離を使うなら上の手順でシークレットに登録してください")
if mount_drive and runtime.in_colab() and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if "SESS" not in globals():
    SESS = pipeline.Session()

In [ ]:
#@title ② 音声とモデル
#@markdown **音源**: ファイル / フォルダ（中の音声・動画をぜんぶ）/ ワイルドカード（例 `/content/drive/MyDrive/会議/*.m4a`）。改行やカンマで複数指定も可。**空欄なら実行時にアップロード画面**が出ます
src = ""  #@param {type:"string"}
#@markdown **出力先**: 空欄なら音源と同じフォルダ。フォルダを指定するとその中に音源名で作ります
output_dir = ""  #@param {type:"string"}
#@markdown ---
#@markdown **モデル**（⑥で比べられます。「カスタム」は ④ で指定）
model = "Qwen3-ASR 1.7B（標準・おすすめ）"  #@param ["Qwen3-ASR 1.7B（標準・おすすめ）", "Qwen3-ASR 0.6B（軽量・速い）", "Qwen3-ASR 1.7B JA（neosophie・固有名詞に強い日本語調整版）", "Qwen3-ASR 1.7B × vLLM（A100/H100で爆速）", "Cohere Transcribe 2B（別系統の有力候補）", "Cohere Transcribe 2B × vLLM", "Whisper large-v3-turbo（faster-whisper）", "Whisper large-v3（faster-whisper）", "kotoba-whisper v2.0（日本語特化 Whisper）", "Parakeet TDT-CTC 0.6B ja（NVIDIA・日本語特化）", "Granite Speech 4.1 2B（IBM・実験的）", "VibeVoice-ASR 8B（Microsoft・実験的）", "カスタム"]
#@markdown **言語**（auto で自動判定。Qwen3-ASR は 30 言語＋中国語方言）
language = "Japanese"  #@param ["Japanese", "auto", "English", "Chinese", "Korean", "Cantonese", "French", "German", "Spanish", "Portuguese", "Italian", "Russian", "Thai", "Vietnamese", "Indonesian"] {allow-input: true}
#@markdown ---
#@markdown **context_label**: 語のリストを包む見出し（[issue #321](https://github.com/TypeWhisper/typewhisper-mac/issues/321) 方式。見出しで包むと精度UP・漏れ激減）
context_label = "固有名詞・専門用語"  #@param ["固有名詞・専門用語", "Proper nouns", "Technical terms", "Vocabulary"] {allow-input: true}
#@markdown **context**: 固有名詞・専門用語（スペース/読点/カンマ区切り）
context = ""  #@param {type:"string", placeholder:"Claude Codex Gemini 競争 脱落"}
#@markdown **用語ファイル**（任意）: 1行1語などのテキストファイル。context に足されます
context_file = ""  #@param {type:"string"}
#@markdown *Tips*: 会議資料を Claude/ChatGPT に渡して「固有名詞・専門用語を読点区切りで列挙して」と頼むと楽。入れすぎると復唱を誘発しやすいけど、v3 は復唱を見つけたら context なしで自動でやり直します
#@markdown ---
#@markdown **出力形式**
txt = True  #@param {type:"boolean"}
srt = True  #@param {type:"boolean"}
json_ = True  #@param {type:"boolean"}
vtt = False  #@param {type:"boolean"}
csv = False  #@param {type:"boolean"}
#@markdown `csv` は Excel 用（BOM 付き）、`md` は議事録づくり用の Markdown、`plain` はタイムスタンプなしのテキスト
md = True  #@param {type:"boolean"}
plain = False  #@param {type:"boolean"}

_fmts = tuple(f for f, on in (("txt", txt), ("srt", srt), ("json", json_), ("vtt", vtt), ("csv", csv), ("md", md), ("plain", plain)) if on)
_terms = core.parse_terms(context)
if context_file.strip():
    _terms += [w for w in core.parse_terms(open(context_file.strip(), encoding="utf-8").read()) if w not in _terms]
print("モデル:", model, "/ 言語:", language)
print("context:", core.build_context(context_label, _terms) or "(なし)")
print("出力形式:", ", ".join(_fmts))

In [ ]:
#@title ③ 話者分離
#@markdown **誰が話したか**を付けます（① で pyannote を入れて HF_TOKEN を登録しておく）。文字起こしと並行して走ります
diarize = True  #@param {type:"boolean"}
#@markdown **人数**: わかっていれば入れると精度が上がる（0 = 自動）。幅で指定したいときは min/max
num_speakers = 0  #@param {type:"integer"}
min_speakers = 0  #@param {type:"integer"}
max_speakers = 0  #@param {type:"integer"}
#@markdown **話者名**: `話者A=田中, 話者B=佐藤` のように置き換え（登場順に `田中, 佐藤` だけでもOK）。一度流して確認してから入れると楽
speaker_names = ""  #@param {type:"string"}
#@markdown **表記**
speaker_style = "話者A"  #@param ["話者A", "SPEAKER_00", "S1"]
#@markdown **文単位の補正**: 文の途中で話者がチラつくのを多数決で直す
smooth_speakers = True  #@param {type:"boolean"}
#@markdown **モデル**
diar_model = "pyannote/speaker-diarization-community-1"  #@param ["pyannote/speaker-diarization-community-1", "pyannote/speaker-diarization-3.1"] {allow-input: true}
#@markdown **RTTM も出力**（話者分離の評価ツール用）
rttm = False  #@param {type:"boolean"}

In [ ]:
#@title ④ 詳細設定（ふだんは触らなくてOK）
#@markdown ### 音声
#@markdown **チャンネル**: ステレオで左右に別の人のマイクが入っているときなど
channel = "mix"  #@param ["mix", "left", "right"]
#@markdown **音声フィルタ**: 小さい声が多い会議は dynaudnorm、ノイズが多いときは afftdn（ffmpeg のフィルタを直接書いてもOK）
audio_filter = "なし"  #@param ["なし", "音量をそろえる (loudnorm)", "小さい声を持ち上げる (dynaudnorm)", "ノイズを少し減らす (afftdn)", "低音ノイズ除去+音量 (highpass+loudnorm)"] {allow-input: true}
#@markdown ### 区間検出（VAD）とクリップ
vad = "fireredvad"  #@param ["fireredvad", "silero", "energy", "none"]
#@markdown **vad_threshold**: 発話と判定するしきい値（上げると厳しめ＝区間が減る）
vad_threshold = 0.4  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
#@markdown **max_clip_sec**: 1回で ASR に渡す最大秒数。「自動」はモデルのおすすめ（ふつう 30 秒。kotoba-whisper などは短め）。長いほど文脈が効くが重い
max_clip_sec = "自動"  #@param ["自動", "10", "15", "20", "30", "45", "60"] {allow-input: true}
#@markdown **max_gap_sec**: これより長い無音をはさむ発話は別クリップにする。「自動」はモデルのおすすめ（ふつう 6 秒）
max_gap_sec = "自動"  #@param ["自動", "1", "2", "3", "6", "10"] {allow-input: true}
#@markdown **overlap_sec**: ハード切り（長い発話の強制分割）の前後の重なり。重複は自動で除去
overlap_sec = 1.0  #@param {type:"number"}
#@markdown ### 推論
#@markdown **batch_size**: 0 = GPU のメモリから自動（OOM したら自動で半分にして続行）
batch_size = 0  #@param {type:"integer"}
max_new_tokens = 1024  #@param {type:"integer"}
#@markdown **retry**: 怪しい結果（context の復唱・ループ・空振り）を自動でやり直す
retry = True  #@param {type:"boolean"}
#@markdown **aligner**: Qwen3-ForcedAligner で単語タイムスタンプを付ける（OFF だとモデル固有 or 概算）
aligner = True  #@param {type:"boolean"}
#@markdown **vllm_concurrency**: vLLM に同時に投げるクリップ数
vllm_concurrency = 64  #@param {type:"integer"}
#@markdown **セカンドオピニオン**: 最後まで怪しいクリップだけ別のモデルでも読んで、怪しさが減るなら差し替える（元の結果も残る。① でそのモデルの環境を入れておく）
second_opinion = "なし"  #@param ["なし", "Qwen3-ASR 1.7B（標準・おすすめ）", "Qwen3-ASR 0.6B（軽量・速い）", "Qwen3-ASR 1.7B JA（neosophie・固有名詞に強い日本語調整版）", "Qwen3-ASR 1.7B × vLLM（A100/H100で爆速）", "Cohere Transcribe 2B（別系統の有力候補）", "Cohere Transcribe 2B × vLLM", "Whisper large-v3-turbo（faster-whisper）", "Whisper large-v3（faster-whisper）", "kotoba-whisper v2.0（日本語特化 Whisper）", "Parakeet TDT-CTC 0.6B ja（NVIDIA・日本語特化）", "Granite Speech 4.1 2B（IBM・実験的）", "VibeVoice-ASR 8B（Microsoft・実験的）"]
#@markdown **要確認リスト**: 怪しい区間・自動で直した区間を `_review.md` に書き出し、実行後に音声つきで表示
review = True  #@param {type:"boolean"}
#@markdown **カスタム**（② で「カスタム」を選んだとき）: エンジンと Hugging Face のモデル ID
custom_engine = "hf-pipeline"  #@param ["hf-pipeline", "hf-speechlm", "cohere", "granite", "vibevoice", "faster-whisper", "nemo", "qwen", "vllm"]
custom_model = ""  #@param {type:"string"}
#@markdown ### テキスト整形
#@markdown **fillers**: 「えー」「あのー」などを消す（safe = 伸ばし音のフィラーだけ / more = 「あの」「まあ」なども）
fillers = "off"  #@param ["off", "safe", "more"]
#@markdown **置換辞書**: `誤=>正` を `;` 区切りで（例 `クロード=>Claude; re:ジェミ[ニ二]=>Gemini`）。ファイル（1行1ルール）も可
replacements = ""  #@param {type:"string"}
replacements_file = ""  #@param {type:"string"}
#@markdown 全角英数字を半角に
halfwidth = True  #@param {type:"boolean"}
#@markdown ### 字幕（SRT/VTT）
cue_max_chars = 30  #@param {type:"integer"}
cue_max_sec = 6.0  #@param {type:"number"}
#@markdown これ以上の無音で字幕を切る（秒）
cue_gap_sec = 0.8  #@param {type:"number"}
#@markdown 話者つき字幕の書式
speaker_fmt = "{speaker}: {text}"  #@param ["{speaker}: {text}", "【{speaker}】{text}", "（{speaker}）{text}"] {allow-input: true}
#@markdown ### そのほか
#@markdown **キャッシュ**: 出力先の `.asr_v3_cache/` に途中結果を保存（切断されても続きから）
use_cache = True  #@param {type:"boolean"}
#@markdown 出力がそろっているファイルは飛ばす（フォルダ一括のとき便利）
skip_existing = False  #@param {type:"boolean"}

In [ ]:
#@title ⑤ 実行
#@markdown 終わったら結果を zip でダウンロード（アップロードで使ったとき便利）
download_zip = False  #@param {type:"boolean"}
import os, glob
_AF = {"なし": "", "音量をそろえる (loudnorm)": "loudnorm=I=-20:TP=-2:LRA=11",
       "小さい声を持ち上げる (dynaudnorm)": "dynaudnorm=f=250:g=15",
       "ノイズを少し減らす (afftdn)": "afftdn=nf=-25",
       "低音ノイズ除去+音量 (highpass+loudnorm)": "highpass=f=80,loudnorm=I=-20:TP=-2:LRA=11"}
_rep = replacements.replace(";", "\n")
if replacements_file.strip():
    _rep += "\n" + open(replacements_file.strip(), encoding="utf-8").read()
ST = pipeline.Settings(
    output_dir=output_dir, formats=_fmts + (("rttm",) if rttm else ()), skip_existing=skip_existing,
    preset=("custom" if model == "カスタム" else model), custom_engine=custom_engine, custom_model=custom_model,
    language=language, batch_size=batch_size, max_new_tokens=max_new_tokens, aligner=aligner,
    channel=channel, af=_AF.get(audio_filter, audio_filter),
    vad=vad, vad_threshold=vad_threshold, max_clip=pipeline.auto_num(max_clip_sec), max_gap=pipeline.auto_num(max_gap_sec),
    overlap=overlap_sec,
    context_label=context_label, context_terms=_terms, retry=retry,
    replacements=_rep, fillers=fillers, halfwidth=halfwidth,
    cue_max_chars=cue_max_chars, cue_max_dur=cue_max_sec, cue_gap=cue_gap_sec, speaker_fmt=speaker_fmt,
    diarize=diarize, diar_model=diar_model, num_speakers=num_speakers, min_speakers=min_speakers,
    max_speakers=max_speakers, speaker_names=speaker_names, speaker_style=speaker_style, smooth_speakers=smooth_speakers,
    vllm_concurrency=vllm_concurrency, cache=use_cache,
    second_opinion=("" if second_opinion == "なし" else second_opinion), review=review,
)
if "SESS" not in globals():
    SESS = pipeline.Session()

# 音源: 空欄ならアップロード
_src = src.strip()
if not _src:
    from google.colab import files
    print("音声/動画ファイルを選んでください")
    _up = files.upload()
    _src = "\n".join(os.path.abspath(k) for k in _up)
if "/content/drive" in _src + output_dir and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
INPUTS = pipeline.resolve_inputs(_src)
if not INPUTS:
    raise FileNotFoundError(f"音声ファイルが見つかりません: {_src}")
print(f"{len(INPUTS)} ファイル: " + ", ".join(os.path.basename(p) for p in INPUTS[:5]) + (" …" if len(INPUTS) > 5 else ""))
OUTS = SESS.run(INPUTS, ST)

if download_zip:
    import zipfile
    from google.colab import files
    _zip = "/content/asr_v3_outputs.zip"
    with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for o in OUTS:
            for p in (o.get("paths") or {}).values():
                z.write(p, os.path.basename(p))
    files.download(_zip)

---
## おまけ
- **⑥ モデル比較**: 同じ区間をいろいろなモデルで文字起こしして比べる
- **⑦ 議事録づくり**: 文字起こし結果から議事録を作る（プロンプトを作るだけ / Claude API）

In [ ]:
#@title ⑥ モデル比較（おまけ）
#@markdown 同じ区間を複数のモデルで文字起こしして、**速さ・VRAM・文字誤り率(CER)** を並べます。① で環境を入れたモデルだけ動きます
#@markdown 結果は出力先の `compare/` フォルダに（モデルごとの txt / srt / json）
#@markdown Qwen3-ASR 1.7B（標準・おすすめ）（v1/v2 と同じ。context で固有名詞に強い）
cmp_qwen3_1_7b = True  #@param {type:"boolean"}
#@markdown Qwen3-ASR 0.6B（軽量・速い）（1.7B より少し精度が落ちるぶん速い）
cmp_qwen3_0_6b = False  #@param {type:"boolean"}
#@markdown Qwen3-ASR 1.7B JA（neosophie・固有名詞に強い日本語調整版）（IT・ビジネス系の固有名詞 F1 が 0.59→0.65。全体の CER は 8.23%→8.92% と少し悪化の報告）
cmp_qwen3_1_7b_ja = True  #@param {type:"boolean"}
#@markdown Qwen3-ASR 1.7B × vLLM（A100/H100で爆速）（中身は標準と同じモデル。大量バッチで数倍〜十数倍速い）
cmp_qwen3_1_7b_vllm = False  #@param {type:"boolean"}
#@markdown Cohere Transcribe 2B（別系統の有力候補）（日本語 CER: FLEURS 2.89% / CV 20.2%(Qwen 5.28% / 26.3%)。HF で規約同意が必要）
cmp_cohere_transcribe = False  #@param {type:"boolean"}
#@markdown Cohere Transcribe 2B × vLLM（Cohere を vLLM で高速に）
cmp_cohere_transcribe_vllm = False  #@param {type:"boolean"}
#@markdown Whisper large-v3-turbo（faster-whisper）（定番。速い）
cmp_whisper_large_v3_turbo = False  #@param {type:"boolean"}
#@markdown Whisper large-v3（faster-whisper）（定番の最高精度版）
cmp_whisper_large_v3 = False  #@param {type:"boolean"}
#@markdown kotoba-whisper v2.0（日本語特化 Whisper）（ReazonSpeech で学習した日本語特化の蒸留モデル。公式のおすすめどおり 15 秒ずつ読む(タイムスタンプはアライナーで付ける。faster-whisper の単語時刻はこのモデルだと segfault するため)）
cmp_kotoba_whisper_v2 = False  #@param {type:"boolean"}
#@markdown Parakeet TDT-CTC 0.6B ja（NVIDIA・日本語特化）（とても速い日本語専用モデル(JSUT の CER 6.60% の報告)。句読点は少なめ）
cmp_parakeet_ja = False  #@param {type:"boolean"}
#@markdown Granite Speech 4.1 2B（IBM・実験的）（日本語対応の2B(2026年4月)。context の語をキーワードとして渡すが、関係ない所にも入れがちなので語は少なめに）
cmp_granite_speech_4_1 = False  #@param {type:"boolean"}
#@markdown VibeVoice-ASR 8B（Microsoft・実験的）（50以上の言語・context 対応の8B。本来は60分一気読み＋話者付けのモデル）
cmp_vibevoice_asr = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **比べる区間**（秒）: 先頭から5分など。長いほど正確だけど時間がかかる
start_sec = 0  #@param {type:"number"}
duration_sec = 300  #@param {type:"number"}
#@markdown **正解テキスト**（任意）: 同じ区間を人が書き起こしたテキストのファイルパス（または本文）。あると CER で比べられる
reference = ""  #@param {type:"string"}
#@markdown **音源**: 空欄なら ⑤ の1つ目のファイル
compare_src = ""  #@param {type:"string"}

_keys = [k for v, k in [('cmp_qwen3_1_7b', 'qwen3-1.7b'), ('cmp_qwen3_0_6b', 'qwen3-0.6b'), ('cmp_qwen3_1_7b_ja', 'qwen3-1.7b-ja'), ('cmp_qwen3_1_7b_vllm', 'qwen3-1.7b-vllm'), ('cmp_cohere_transcribe', 'cohere-transcribe'), ('cmp_cohere_transcribe_vllm', 'cohere-transcribe-vllm'), ('cmp_whisper_large_v3_turbo', 'whisper-large-v3-turbo'), ('cmp_whisper_large_v3', 'whisper-large-v3'), ('cmp_kotoba_whisper_v2', 'kotoba-whisper-v2'), ('cmp_parakeet_ja', 'parakeet-ja'), ('cmp_granite_speech_4_1', 'granite-speech-4.1'), ('cmp_vibevoice_asr', 'vibevoice-asr')] if globals().get(v)]
_csrc = compare_src.strip() or (INPUTS[0] if "INPUTS" in globals() and INPUTS else "")
if not _csrc:
    raise ValueError("比較する音源がありません(compare_src に入れるか、先に ⑤ を実行)")
if "SESS" not in globals():
    SESS = pipeline.Session()
_st = ST if "ST" in globals() else pipeline.Settings()
CMP = SESS.compare(_csrc, _keys, _st, start=start_sec, duration=duration_sec, reference=reference)

In [ ]:
#@title ⑦ 議事録づくり（おまけ）
#@markdown ⑤ の結果（`.md` か `.txt`）から議事録を作ります
#@markdown - **プロンプトだけ作る**: `_minutes_prompt.md` を書き出すので、Claude や ChatGPT に貼るだけ（外には何も送りません）
#@markdown - **Gemini（Colab AI・無料）**: Colab に組み込みの Gemini で、API キーなしでここで議事録まで作ります
#@markdown - **Claude API**: Colab のシークレットに `ANTHROPIC_API_KEY` を登録しておくと、ここで議事録まで作ります
#@markdown
#@markdown ※ Gemini / Claude を選ぶと、文字起こしの中身がそのサービスに送られます
mode = "プロンプトだけ作る"  #@param ["プロンプトだけ作る", "Gemini（Colab AI・無料）", "Claude API"]
style = "議事録（決定事項・TODO つき）"  #@param ["議事録（決定事項・TODO つき）", "要約（3行＋詳細）", "発言者ごとの要点"] {allow-input: true}
#@markdown 対象（空欄なら ⑤ の最後のファイル）
transcript = ""  #@param {type:"string"}
#@markdown Gemini のモデル（`from google.colab import ai; ai.list_models()` で一覧）
gemini_model = "google/gemini-3.5-flash"  #@param ["google/gemini-3.5-flash", "google/gemini-3.1-pro-preview"] {allow-input: true}
#@markdown Claude のモデル（既定は Claude Opus 5。安全フィルタで断られたときは自動で別モデルに引き継ぐ設定にしてあります）
claude_model = "claude-opus-5"  #@param ["claude-opus-5", "claude-sonnet-5", "claude-haiku-4-5"] {allow-input: true}
import os
_t = transcript.strip()
if not _t and "OUTS" in globals() and OUTS:
    _paths = OUTS[-1].get("paths", {})
    _t = _paths.get("md") or _paths.get("txt") or ""
if not _t:
    raise ValueError("対象のテキストがありません(⑤ を実行するか transcript にパスを入れてください)")
MINUTES_OUT = pipeline.make_minutes(_t, style=style, mode=mode, model=claude_model, gemini_model=gemini_model,
                                    glossary=_terms if "_terms" in globals() else ())

In [ ]:
#@title ⑧ 後片付け（VRAM 解放）
#@markdown モデルを読み込んだままのワーカーと vLLM サーバーを止めて GPU メモリを空けます（次の実行では自動で読み込み直し）
if "SESS" in globals():
    print("\n".join(SESS.h.status()) or "(動いているものはありません)")
    SESS.free()